# Detector convencional de cilindros industriais — Hough + arcos + HOG/SVM — dois mapas e remoção de sólidos por erosão + diagnóstico de linhas/arcos orientados

Este notebook implementa uma pipeline clássica para detectar cilindros em ambientes industriais usando:

1. seleção manual persistente de imagens de treino/teste, incluindo cópias com **zoom out** quando marcadas;
2. ajuste interativo de pré-processamento, Canny, morfologia, Hough, união de segmentos colineares e critérios geométricos;
3. geração de candidatos por **linhas Hough em qualquer direção**, com união de segmentos separados por GAP;
4. busca de **arcos/tampas** apenas em regiões ao redor das retas filtradas;
5. expansão orientada da bounding box guiada por evidência de borda, arcos e **score do SVM**;
6. validação das ROIs por HOG + SVM, com características geométricas/arcos opcionais;
7. treinamento protegido por botão, sem execução automática ao rodar todas as células;
8. histórico cumulativo com métricas do classificador e do detector completo por IoU.

A versão atual está ajustada para o dataset **CylinDeRS-1** no formato `train/images`, `train/labels`, `test/images` e `test/labels`.

**Dois mapas de bordas:** o Canny completo é preservado para arcos/tampas e gradientes, enquanto um segundo Canny filtrado remove traços curtos antes da morfologia e da Hough.

**Revisão v21:** a remoção de regiões sólidas não usa mais a área do componente conectado completo. Agora o notebook salva o mapa após o fechamento, aplica uma **erosão forte** para apagar linhas finas e manter apenas núcleos sólidos, dilata controladamente esses núcleos e subtrai essa máscara do mapa fechado. Assim, linhas conectadas a um sólido tendem a ser preservadas.


## Organização esperada no projeto BLAZE

Este notebook foi adaptado para rodar dentro da estrutura portátil do repositório **BLAZE**, sem depender dos caminhos do servidor Jupyter da UFSC.

Estrutura esperada para o dataset de cilindros:

```text
BLAZE/
└── vision/
    └── datasets/
        └── cylinders/
            └── CylinDeRS-1/
                ├── train/
                │   ├── images/
                │   └── labels/
                ├── valid/
                │   ├── images/
                │   └── labels/
                └── test/
                    ├── images/
                    └── labels/
```

Os resultados gerados pelo notebook serão salvos em:

```text
BLAZE/vision/results/
```

Os setups oficiais ficam em:

```text
BLAZE/vision/parameters_setups/official/cylinders_detect/
```

Os setups modificados pelo usuário são copiados/salvos localmente em:

```text
BLAZE/vision/parameters_setups/user/cylinders_detect/
```

A pasta `user/` é ignorada pelo GitHub, evitando alterações acidentais nos setups oficiais.


In [ ]:
# ============================================================
# 1. Imports e caminhos principais
# ============================================================

from pathlib import Path
from collections import OrderedDict
import os
import sys
import shutil
import cv2
import json
import time
import math
import hashlib
import traceback
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

try:
    import joblib
except Exception as e:
    raise ImportError("Instale o joblib para salvar o modelo: pip install joblib") from e


# Extensões de imagem aceitas pelo notebook.
# Mantido aqui para que as funções de listagem do dataset funcionem
# independentemente da ordem de execução das células.
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}



# ============================================================
# Caminhos fornecidos para o projeto
# ============================================================

# Caminhos portáteis do projeto BLAZE
CURRENT_DIR = Path.cwd().resolve()

for _candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (_candidate / 'src' / 'blaze_paths.py').exists():
        PROJECT_ROOT = _candidate
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a raiz do projeto BLAZE. '
        'Abra este notebook a partir de uma pasta dentro do repositório BLAZE.'
    )

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from blaze_paths import (
    VISION_DATASETS_DIR,
    VISION_RESULTS_DIR,
    VISION_OFFICIAL_SETUPS_DIR,
    VISION_USER_SETUPS_DIR,
    ensure_base_dirs,
)

ensure_base_dirs()

DATASET_DIR = VISION_DATASETS_DIR / 'cylinders' / 'CylinDeRS-1'

TRAIN_IMG_DIR = DATASET_DIR / 'train' / 'images'
TRAIN_LABEL_DIR = DATASET_DIR / 'train' / 'labels'
TEST_IMG_DIR = DATASET_DIR / 'test' / 'images'
TEST_LABEL_DIR = DATASET_DIR / 'test' / 'labels'

RESULTS_DIR = VISION_RESULTS_DIR
GEOM_DIR = RESULTS_DIR / 'geometric_realtime_detector'
GEOM_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_DIR = GEOM_DIR / 'configs'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

CLASSICAL_DIR = RESULTS_DIR / 'classical_cylinder_detector'
CLASSICAL_DIR.mkdir(parents=True, exist_ok=True)

SELECAO_TREINO_PATH = CLASSICAL_DIR / 'imagens_selecionadas_treino.json'
SELECAO_TESTE_PATH = CLASSICAL_DIR / 'imagens_selecionadas_teste.json'

ZOOMOUT_TREINO_PATH = CLASSICAL_DIR / 'imagens_zoomout_treino.json'
ZOOMOUT_TESTE_PATH = CLASSICAL_DIR / 'imagens_zoomout_teste.json'
ZOOMOUT_CONFIG_PATH = CLASSICAL_DIR / 'zoomout_config.json'

ZOOMOUT_DATASET_DIR = CLASSICAL_DIR / 'dataset_zoomout_aplicado'
ZOOMOUT_DATASET_DIR.mkdir(parents=True, exist_ok=True)
ZOOMOUT_MANIFEST_PATH = CLASSICAL_DIR / 'dataset_zoomout_manifest.json'

MODEL_PATH = CLASSICAL_DIR / 'modelo_hog_svm_cilindros.joblib'
SCALER_PATH = CLASSICAL_DIR / 'scaler_hog_svm_cilindros.joblib'
METADATA_PATH = CLASSICAL_DIR / 'metadata_modelo.json'
HISTORICO_PATH = CLASSICAL_DIR / 'historico_treinos.csv'
# Compatibilidade com funções antigas do notebook.
HISTORY_PATH = HISTORICO_PATH

OFFICIAL_PARAM_SETUPS_PATH = VISION_OFFICIAL_SETUPS_DIR / 'cylinders_detect' / 'setups_parametricos.json'
USER_PARAM_SETUPS_PATH = VISION_USER_SETUPS_DIR / 'cylinders_detect' / 'setups_parametricos_user.json'

OFFICIAL_REFINE_SETUPS_PATH = VISION_OFFICIAL_SETUPS_DIR / 'cylinders_detect' / 'setups_refinamento_expansao.json'
USER_REFINE_SETUPS_PATH = VISION_USER_SETUPS_DIR / 'cylinders_detect' / 'setups_refinamento_expansao_user.json'


def inicializar_arquivo_setup_local(official_path, user_path):
    official_path = Path(official_path)
    user_path = Path(user_path)
    user_path.parent.mkdir(parents=True, exist_ok=True)

    if not user_path.exists() and official_path.exists():
        shutil.copy2(official_path, user_path)
        print(f'Arquivo oficial copiado para edição local: {user_path}')

    return user_path


PARAM_SETUPS_PATH = inicializar_arquivo_setup_local(
    OFFICIAL_PARAM_SETUPS_PATH,
    USER_PARAM_SETUPS_PATH
)

REFINE_SETUPS_PATH = inicializar_arquivo_setup_local(
    OFFICIAL_REFINE_SETUPS_PATH,
    USER_REFINE_SETUPS_PATH
)


print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_DIR:', DATASET_DIR)

if not DATASET_DIR.exists():
    print('')
    print('ATENÇÃO: dataset CylinDeRS-1 não encontrado.')
    print('Coloque ou vincule o dataset em:')
    print(DATASET_DIR)
    print('')
    print('Estrutura esperada:')
    print('CylinDeRS-1/train/images')
    print('CylinDeRS-1/train/labels')
    print('CylinDeRS-1/test/images')
    print('CylinDeRS-1/test/labels')
else:
    print('Dataset encontrado.')
print('TRAIN_IMG_DIR:', TRAIN_IMG_DIR)
print('TRAIN_LABEL_DIR:', TRAIN_LABEL_DIR)
print('TEST_IMG_DIR:', TEST_IMG_DIR)
print('TEST_LABEL_DIR:', TEST_LABEL_DIR)
print('CLASSICAL_DIR:', CLASSICAL_DIR)
print('ZOOMOUT_DATASET_DIR:', ZOOMOUT_DATASET_DIR)


In [ ]:
# ============================================================
# 2. Funções utilitárias gerais e leitura do dataset YOLO
# ============================================================

def caminhos_dataset(dataset_dir):
    """Retorna os caminhos padronizados para um dataset no formato YOLO train/test."""
    dataset_dir = Path(dataset_dir).expanduser().resolve()
    return {
        'dataset_dir': dataset_dir,
        'train_img_dir': dataset_dir / 'train' / 'images',
        'train_label_dir': dataset_dir / 'train' / 'labels',
        'test_img_dir': dataset_dir / 'test' / 'images',
        'test_label_dir': dataset_dir / 'test' / 'labels',
    }


def label_path_para_imagem(img_path, img_dir, label_dir):
    """Converte caminho de imagem em caminho esperado do label YOLO correspondente."""
    img_path = Path(img_path)
    img_dir = Path(img_dir)
    label_dir = Path(label_dir)
    try:
        rel = img_path.relative_to(img_dir)
    except ValueError:
        rel = Path(img_path.name)
    return (label_dir / rel).with_suffix('.txt')


def contar_labels_yolo(label_path):
    """Conta apenas linhas YOLO válidas e finitas em um arquivo de label.

    Alguns arquivos podem conter valores ausentes/NaN. Esses registros são ignorados
    para evitar erros posteriores na conversão para bounding boxes.
    """
    label_path = Path(label_path)
    if not label_path.exists():
        return 0

    n = 0
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls = float(parts[0])
                vals = list(map(float, parts[1:5]))
            except Exception:
                continue
            if not (np.isfinite(cls) and all(np.isfinite(v) for v in vals)):
                continue
            xc, yc, bw, bh = vals
            if bw <= 0 or bh <= 0:
                continue
            n += 1
    return n


def listar_imagens_split(split, img_dir, label_dir):
    """Lista imagens de um split e associa cada uma ao arquivo de label YOLO."""
    img_dir = Path(img_dir).expanduser().resolve()
    label_dir = Path(label_dir).expanduser().resolve()

    rows = []
    if not img_dir.exists():
        return rows

    for p in sorted(img_dir.rglob('*')):
        if not (p.is_file() and p.suffix.lower() in IMG_EXTS):
            continue

        lab = label_path_para_imagem(p, img_dir, label_dir)
        n_obj = contar_labels_yolo(lab)
        rows.append({
            'split': split,
            'path': str(p),
            'arquivo': p.name,
            'stem': p.stem,
            'label_path': str(lab),
            'label_existe': lab.exists(),
            'n_obj': int(n_obj),
            'classe': 'com_anotacao' if n_obj > 0 else 'sem_anotacao'
        })

    return rows


def listar_imagens_dataset(dataset_dir):
    """Lista imagens de train e test a partir da raiz do CylinDeRS-1."""
    paths = caminhos_dataset(dataset_dir)

    rows = []
    rows.extend(listar_imagens_split('train', paths['train_img_dir'], paths['train_label_dir']))
    rows.extend(listar_imagens_split('test', paths['test_img_dir'], paths['test_label_dir']))

    return pd.DataFrame(rows, columns=[
        'split', 'path', 'arquivo', 'stem', 'label_path', 'label_existe', 'n_obj', 'classe'
    ])


def ler_imagem_bgr(path_img):
    """Lê imagem com OpenCV em BGR."""
    img = cv2.imread(str(path_img), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Não foi possível carregar a imagem: {path_img}')
    return img


def bgr_para_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def clip_bbox(x1, y1, x2, y2, w, h):
    x1 = max(0, min(int(x1), w - 1))
    y1 = max(0, min(int(y1), h - 1))
    x2 = max(1, min(int(x2), w))
    y2 = max(1, min(int(y2), h))
    return x1, y1, x2, y2


def resumo_dataset(df):
    if df is None or len(df) == 0:
        return 'Nenhuma imagem encontrada.'

    total = len(df)
    n_train = int((df['split'] == 'train').sum())
    n_test = int((df['split'] == 'test').sum())
    com_label = int(df['label_existe'].sum())
    imagens_com_obj = int((df['n_obj'] > 0).sum())
    total_obj = int(df['n_obj'].sum())

    return (
        f'Imagens: {total} | Train: {n_train} | Test: {n_test} | '
        f'Labels existentes: {com_label} | Imagens com objeto: {imagens_com_obj} | '
        f'Objetos anotados: {total_obj}'
    )


def exibir_tabela_compacta(df, max_rows=12):
    """
    Mostra uma tabela compacta com rolagem horizontal controlada,
    para evitar que ela ultrapasse a largura visual do notebook.
    """
    if df is None or len(df) == 0:
        display(HTML('<em>Nenhum registro para mostrar.</em>'))
        return

    df_show = df.tail(max_rows).copy()

    for col in df_show.columns:
        if df_show[col].dtype == object:
            df_show[col] = df_show[col].astype(str).str.slice(0, 46)

    html = df_show.to_html(index=False, escape=False)
    display(HTML(f'''
    <div style="
        max-width: 100%;
        overflow-x: auto;
        border: 1px solid #444;
        padding: 6px;
        border-radius: 8px;
    ">
        <style>
            table.dataframe {{
                font-size: 11px;
                border-collapse: collapse;
                width: max-content;
                max-width: 100%;
            }}
            table.dataframe th, table.dataframe td {{
                white-space: nowrap;
                padding: 4px 7px;
                text-align: center;
            }}
        </style>
        {html}
    </div>
    '''))


def salvar_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def carregar_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


In [ ]:
# ============================================================
# 3. Pipeline de pré-processamento com dois mapas de bordas
# ============================================================


def filtrar_componentes_borda(edge_map, p):
    """
    Remove pequenos componentes conectados do mapa binário de bordas.

    Objetivo:
        - preservar o Canny completo para arcos/tampas e gradientes;
        - gerar um segundo mapa mais limpo para Hough/morfologia;
        - evitar que o fechamento morfológico cole muitos traços curtos e forme blobs sólidos.

    Critérios de permanência:
        componente permanece se:
            área >= edge_min_area E maior dimensão >= edge_min_extent

    Retorna:
        edge_filtrado, info
    """
    if not bool(p.get('edge_filter_ativo', True)):
        return edge_map.copy(), {
            'enabled': False,
            'n_components': 0,
            'n_removed': 0,
            'n_kept': 0,
            'min_area': 0,
            'min_extent': 0
        }

    min_area = max(0, int(p.get('edge_min_area', 12)))
    min_extent = max(0, int(p.get('edge_min_extent', 10)))

    bin_map = (edge_map > 0).astype(np.uint8)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_map, connectivity=8)

    out = np.zeros_like(edge_map)
    kept = 0
    removed = 0

    for lab in range(1, n_labels):
        x, y, w, h, area = stats[lab]
        extent = max(int(w), int(h))
        keep = (int(area) >= min_area) and (extent >= min_extent)
        if keep:
            out[labels == lab] = 255
            kept += 1
        else:
            removed += 1

    info = {
        'enabled': True,
        'n_components': int(max(0, n_labels - 1)),
        'n_removed': int(removed),
        'n_kept': int(kept),
        'min_area': int(min_area),
        'min_extent': int(min_extent)
    }
    return out, info


def _kernel_rect_impar(w, h):
    """Cria kernel retangular garantindo dimensões positivas e ímpares."""
    w = max(1, int(w))
    h = max(1, int(h))
    if w % 2 == 0:
        w += 1
    if h % 2 == 0:
        h += 1
    return cv2.getStructuringElement(cv2.MORPH_RECT, (w, h)), w, h


def _filtrar_componentes_por_area(bin_map, min_area):
    """Mantém apenas componentes com área mínima em um mapa binário 0/255."""
    min_area = max(0, int(min_area))
    bin_u8 = (bin_map > 0).astype(np.uint8)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_u8, connectivity=8)
    out = np.zeros_like(bin_map, dtype=np.uint8)
    kept = 0
    removed = 0

    for lab in range(1, n_labels):
        area = int(stats[lab, cv2.CC_STAT_AREA])
        if area >= min_area:
            out[labels == lab] = 255
            kept += 1
        else:
            removed += 1

    return out, {'n_components': int(max(0, n_labels - 1)), 'n_kept': int(kept), 'n_removed': int(removed)}


def remover_regioes_solidas_por_erosao(mapa_fechado, p):
    """
    Remove regiões sólidas criadas pelo fechamento preservando linhas conectadas.

    Nova lógica:
        1. salva o mapa após o fechamento;
        2. aplica erosão forte sobre esse mapa;
        3. linhas finas desaparecem, mas núcleos sólidos permanecem;
        4. filtra núcleos pequenos;
        5. dilata controladamente o núcleo sólido;
        6. limita a máscara dilatada ao mapa fechado original;
        7. subtrai somente essa máscara sólida do mapa fechado salvo.
    """
    if not bool(p.get('solid_filter_ativo', True)):
        vazio = np.zeros_like(mapa_fechado)
        return mapa_fechado.copy(), vazio, vazio, {
            'enabled': False,
            'n_core_components': 0,
            'n_core_kept': 0,
            'n_core_removed': 0,
            'core_pixels': 0,
            'mask_pixels': 0,
            'core_min_area': 0,
            'erode_kernel': '0x0:0',
            'dilate_kernel': '0x0:0'
        }

    erode_kw = int(p.get('solid_erode_kernel_w', 9))
    erode_kh = int(p.get('solid_erode_kernel_h', 9))
    erode_iter = max(0, int(p.get('solid_erode_iter', 1)))
    dilate_kw = int(p.get('solid_dilate_kernel_w', 5))
    dilate_kh = int(p.get('solid_dilate_kernel_h', 5))
    dilate_iter = max(0, int(p.get('solid_dilate_iter', 1)))
    core_min_area = max(0, int(p.get('solid_min_area', 80)))

    bin_fechado = ((mapa_fechado > 0).astype(np.uint8) * 255)

    ker_e, erode_kw, erode_kh = _kernel_rect_impar(erode_kw, erode_kh)
    if erode_iter > 0:
        nucleo_raw = cv2.erode(bin_fechado, ker_e, iterations=erode_iter)
    else:
        nucleo_raw = bin_fechado.copy()

    nucleo_solido, comp_info = _filtrar_componentes_por_area(nucleo_raw, core_min_area)

    ker_d, dilate_kw, dilate_kh = _kernel_rect_impar(dilate_kw, dilate_kh)
    if dilate_iter > 0:
        mascara_solida = cv2.dilate(nucleo_solido, ker_d, iterations=dilate_iter)
    else:
        mascara_solida = nucleo_solido.copy()

    # A máscara dilatada só remove pixels existentes no mapa fechado.
    mascara_solida = cv2.bitwise_and(mascara_solida, bin_fechado)

    mapa_sem_solidos = bin_fechado.copy()
    mapa_sem_solidos[mascara_solida > 0] = 0

    info = {
        'enabled': True,
        'n_core_components': int(comp_info['n_components']),
        'n_core_kept': int(comp_info['n_kept']),
        'n_core_removed': int(comp_info['n_removed']),
        'core_pixels': int(np.count_nonzero(nucleo_solido)),
        'mask_pixels': int(np.count_nonzero(mascara_solida)),
        'core_min_area': int(core_min_area),
        'erode_kernel': f'{erode_kw}x{erode_kh}:{erode_iter}',
        'dilate_kernel': f'{dilate_kw}x{dilate_kh}:{dilate_iter}'
    }
    return mapa_sem_solidos, nucleo_solido, mascara_solida, info


# Alias com o nome usado nas versões anteriores.
def remover_regioes_solidas_morfologia(morph_map, p):
    return remover_regioes_solidas_por_erosao(morph_map, p)


def aplicar_preprocessamento(img_bgr, p):
    """
    Aplica as etapas de preparação da imagem usando dois mapas de bordas.

    Ordem da morfologia nesta versão:
        Canny filtrado -> fechamento -> remoção de sólidos por erosão -> abertura final
    """
    rgb = bgr_para_rgb(img_bgr)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    if p.get("clahe_ativo", True):
        clahe = cv2.createCLAHE(
            clipLimit=float(p.get("clahe_clip", 2.0)),
            tileGridSize=(int(p.get("clahe_grid", 8)), int(p.get("clahe_grid", 8)))
        )
        gray_eq = clahe.apply(gray)
    else:
        gray_eq = gray.copy()

    d = int(p.get("bilateral_d", 5))
    if d < 1:
        d = 1
    if d % 2 == 0:
        d += 1

    bilateral = cv2.bilateralFilter(
        gray_eq,
        d=d,
        sigmaColor=float(p.get("bilateral_sigma_color", 50)),
        sigmaSpace=float(p.get("bilateral_sigma_space", 50))
    )

    med = float(np.median(bilateral))
    sigma = float(p.get("canny_sigma", 0.33))
    lower = int(max(0, (1.0 - sigma) * med))
    upper = int(min(255, (1.0 + sigma) * med))

    if upper <= lower:
        upper = min(255, lower + 20)

    aperture = int(p.get("canny_aperture", 3))
    if aperture not in [3, 5, 7]:
        aperture = 3

    canny_full = cv2.Canny(
        bilateral,
        threshold1=lower,
        threshold2=upper,
        apertureSize=aperture,
        L2gradient=bool(p.get("canny_l2gradient", True))
    )

    canny_hough, edge_filter_info = filtrar_componentes_borda(canny_full, p)

    kcw = max(1, int(p.get("close_kernel_w", 3)))
    kch = max(1, int(p.get("close_kernel_h", 9)))
    kow = max(1, int(p.get("open_kernel_w", 3)))
    koh = max(1, int(p.get("open_kernel_h", 3)))

    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (kcw, kch))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (kow, koh))

    morph_close = cv2.morphologyEx(
        canny_hough,
        cv2.MORPH_CLOSE,
        kernel_close,
        iterations=int(p.get("close_iter", 1))
    )

    morph_no_solids, solid_core, solid_mask, solid_filter_info = remover_regioes_solidas_por_erosao(morph_close, p)

    morph = cv2.morphologyEx(
        morph_no_solids,
        cv2.MORPH_OPEN,
        kernel_open,
        iterations=int(p.get("open_iter", 1))
    )

    return {
        "rgb": rgb,
        "gray": gray,
        "gray_eq": gray_eq,
        "bilateral": bilateral,
        "canny": canny_full,
        "canny_full": canny_full,
        "canny_hough": canny_hough,
        "canny_filtrado": canny_hough,
        "morph_close": morph_close,
        "morph_close_saved": morph_close,
        "solid_core": solid_core,
        "solid_mask": solid_mask,
        "morph_no_solids": morph_no_solids,
        "morph_sem_solidos": morph_no_solids,
        "morph_open_raw": morph_no_solids,
        "morph": morph,
        "canny_lower": lower,
        "canny_upper": upper,
        "edge_filter_info": edge_filter_info,
        "solid_filter_info": solid_filter_info
    }


def selecionar_imagem_hog(pre, p):
    """
    Escolhe a imagem usada para extração HOG.
    """
    escolha = p["hog_input"]

    if escolha == "gray":
        return pre["gray"]
    if escolha == "gray_eq":
        return pre["gray_eq"]
    if escolha == "bilateral":
        return pre["bilateral"]
    if escolha in ["canny", "canny_full"]:
        return pre["canny_full"]
    if escolha in ["canny_hough", "canny_filtrado"]:
        return pre["canny_hough"]
    if escolha == "morph":
        return pre["morph"]

    return pre["bilateral"]


In [ ]:
# ============================================================
# 4. Hough, pareamento geométrico e geração de ROIs
# ============================================================

def detectar_linhas_hough(img_edges, p):
    """
    Detecta segmentos de reta com HoughLinesP.
    """
    lines = cv2.HoughLinesP(
        img_edges,
        rho=float(p["hough_rho"]),
        theta=np.deg2rad(float(p["hough_theta_deg"])),
        threshold=int(p["hough_threshold"]),
        minLineLength=int(p["hough_min_line_length"]),
        maxLineGap=int(p["hough_max_line_gap"])
    )

    if lines is None:
        return []

    return [tuple(map(int, l[0])) for l in lines]


def metrica_linha(line):
    x1, y1, x2, y2 = line
    dx = x2 - x1
    dy = y2 - y1
    length = float(math.hypot(dx, dy))
    angle = math.degrees(math.atan2(dy, dx)) % 180.0
    center = np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0], dtype=float)

    if length == 0:
        u = np.array([1.0, 0.0])
    else:
        u = np.array([dx / length, dy / length], dtype=float)

    return {
        "line": line,
        "length": length,
        "angle": angle,
        "center": center,
        "u": u
    }


def diff_angular_graus(a, b):
    """
    Diferença angular mínima considerando orientação de reta, não vetor.
    Retorna valor entre 0 e 90 graus.
    """
    d = abs((a - b + 90.0) % 180.0 - 90.0)
    return float(d)


def intervalo_projetado(line, u):
    x1, y1, x2, y2 = line
    p1 = np.array([x1, y1], dtype=float)
    p2 = np.array([x2, y2], dtype=float)
    v1 = float(np.dot(p1, u))
    v2 = float(np.dot(p2, u))
    return min(v1, v2), max(v1, v2)


def razao_sobreposicao(i1, i2):
    a1, a2 = i1
    b1, b2 = i2

    inter = max(0.0, min(a2, b2) - max(a1, b1))
    len1 = max(1e-6, a2 - a1)
    len2 = max(1e-6, b2 - b1)

    return float(inter / min(len1, len2))


def encontrar_rois_por_pares_de_linhas(lines, shape_hw, p):
    """
    Procura pares de retas paralelas e gera ROIs candidatas.
    """
    h, w = shape_hw
    metricas = [metrica_linha(l) for l in lines]
    metricas = [
        m for m in metricas
        if p["line_length_min"] <= m["length"] <= p["line_length_max"]
    ]

    candidatos = []

    for i in range(len(metricas)):
        for j in range(i + 1, len(metricas)):
            m1 = metricas[i]
            m2 = metricas[j]

            adiff = diff_angular_graus(m1["angle"], m2["angle"])
            if adiff > p["pair_angle_tol_deg"]:
                continue

            # Orientação média do par.
            u = m1["u"] + m2["u"]
            if np.linalg.norm(u) < 1e-6:
                u = m1["u"]
            else:
                u = u / np.linalg.norm(u)

            # Normal à direção da reta.
            n = np.array([-u[1], u[0]], dtype=float)

            # Distância transversal entre centros.
            dist = abs(float(np.dot(m2["center"] - m1["center"], n)))
            if not (p["pair_dist_min"] <= dist <= p["pair_dist_max"]):
                continue

            # Sobreposição longitudinal.
            int1 = intervalo_projetado(m1["line"], u)
            int2 = intervalo_projetado(m2["line"], u)
            overlap = razao_sobreposicao(int1, int2)
            if overlap < p["pair_overlap_min"]:
                continue

            pts = np.array([
                [m1["line"][0], m1["line"][1]],
                [m1["line"][2], m1["line"][3]],
                [m2["line"][0], m2["line"][1]],
                [m2["line"][2], m2["line"][3]],
            ], dtype=float)

            margin = int(p["roi_margin_px"])
            x1 = int(np.floor(pts[:, 0].min())) - margin
            y1 = int(np.floor(pts[:, 1].min())) - margin
            x2 = int(np.ceil(pts[:, 0].max())) + margin
            y2 = int(np.ceil(pts[:, 1].max())) + margin

            x1, y1, x2, y2 = clip_bbox(x1, y1, x2, y2, w, h)

            roi_w = max(1, x2 - x1)
            roi_h = max(1, y2 - y1)
            aspect = roi_h / roi_w

            if not (p["roi_aspect_min"] <= aspect <= p["roi_aspect_max"]):
                continue

            area = roi_w * roi_h

            candidatos.append({
                "bbox": (x1, y1, x2, y2),
                "line1": m1["line"],
                "line2": m2["line"],
                "angle_diff": adiff,
                "distance": dist,
                "overlap": overlap,
                "aspect": aspect,
                "area": area
            })

    # Ordenação simples: prioriza boa sobreposição e baixa diferença angular.
    candidatos = sorted(
        candidatos,
        key=lambda c: (-c["overlap"], c["angle_diff"], abs(c["distance"] - (p["pair_dist_min"] + p["pair_dist_max"]) / 2))
    )

    return candidatos[:int(p["max_rois"])]


def desenhar_linhas(img_rgb, lines, max_lines=80):
    out = img_rgb.copy()
    for line in lines[:max_lines]:
        x1, y1, x2, y2 = line
        cv2.line(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
    return out


def desenhar_rois(img_rgb, candidatos, mostrar_pares=True):
    out = img_rgb.copy()

    for idx, c in enumerate(candidatos):
        x1, y1, x2, y2 = c["bbox"]

        # ROI em azul/laranja.
        cv2.rectangle(out, (x1, y1), (x2, y2), (255, 120, 0), 2)
        cv2.putText(
            out,
            f"ROI {idx+1}",
            (x1, max(15, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 120, 0),
            1,
            cv2.LINE_AA
        )

        if mostrar_pares:
            for line in [c["line1"], c["line2"]]:
                lx1, ly1, lx2, ly2 = line
                cv2.line(out, (lx1, ly1), (lx2, ly2), (0, 255, 0), 2)

    return out

In [ ]:
# ============================================================
# 5. HOG, labels YOLO, assinatura de parâmetros e histórico
# ============================================================

def extrair_hog_de_imagem(img_bgr, p):
    """Extrai vetor HOG de uma imagem/crop."""
    pre = aplicar_preprocessamento(img_bgr, p)
    img_hog = selecionar_imagem_hog(pre, p)

    w = int(p['hog_resize_w'])
    h = int(p['hog_resize_h'])
    img_resized = cv2.resize(img_hog, (w, h), interpolation=cv2.INTER_AREA)

    features = hog(
        img_resized,
        orientations=int(p['hog_orientations']),
        pixels_per_cell=(int(p['hog_pixels_per_cell']), int(p['hog_pixels_per_cell'])),
        cells_per_block=(int(p['hog_cells_per_block']), int(p['hog_cells_per_block'])),
        block_norm='L2-Hys',
        transform_sqrt=True,
        feature_vector=True
    )

    return features.astype(np.float32)


def extrair_hog_visualizacao_de_imagem(img_bgr, p):
    """
    Extrai HOG e também retorna a imagem usada e a visualização dos gradientes.

    A função é usada apenas para diagnóstico visual. O treinamento continua usando
    `extrair_hog_de_imagem`, com os mesmos parâmetros.
    """
    pre = aplicar_preprocessamento(img_bgr, p)
    img_hog = selecionar_imagem_hog(pre, p)

    w = int(p['hog_resize_w'])
    h = int(p['hog_resize_h'])
    img_resized = cv2.resize(img_hog, (w, h), interpolation=cv2.INTER_AREA)

    features, hog_image = hog(
        img_resized,
        orientations=int(p['hog_orientations']),
        pixels_per_cell=(int(p['hog_pixels_per_cell']), int(p['hog_pixels_per_cell'])),
        cells_per_block=(int(p['hog_cells_per_block']), int(p['hog_cells_per_block'])),
        block_norm='L2-Hys',
        transform_sqrt=True,
        visualize=True,
        feature_vector=True
    )

    hog_image = np.asarray(hog_image, dtype=np.float32)
    if hog_image.max() > hog_image.min():
        hog_image = (hog_image - hog_image.min()) / (hog_image.max() - hog_image.min())

    return img_resized, hog_image, features.astype(np.float32)


def ler_bboxes_yolo(label_path, img_w, img_h, class_filter=None):
    """
    Lê bboxes YOLO normalizadas e retorna bboxes em pixel.

    Retorna lista de dicionários:
        {'class_id': int, 'bbox': (x1, y1, x2, y2)}

    Registros inválidos, NaN, infinitos, largura/altura não positivas ou caixas
    degeneradas são ignorados. Isso evita erros do tipo
    "cannot convert float NaN to integer" durante a validação visual.
    """
    label_path = Path(label_path)
    if not label_path.exists():
        return []

    boxes = []
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            try:
                cls_float = float(parts[0])
                vals = list(map(float, parts[1:5]))
            except Exception:
                continue

            if not (np.isfinite(cls_float) and all(np.isfinite(v) for v in vals)):
                continue

            cls = int(cls_float)
            xc, yc, bw, bh = vals
            if bw <= 0 or bh <= 0:
                continue

            if class_filter is not None and cls not in class_filter:
                continue

            # Aceita anotações levemente fora de [0, 1], mas descarta caixas totalmente inválidas.
            x1 = (xc - bw / 2.0) * img_w
            y1 = (yc - bh / 2.0) * img_h
            x2 = (xc + bw / 2.0) * img_w
            y2 = (yc + bh / 2.0) * img_h
            if not all(np.isfinite(v) for v in [x1, y1, x2, y2]):
                continue

            # Se a caixa inteira ficou fora da imagem, ignora.
            if x2 <= 0 or y2 <= 0 or x1 >= img_w or y1 >= img_h:
                continue

            x1, y1, x2, y2 = clip_bbox(x1, y1, x2, y2, img_w, img_h)

            if (x2 - x1) >= 4 and (y2 - y1) >= 4:
                boxes.append({'class_id': cls, 'bbox': (x1, y1, x2, y2)})

    return boxes


def expandir_bbox(bbox, margin_pct, img_w, img_h):
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1
    mx = bw * float(margin_pct)
    my = bh * float(margin_pct)
    return clip_bbox(x1 - mx, y1 - my, x2 + mx, y2 + my, img_w, img_h)


def iou_bbox(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(1, (ax2 - ax1) * (ay2 - ay1))
    area_b = max(1, (bx2 - bx1) * (by2 - by1))
    return inter / float(area_a + area_b - inter + 1e-9)


def bbox_aleatorio(img_w, img_h, target_w, target_h, rng):
    target_w = int(max(8, min(target_w, img_w)))
    target_h = int(max(8, min(target_h, img_h)))
    if img_w <= target_w or img_h <= target_h:
        return (0, 0, img_w, img_h)
    x1 = int(rng.integers(0, img_w - target_w))
    y1 = int(rng.integers(0, img_h - target_h))
    return (x1, y1, x1 + target_w, y1 + target_h)


def recortar_bbox(img_bgr, bbox):
    x1, y1, x2, y2 = map(int, bbox)
    return img_bgr[y1:y2, x1:x2].copy()


def extrair_strings_json(obj):
    """Extrai recursivamente strings de um JSON de seleção."""
    vals = []
    if obj is None:
        return vals
    if isinstance(obj, str):
        return [obj]
    if isinstance(obj, dict):
        for v in obj.values():
            vals.extend(extrair_strings_json(v))
    elif isinstance(obj, (list, tuple, set)):
        for v in obj:
            vals.extend(extrair_strings_json(v))
    return vals


def carregar_chaves_selecao(path):
    """Lê um JSON de seleção e retorna chaves por caminho, nome de arquivo e stem."""
    obj = carregar_json(path, default=None)
    strings = extrair_strings_json(obj)
    chaves = set()
    for s in strings:
        p = Path(str(s))
        chaves.add(str(s))
        chaves.add(p.name)
        chaves.add(p.stem)
    return chaves



def _json_zoomout_split(split):
    """Caminho do JSON que guarda quais imagens receberão zoom out em cada split."""
    return ZOOMOUT_TREINO_PATH if split == 'train' else ZOOMOUT_TESTE_PATH


def carregar_lista_zoomout(split):
    """Lista persistente das imagens marcadas para gerar cópia com zoom out aplicado."""
    obj = carregar_json(_json_zoomout_split(split), default=[])
    strings = extrair_strings_json(obj)
    # Remove duplicatas preservando a ordem.
    vistos = set()
    saida = []
    for s in strings:
        ss = str(s)
        if ss not in vistos:
            saida.append(ss)
            vistos.add(ss)
    return saida


def carregar_chaves_zoomout(split):
    """Lê a seleção persistente de imagens que receberão zoom out e retorna chaves de comparação."""
    strings = carregar_lista_zoomout(split)
    chaves = set()
    for s in strings:
        p = Path(str(s))
        chaves.add(str(s))
        chaves.add(str(p))
        chaves.add(p.name)
        chaves.add(p.stem)
    return chaves


def carregar_zoomout_config():
    """Configuração global do zoom out artificial."""
    cfg = carregar_json(ZOOMOUT_CONFIG_PATH, default={})
    if not isinstance(cfg, dict):
        cfg = {}
    cfg.setdefault('zoom_out_pct', 0)
    return cfg


def salvar_zoomout_config(zoom_out_pct):
    """Salva o percentual global de zoom out."""
    salvar_json(ZOOMOUT_CONFIG_PATH, {
        'zoom_out_pct': int(zoom_out_pct),
        'atualizado_em': time.strftime('%Y-%m-%d %H:%M:%S')
    })


def obter_zoom_out_pct():
    """Retorna o percentual atual de zoom out, priorizando o estado do notebook."""
    state = globals().get('STATE', {})
    if isinstance(state, dict) and 'zoom_out_pct' in state:
        try:
            return int(state.get('zoom_out_pct', 0))
        except Exception:
            return 0
    return int(carregar_zoomout_config().get('zoom_out_pct', 0))


def _path_em_chaves(path, chaves):
    p = Path(str(path))
    return (str(path) in chaves) or (str(p) in chaves) or (p.name in chaves) or (p.stem in chaves)


def imagem_tem_zoomout_path(path, split=None):
    """Verifica se uma imagem original foi marcada para gerar versão com zoom out."""
    path = Path(str(path))
    # Evita aplicar zoom out duas vezes em cópias já geradas.
    if '__zoomout_' in path.stem:
        return False
    try:
        if ZOOMOUT_DATASET_DIR in path.parents:
            return False
    except Exception:
        pass

    splits = [split] if split in ['train', 'test'] else ['train', 'test']
    for sp in splits:
        if _path_em_chaves(path, carregar_chaves_zoomout(sp)):
            return True
    return False


def aplicar_zoom_out_imagem_e_bboxes(img_bgr, boxes=None, zoom_pct=0):
    """
    Aplica zoom out artificial mantendo o tamanho final da imagem.

    Exemplo: zoom_pct=30 reduz o conteúdo para 70% do tamanho original e
    preenche as margens resultantes com preto. As caixas em pixels são
    transformadas para a nova posição.
    """
    boxes = boxes or []
    pct = max(0, min(90, int(zoom_pct)))
    if pct <= 0:
        return img_bgr, list(boxes)

    h, w = img_bgr.shape[:2]
    scale = max(0.05, 1.0 - pct / 100.0)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    img_small = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros_like(img_bgr)

    x0 = (w - new_w) // 2
    y0 = (h - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = img_small

    boxes_zoom = []
    for bbox in boxes:
        x1, y1, x2, y2 = bbox
        nx1 = x0 + x1 * scale
        ny1 = y0 + y1 * scale
        nx2 = x0 + x2 * scale
        ny2 = y0 + y2 * scale
        boxes_zoom.append(clip_bbox(nx1, ny1, nx2, ny2, w, h))

    return canvas, boxes_zoom


def _zoomout_nome_arquivo(row, zoom_pct):
    p = Path(str(row['path']))
    return f"{p.stem}__zoomout_{int(zoom_pct):02d}{p.suffix.lower()}"


def caminhos_zoomout_para_row(row, zoom_pct):
    """Caminhos da cópia com zoom out aplicada para uma imagem do split."""
    split = row.get('split', 'train')
    nome_img = _zoomout_nome_arquivo(row, zoom_pct)
    stem = Path(nome_img).stem
    img_dir = ZOOMOUT_DATASET_DIR / split / 'images'
    label_dir = ZOOMOUT_DATASET_DIR / split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    return img_dir / nome_img, label_dir / f'{stem}.txt'


def escrever_bboxes_yolo(label_path, objetos, img_w, img_h):
    """Escreve bboxes em formato YOLO normalizado."""
    label_path = Path(label_path)
    label_path.parent.mkdir(parents=True, exist_ok=True)
    linhas = []
    for obj in objetos:
        cls = int(obj.get('class_id', 0))
        x1, y1, x2, y2 = obj['bbox']
        x1, y1, x2, y2 = clip_bbox(x1, y1, x2, y2, img_w, img_h)
        bw = max(0.0, x2 - x1)
        bh = max(0.0, y2 - y1)
        if bw < 1 or bh < 1:
            continue
        xc = ((x1 + x2) / 2.0) / img_w
        yc = ((y1 + y2) / 2.0) / img_h
        wn = bw / img_w
        hn = bh / img_h
        linhas.append(f"{cls} {xc:.8f} {yc:.8f} {wn:.8f} {hn:.8f}")
    label_path.write_text('\n'.join(linhas) + ('\n' if linhas else ''), encoding='utf-8')


def criar_imagem_zoomout_salva(row, zoom_pct=None, sobrescrever=False):
    """
    Cria uma cópia da imagem com zoom out aplicado e uma label YOLO ajustada.

    As imagens originais não são sobrescritas. A cópia é salva em
    `ZOOMOUT_DATASET_DIR/<split>/images` e a label correspondente em
    `ZOOMOUT_DATASET_DIR/<split>/labels`.
    """
    pct = int(obter_zoom_out_pct() if zoom_pct is None else zoom_pct)
    if pct <= 0:
        return None, None

    img_out_path, label_out_path = caminhos_zoomout_para_row(row, pct)
    if img_out_path.exists() and label_out_path.exists() and not sobrescrever:
        return img_out_path, label_out_path

    img = ler_imagem_bgr(row['path'])
    h, w = img.shape[:2]
    objetos_orig = ler_bboxes_yolo(row['label_path'], w, h)
    bboxes_orig = [obj['bbox'] for obj in objetos_orig]
    img_zoom, bboxes_zoom = aplicar_zoom_out_imagem_e_bboxes(img, bboxes_orig, pct)

    ok = cv2.imwrite(str(img_out_path), img_zoom)
    if not ok:
        raise RuntimeError(f'Falha ao salvar imagem com zoom out: {img_out_path}')

    objetos_zoom = []
    for obj, bbox_zoom in zip(objetos_orig, bboxes_zoom):
        objetos_zoom.append({'class_id': int(obj.get('class_id', 0)), 'bbox': bbox_zoom})
    escrever_bboxes_yolo(label_out_path, objetos_zoom, w, h)

    return img_out_path, label_out_path


def row_com_zoomout_salvo(row, zoom_pct=None):
    """Retorna uma cópia da linha apontando para a imagem/label com zoom out salva."""
    pct = int(obter_zoom_out_pct() if zoom_pct is None else zoom_pct)
    if pct <= 0:
        return row

    # Se a linha já aponta para uma cópia gerada, não gera outra.
    p_atual = Path(str(row['path']))
    if '__zoomout_' in p_atual.stem:
        return row

    img_out_path, label_out_path = criar_imagem_zoomout_salva(row, pct, sobrescrever=False)
    if img_out_path is None:
        return row

    new_row = row.copy()
    new_row['orig_path'] = str(row['path'])
    new_row['orig_label_path'] = str(row.get('label_path', ''))
    new_row['path'] = str(img_out_path)
    new_row['label_path'] = str(label_out_path)
    new_row['arquivo'] = img_out_path.name
    new_row['stem'] = img_out_path.stem
    new_row['zoom_out_aplicado_salvo'] = True
    new_row['zoom_out_pct'] = pct
    return new_row


def aplicar_zoomout_salvo_df(df_split, split):
    """Substitui linhas marcadas por linhas que apontam para cópias com zoom out já salvas."""
    if df_split is None or len(df_split) == 0:
        return df_split
    pct = int(obter_zoom_out_pct())
    if pct <= 0:
        return df_split.copy()

    chaves = carregar_chaves_zoomout(split)
    if not chaves:
        return df_split.copy()

    rows = []
    for _, row in df_split.iterrows():
        if imagem_tem_zoomout_path(row['path'], split):
            rows.append(row_com_zoomout_salvo(row, pct))
        else:
            rows.append(row.copy())
    return pd.DataFrame(rows).reset_index(drop=True)


def salvar_dataset_zoomout_para_selecoes(df=None, splits=('train', 'test'), zoom_pct=None, sobrescrever=False):
    """Gera/atualiza no disco as cópias com zoom out das imagens marcadas nos JSONs."""
    if df is None:
        df = globals().get('STATE', {}).get('df_dataset', pd.DataFrame())
    if df is None or len(df) == 0:
        return []

    pct = int(obter_zoom_out_pct() if zoom_pct is None else zoom_pct)
    gerados = []
    if pct <= 0:
        salvar_json(ZOOMOUT_MANIFEST_PATH, {'zoom_out_pct': pct, 'gerados': [], 'atualizado_em': time.strftime('%Y-%m-%d %H:%M:%S')})
        return gerados

    for split in splits:
        chaves = carregar_chaves_zoomout(split)
        if not chaves:
            continue
        df_split = df[df['split'] == split].copy()
        for _, row in df_split.iterrows():
            if imagem_tem_zoomout_path(row['path'], split):
                img_out, lab_out = criar_imagem_zoomout_salva(row, pct, sobrescrever=sobrescrever)
                if img_out is not None:
                    gerados.append({
                        'split': split,
                        'orig_path': str(row['path']),
                        'orig_label_path': str(row.get('label_path', '')),
                        'zoom_img_path': str(img_out),
                        'zoom_label_path': str(lab_out),
                        'zoom_out_pct': pct
                    })

    salvar_json(ZOOMOUT_MANIFEST_PATH, {
        'zoom_out_pct': pct,
        'n_gerados': len(gerados),
        'gerados': gerados,
        'atualizado_em': time.strftime('%Y-%m-%d %H:%M:%S')
    })
    return gerados


def ler_imagem_row_com_zoomout(row, retornar_info=False):
    """Lê a imagem. Se a linha original foi marcada, usa a cópia salva com zoom out."""
    if imagem_tem_zoomout_path(row['path'], row.get('split')) and int(obter_zoom_out_pct()) > 0:
        row = row_com_zoomout_salvo(row)
    img = ler_imagem_bgr(row['path'])
    aplicar = bool(row.get('zoom_out_aplicado_salvo', False)) or ('__zoomout_' in Path(str(row['path'])).stem)
    info = {'zoom_out_aplicado': bool(aplicar), 'zoom_out_pct': int(row.get('zoom_out_pct', obter_zoom_out_pct() if aplicar else 0))}
    return (img, info) if retornar_info else img


def carregar_imagem_e_bboxes_row(row):
    """Lê imagem e bboxes YOLO. Se marcada, usa a cópia salva com zoom out e label já ajustada."""
    if imagem_tem_zoomout_path(row['path'], row.get('split')) and int(obter_zoom_out_pct()) > 0:
        row = row_com_zoomout_salvo(row)

    img = ler_imagem_bgr(row['path'])
    h, w = img.shape[:2]
    boxes_raw = ler_bboxes_yolo(row['label_path'], w, h)
    boxes = [b['bbox'] for b in boxes_raw]

    aplicar = bool(row.get('zoom_out_aplicado_salvo', False)) or ('__zoomout_' in Path(str(row['path'])).stem)
    return img, boxes, {'zoom_out_aplicado': bool(aplicar), 'zoom_out_pct': int(row.get('zoom_out_pct', obter_zoom_out_pct() if aplicar else 0))}


def aplicar_selecao_salva(df, split, usar_selecao=True):
    """Filtra o split usando a seleção manual persistente.

    Quando `usar_selecao=True`, os JSONs de seleção comandam quais imagens entram
    em treino/teste. Se o JSON ainda não existir, usa o split completo como fallback
    inicial. Se o JSON existir, mas nenhuma imagem casar com os nomes atuais, retorna
    vazio para deixar claro que a curadoria precisa ser revisada.
    """
    df_split = df[df['split'] == split].copy()
    if not usar_selecao:
        return aplicar_zoomout_salvo_df(df_split, split)

    sel_path = SELECAO_TREINO_PATH if split == 'train' else SELECAO_TESTE_PATH
    if not Path(sel_path).exists():
        return aplicar_zoomout_salvo_df(df_split, split)

    chaves = carregar_chaves_selecao(sel_path)
    if not chaves:
        return df_split.iloc[0:0].copy()

    mask = df_split.apply(
        lambda r: (str(r['path']) in chaves) or (r['arquivo'] in chaves) or (r['stem'] in chaves),
        axis=1
    )
    df_sel = df_split[mask].copy()
    return aplicar_zoomout_salvo_df(df_sel, split)


def limitar_df(df, max_n, random_state=42):
    max_n = int(max_n)
    if max_n <= 0 or len(df) <= max_n:
        return df
    return df.sample(n=max_n, random_state=int(random_state)).copy()


def gerar_amostras_de_split(df_split, p, split_nome='train', status_callback=None):
    """
    Gera amostras positivas e negativas a partir de um split YOLO.

    Positivos: crops das bboxes anotadas.
    Negativos: crops aleatórios com IoU baixo em relação às bboxes.
    """
    rng = np.random.default_rng(int(p['random_state']) + (0 if split_nome == 'train' else 999))

    X = []
    y = []
    erros = []
    info = []

    neg_por_pos = int(p['negativos_por_positivo'])
    neg_iou_max = float(p['neg_iou_max'])
    pos_margin = float(p['pos_margin_pct'])
    neg_attempts = int(p['neg_attempts'])

    # Tamanho reserva para negativos em imagens sem objeto.
    wh_positivos = []

    total = len(df_split)
    for k, (_, row) in enumerate(df_split.iterrows(), start=1):
        if status_callback is not None and (k == 1 or k % 20 == 0 or k == total):
            status_callback(f'{split_nome}: gerando amostras {k}/{total}')

        try:
            img, boxes, zoom_info = carregar_imagem_e_bboxes_row(row)
            h, w = img.shape[:2]

            for bbox in boxes:
                bbox_pos = expandir_bbox(bbox, pos_margin, w, h)
                crop = recortar_bbox(img, bbox_pos)
                if crop.size == 0:
                    continue
                X.append(extrair_hog_de_imagem(crop, p))
                y.append(1)
                info.append({'split': split_nome, 'tipo': 'pos', 'arquivo': row['arquivo'], 'bbox': bbox_pos, **zoom_info})
                wh_positivos.append((bbox_pos[2] - bbox_pos[0], bbox_pos[3] - bbox_pos[1]))

            # Negativos proporcionais aos positivos da imagem.
            # Se a imagem não tiver objetos, gera pelo menos 1 negativo com tamanho mediano.
            n_base = max(1, len(boxes))
            n_neg = neg_por_pos * n_base

            if wh_positivos:
                median_w = int(np.median([v[0] for v in wh_positivos]))
                median_h = int(np.median([v[1] for v in wh_positivos]))
            else:
                median_w = max(24, int(w * 0.12))
                median_h = max(48, int(h * 0.25))

            for _ in range(n_neg):
                # Usa tamanho parecido com caixas reais quando possível.
                if boxes:
                    ref = boxes[int(rng.integers(0, len(boxes)))]
                    target_w = ref[2] - ref[0]
                    target_h = ref[3] - ref[1]
                else:
                    target_w = median_w
                    target_h = median_h

                # pequena variação de escala para aumentar diversidade.
                scale = float(rng.uniform(0.80, 1.25))
                tw = int(target_w * scale)
                th = int(target_h * scale)

                escolhido = None
                for _try in range(neg_attempts):
                    cand = bbox_aleatorio(w, h, tw, th, rng)
                    if not boxes or max(iou_bbox(cand, b) for b in boxes) <= neg_iou_max:
                        escolhido = cand
                        break

                if escolhido is None:
                    continue

                crop = recortar_bbox(img, escolhido)
                if crop.size == 0:
                    continue
                X.append(extrair_hog_de_imagem(crop, p))
                y.append(0)
                info.append({'split': split_nome, 'tipo': 'neg', 'arquivo': row['arquivo'], 'bbox': escolhido, **zoom_info})

        except Exception as e:
            erros.append((row.get('path', ''), str(e)))

    if len(X) == 0:
        return np.empty((0, 0), dtype=np.float32), np.array([], dtype=int), erros, info

    return np.vstack(X), np.array(y, dtype=int), erros, info


def preparar_dados_treino_teste(df, p, status_callback=None):
    """Prepara X_train, X_test, y_train, y_test usando train/test reais quando possível."""
    usar_selecao = bool(p.get('usar_selecao_salva', True))
    rs = int(p['random_state'])

    df_train = aplicar_selecao_salva(df, 'train', usar_selecao=usar_selecao)
    df_test = aplicar_selecao_salva(df, 'test', usar_selecao=usar_selecao)

    df_train = limitar_df(df_train, int(p['max_train_images']), rs)
    df_test = limitar_df(df_test, int(p['max_test_images']), rs + 1)

    X_train_all, y_train_all, erros_train, info_train = gerar_amostras_de_split(
        df_train, p, split_nome='train', status_callback=status_callback
    )

    X_test_real, y_test_real, erros_test, info_test = gerar_amostras_de_split(
        df_test, p, split_nome='test', status_callback=status_callback
    )

    erros = erros_train + erros_test

    if len(y_train_all) < 4 or len(np.unique(y_train_all)) < 2:
        raise RuntimeError(
            'Amostras de treino insuficientes. Verifique se train/labels possui anotações YOLO válidas.'
        )

    usar_test_real = bool(p.get('usar_test_real', True))
    if usar_test_real and len(y_test_real) >= 2 and len(np.unique(y_test_real)) == 2:
        X_train, y_train = X_train_all, y_train_all
        X_test, y_test = X_test_real, y_test_real
        modo_eval = 'pasta_test'
    else:
        test_size = float(p['test_size'])
        X_train, X_test, y_train, y_test = train_test_split(
            X_train_all, y_train_all,
            test_size=test_size,
            random_state=rs,
            stratify=y_train_all
        )
        modo_eval = 'holdout_train'

    stats = {
        'modo_eval': modo_eval,
        'n_img_train': int(len(df_train)),
        'n_img_test': int(len(df_test)),
        'n_train': int(len(y_train)),
        'n_test': int(len(y_test)),
        'n_total': int(len(y_train) + len(y_test)),
        'n_pos_train': int((y_train == 1).sum()),
        'n_neg_train': int((y_train == 0).sum()),
        'n_pos_test': int((y_test == 1).sum()),
        'n_neg_test': int((y_test == 0).sum()),
        'n_pos_total': int((np.concatenate([y_train, y_test]) == 1).sum()),
        'n_neg_total': int((np.concatenate([y_train, y_test]) == 0).sum()),
        'n_erros': int(len(erros)),
    }

    return X_train, X_test, y_train, y_test, stats, erros, info_train + info_test


def assinatura_dataset(df):
    """Cria assinatura do dataset considerando imagens e arquivos de label."""
    if df is None or len(df) == 0:
        return 'dataset_vazio'

    rows = []
    for _, row in df.sort_values(['split', 'path']).iterrows():
        item = {
            'split': row.get('split'),
            'path': str(row.get('path')),
            'label_path': str(row.get('label_path')),
            'n_obj': int(row.get('n_obj', 0)),
        }
        for key in ['path', 'label_path']:
            pp = Path(item[key])
            if pp.exists():
                st = pp.stat()
                item[f'{key}_size'] = int(st.st_size)
                item[f'{key}_mtime'] = float(st.st_mtime)
            else:
                item[f'{key}_size'] = None
                item[f'{key}_mtime'] = None
        rows.append(item)

    # Também inclui os JSONs opcionais de seleção/zoom out, se existirem.
    for sel in [SELECAO_TREINO_PATH, SELECAO_TESTE_PATH, ZOOMOUT_TREINO_PATH, ZOOMOUT_TESTE_PATH, ZOOMOUT_CONFIG_PATH, ZOOMOUT_MANIFEST_PATH]:
        if sel.exists():
            st = sel.stat()
            rows.append({'selection_json': str(sel), 'size': int(st.st_size), 'mtime': float(st.st_mtime)})

    payload = json.dumps(rows, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def assinatura_treinamento(df, p):
    """Assinatura completa: dataset + parâmetros."""
    payload = {
        'versao_notebook': 'cilindros_hough_hog_svm_cylinders_yolo_v4_zoomout_salvo',
        'dataset': assinatura_dataset(df),
        'params': p,
        'zoom_out_pct': obter_zoom_out_pct()
    }
    payload_str = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload_str.encode('utf-8')).hexdigest()


def carregar_historico():
    if HISTORY_PATH.exists():
        return pd.read_csv(HISTORY_PATH)
    return pd.DataFrame()


def salvar_linha_historico(row):
    hist = carregar_historico()
    hist = pd.concat([hist, pd.DataFrame([row])], ignore_index=True)
    HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
    hist.to_csv(HISTORY_PATH, index=False)
    return hist


def abreviar_float(x, nd=4):
    try:
        return round(float(x), nd)
    except Exception:
        return x


In [ ]:
# ============================================================
# 6. Estado compartilhado e atualização automática de visualizações
# ============================================================

STATE = {
    'data_path': DATASET_DIR,
    'df_dataset': pd.DataFrame(),
    'ref_path': None,
    'params': OrderedDict(),
    'renderers': OrderedDict(),
    'refresh_em_andamento': False,
    'setup_id': 'PADRAO',
    'setup_label': 'Setup padrão',
    'setup_modificado': False,
    'loading_setup': False,
    'setup_dropdown_updating': False,
    'zoom_out_pct': int(carregar_zoomout_config().get('zoom_out_pct', 0)),
    'validacao_random_seed': 42
}


def registrar_renderer(nome, out_widget, funcao_renderer):
    """Registra uma visualização posterior para atualização automática."""
    STATE['renderers'][nome] = {
        'out': out_widget,
        'fn': funcao_renderer
    }
    atualizar_renderer(nome)


def atualizar_renderer(nome):
    """Atualiza uma visualização específica."""
    if nome not in STATE['renderers']:
        return

    item = STATE['renderers'][nome]
    out = item['out']
    fn = item['fn']

    with out:
        clear_output(wait=True)
        try:
            fn()
        except Exception as e:
            print(f"Erro ao atualizar '{nome}': {e}")
            traceback.print_exc(limit=1)


def atualizar_todas_visualizacoes(*args, **kwargs):
    """Atualiza todas as células posteriores que registraram visualização."""
    if STATE['refresh_em_andamento']:
        return

    STATE['refresh_em_andamento'] = True

    try:
        for nome in list(STATE['renderers'].keys()):
            atualizar_renderer(nome)
    finally:
        STATE['refresh_em_andamento'] = False


def obter_imagem_referencia():
    ref_path = STATE.get('ref_path')
    if ref_path is None:
        return None

    df = STATE.get('df_dataset', pd.DataFrame())
    if df is not None and len(df) > 0:
        row_df = df[df['path'].astype(str) == str(ref_path)]
        if len(row_df) > 0:
            return ler_imagem_row_com_zoomout(row_df.iloc[0])

    return ler_imagem_bgr(ref_path)


def obter_preprocessamento_referencia():
    img = obter_imagem_referencia()
    if img is None:
        return None
    return aplicar_preprocessamento(img, STATE['params'])


def definir_params_a_partir_widgets(widgets_param):
    """Copia os valores dos widgets para STATE['params']."""
    p = OrderedDict()
    for nome, wid in widgets_param.items():
        p[nome] = wid.value
    STATE['params'] = p


In [ ]:
# ============================================================
# 7. Seleção manual persistente antes dos ajustes paramétricos
# ============================================================

# Esta célula deve ser executada antes da célula de parâmetros.
# Ela carrega o dataset, mantém a curadoria manual de treino/teste,
# permite marcar imagens que terão cópias salvas com zoom out artificial e restringe
# a imagem de referência às imagens selecionadas.

w_data_path = widgets.Text(
    value=str(DATASET_DIR),
    description='Dataset:',
    layout=widgets.Layout(width='90%')
)

btn_recarregar_dataset = widgets.Button(
    description='Recarregar dataset',
    button_style='info',
    icon='refresh',
    layout=widgets.Layout(width='200px')
)

w_ref_image = widgets.Dropdown(
    options=[],
    description='Referência:',
    layout=widgets.Layout(width='90%')
)

# Percentual global aplicado às imagens marcadas para gerar cópias com zoom out.
_zoom_cfg = carregar_zoomout_config()
w_zoom_out_pct = widgets.IntSlider(
    value=int(_zoom_cfg.get('zoom_out_pct', 0)),
    min=0,
    max=80,
    step=5,
    description='zoom out %',
    continuous_update=False,
    layout=widgets.Layout(width='420px'),
    style={'description_width': '95px'}
)
STATE['zoom_out_pct'] = int(w_zoom_out_pct.value)

THUMB_CACHE = {}
GRID_SELECTION_STATE = {
    'select_checkboxes': {},
    'zoom_checkboxes': {},
    'paths': [],
    'page': 1,
    'visible_paths': set(),
}

PAGE_SIZE_CURADORIA = 200

split_curadoria_widget = widgets.Dropdown(
    options=[('Treino', 'train'), ('Teste', 'test')],
    value='train',
    description='conjunto:',
    layout=widgets.Layout(width='220px')
)

btn_exibir_grade = widgets.Button(
    description='Exibir/atualizar grade',
    button_style='info',
    icon='th',
    layout=widgets.Layout(width='210px')
)

btn_pagina_anterior = widgets.Button(
    description='◀ Página anterior',
    button_style='',
    layout=widgets.Layout(width='160px')
)

w_pagina_curadoria = widgets.BoundedIntText(
    value=1,
    min=1,
    max=1,
    step=1,
    description='página:',
    layout=widgets.Layout(width='145px'),
    style={'description_width': '55px'}
)

lbl_total_paginas = widgets.HTML('<span style="opacity:.8;">/ 1</span>')

btn_pagina_proxima = widgets.Button(
    description='Próxima página ▶',
    button_style='',
    layout=widgets.Layout(width='160px')
)

btn_salvar_grade = widgets.Button(
    description='Salvar seleção',
    button_style='success',
    icon='save',
    layout=widgets.Layout(width='160px')
)

btn_marcar_todas_visiveis = widgets.Button(
    description='Marcar todas',
    button_style='',
    layout=widgets.Layout(width='135px')
)

btn_desmarcar_todas_visiveis = widgets.Button(
    description='Desmarcar todas',
    button_style='warning',
    layout=widgets.Layout(width='150px')
)

btn_limpar_zoom_visivel = widgets.Button(
    description='Limpar zoom',
    button_style='',
    layout=widgets.Layout(width='125px')
)

status_curadoria = widgets.HTML('')
out_curadoria_resumo = widgets.Output()
out_ref_preview = widgets.Output(layout=widgets.Layout(width='100%'))
out_zoom_preview = widgets.Output(layout=widgets.Layout(width='100%'))
out_grade_curadoria = widgets.Output()


def _nome_split_curadoria(split):
    return 'treino' if split == 'train' else 'teste'


def _json_split_curadoria(split):
    return SELECAO_TREINO_PATH if split == 'train' else SELECAO_TESTE_PATH


def _json_zoomout_curadoria(split):
    return ZOOMOUT_TREINO_PATH if split == 'train' else ZOOMOUT_TESTE_PATH



def _salvar_zoomout_split(split, paths):
    """Salva múltiplas imagens marcadas para gerar cópias com zoom out no split."""
    paths = [str(p) for p in paths]
    salvar_json(_json_zoomout_curadoria(split), paths)


def _chaves_zoomout_split(split):
    return carregar_chaves_zoomout(split)


def _lista_zoomout_split(split):
    return carregar_lista_zoomout(split)


def _df_split_completo(split):
    df = STATE.get('df_dataset', pd.DataFrame())
    if df is None or len(df) == 0:
        return pd.DataFrame()
    return df[df['split'] == split].copy().sort_values('arquivo').reset_index(drop=True)


def _chaves_selecionadas_split(split):
    return carregar_chaves_selecao(_json_split_curadoria(split))


def _chaves_zoomout_split(split):
    return carregar_chaves_zoomout(split)


def _linha_esta_em_chaves(row, chaves):
    return (
        str(row['path']) in chaves
        or str(Path(row['path'])) in chaves
        or row['arquivo'] in chaves
        or row['stem'] in chaves
    )


def _linha_esta_selecionada(row, chaves):
    return _linha_esta_em_chaves(row, chaves)


def _linha_esta_zoomout(row, chaves):
    return _linha_esta_em_chaves(row, chaves)


def df_por_selecao_persistente(df, split):
    """Retorna apenas as imagens selecionadas no JSON persistente daquele split."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=df.columns if df is not None else [])
    return aplicar_selecao_salva(df, split, usar_selecao=True)


def df_referencia_por_selecao(df):
    """A imagem de referência deve vir somente da seleção manual de treino/teste."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=df.columns if df is not None else [])

    partes = []
    for split in ['train', 'test']:
        sel = df_por_selecao_persistente(df, split)
        if sel is not None and len(sel) > 0:
            partes.append(sel)

    if not partes:
        return pd.DataFrame(columns=df.columns)

    return pd.concat(partes, ignore_index=True).sort_values(['split', 'arquivo']).reset_index(drop=True)


def _thumb_bytes(img_path, size=96):
    img_path = str(img_path)
    if img_path in THUMB_CACHE:
        return THUMB_CACHE[img_path]

    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    escala = size / max(h, w)
    novo_w = max(1, int(w * escala))
    novo_h = max(1, int(h * escala))
    thumb = cv2.resize(img, (novo_w, novo_h), interpolation=cv2.INTER_AREA)

    canvas = np.full((size, size, 3), 245, dtype=np.uint8)
    y0 = (size - novo_h) // 2
    x0 = (size - novo_w) // 2
    canvas[y0:y0 + novo_h, x0:x0 + novo_w] = thumb

    ok, buf = cv2.imencode('.jpg', cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 82])
    if not ok:
        return None

    data = bytes(buf)
    THUMB_CACHE[img_path] = data
    return data


def _criar_card_curadoria(row, marcada, zoom_marcado):
    path = row['path']

    thumb = _thumb_bytes(path)
    if thumb is None:
        img_widget = widgets.HTML(
            "<div style='width:96px;height:96px;display:flex;align-items:center;justify-content:center;border:1px solid #777;'>erro</div>"
        )
    else:
        img_widget = widgets.Image(
            value=thumb,
            format='jpg',
            layout=widgets.Layout(width='96px', height='96px')
        )

    cb_select = widgets.Checkbox(
        value=bool(marcada),
        description='',
        indent=False,
        layout=widgets.Layout(width='24px', height='22px', margin='0')
    )
    cb_zoom = widgets.Checkbox(
        value=bool(zoom_marcado),
        description='',
        indent=False,
        layout=widgets.Layout(width='24px', height='22px', margin='0')
    )

    path_str = str(path)
    GRID_SELECTION_STATE['select_checkboxes'][path_str] = cb_select
    GRID_SELECTION_STATE['zoom_checkboxes'][path_str] = cb_zoom

    def _on_select_change(change):
        if change.get('name') == 'value':
            renderizar_preview_zoom_out()

    def _on_zoom_change(change):
        if change.get('name') != 'value':
            return
        # A miniatura da grade permanece original. Apenas a prévia abaixo é recalculada.
        renderizar_preview_zoom_out()

    cb_select.observe(_on_select_change, names='value')
    cb_zoom.observe(_on_zoom_change, names='value')

    labels = widgets.HTML(
        "<div style='font-size:9px; line-height:11px; width:96px; display:flex; justify-content:space-around; opacity:.85;'>"
        "<span>seleção</span><span>zoom out</span></div>"
    )

    card_border = '2px solid #2a7' if marcada else '1px solid #555'
    if zoom_marcado:
        card_border = '2px solid #d98200'

    return widgets.VBox(
        [
            img_widget,
            widgets.HBox(
                [cb_select, cb_zoom],
                layout=widgets.Layout(justify_content='space-around', width='96px', height='24px')
            ),
            labels
        ],
        layout=widgets.Layout(
            width='112px',
            min_width='112px',
            max_width='112px',
            height='142px',
            overflow='hidden',
            align_items='center',
            border=card_border,
            border_radius='6px',
            padding='3px',
            margin='2px'
        )
    )


def _atualizar_dropdown_referencia():
    df = STATE.get('df_dataset', pd.DataFrame())
    df_ref = df_referencia_por_selecao(df)

    opcoes = []
    for _, row in df_ref.iterrows():
        opcoes.append((f"{row['split']} | {row['arquivo']} [{row['n_obj']} obj]", row['path']))

    atual = STATE.get('ref_path')
    valores = [v for _, v in opcoes]
    w_ref_image.options = opcoes

    if valores:
        if atual in valores:
            w_ref_image.value = atual
        else:
            STATE['ref_path'] = valores[0]
            w_ref_image.value = valores[0]
    else:
        STATE['ref_path'] = None
        try:
            w_ref_image.value = None
        except Exception:
            pass


def atualizar_resumo_curadoria(mensagem=''):
    with out_curadoria_resumo:
        clear_output(wait=True)

        df = STATE.get('df_dataset', pd.DataFrame())
        if df is None or len(df) == 0:
            print('Dataset ainda não carregado.')
            return

        train_sel = df_por_selecao_persistente(df, 'train')
        test_sel = df_por_selecao_persistente(df, 'test')
        df_ref = df_referencia_por_selecao(df)
        z_train = len(carregar_lista_zoomout('train'))
        z_test = len(carregar_lista_zoomout('test'))

        if mensagem:
            print(mensagem)
            print()

        print(resumo_dataset(df))
        print()
        print('Curadoria persistente em uso:')
        print(f'- treino selecionado: {len(train_sel)} imagens | JSON: {SELECAO_TREINO_PATH.exists()}')
        print(f'- teste selecionado : {len(test_sel)} imagens | JSON: {SELECAO_TESTE_PATH.exists()}')
        print(f'- imagens marcadas para zoom out: treino={z_train} | teste={z_test} | percentual={STATE.get("zoom_out_pct", 0)}%')
        print(f'- referências disponíveis: {len(df_ref)} imagens selecionadas')
        print()
        print('Arquivos JSON:')
        print('-', SELECAO_TREINO_PATH)
        print('-', SELECAO_TESTE_PATH)
        print('-', ZOOMOUT_TREINO_PATH)
        print('-', ZOOMOUT_TESTE_PATH)
        print('-', ZOOMOUT_CONFIG_PATH)
        if len(df_ref) == 0:
            print('\nAtenção: nenhuma imagem selecionada foi encontrada. Marque imagens na grade e salve a seleção.')


def renderizar_preview_referencia():
    """Função mantida apenas por compatibilidade; a prévia duplicada da referência não é exibida."""
    with out_ref_preview:
        clear_output(wait=True)


def _row_para_preview_zoom():
    # Prioridade: primeira imagem marcada com zoom na grade atual.
    for path, cb in GRID_SELECTION_STATE.get('zoom_checkboxes', {}).items():
        if cb.value:
            df = STATE.get('df_dataset', pd.DataFrame())
            row_df = df[df['path'].astype(str) == str(path)]
            if len(row_df) > 0:
                return row_df.iloc[0]

    # Depois, primeira imagem marcada para zoom out salva no split atual ou no outro split.
    df = STATE.get('df_dataset', pd.DataFrame())
    if df is not None and len(df) > 0:
        for sp in [split_curadoria_widget.value, 'train', 'test']:
            for zoom_path in carregar_lista_zoomout(sp):
                pz = Path(str(zoom_path))
                row_df = df[
                    (df['path'].astype(str) == str(zoom_path))
                    | (df['arquivo'].astype(str) == pz.name)
                    | (df['stem'].astype(str) == pz.stem)
                ]
                if len(row_df) > 0:
                    return row_df.iloc[0]

    # Depois, primeira imagem selecionada na grade atual.
    for path, cb in GRID_SELECTION_STATE.get('select_checkboxes', {}).items():
        if cb.value:
            df = STATE.get('df_dataset', pd.DataFrame())
            row_df = df[df['path'].astype(str) == str(path)]
            if len(row_df) > 0:
                return row_df.iloc[0]

    # Por fim, imagem de referência atual ou primeira imagem do split.
    ref_path = STATE.get('ref_path')
    df = STATE.get('df_dataset', pd.DataFrame())
    if ref_path and df is not None and len(df) > 0:
        row_df = df[df['path'].astype(str) == str(ref_path)]
        if len(row_df) > 0:
            return row_df.iloc[0]

    df_split = _df_split_completo(split_curadoria_widget.value)
    if len(df_split) > 0:
        return df_split.iloc[0]

    return None


def renderizar_preview_zoom_out(*args, **kwargs):
    with out_zoom_preview:
        clear_output(wait=True)

        row = _row_para_preview_zoom()
        if row is None:
            display(HTML('<em>Nenhuma imagem disponível para pré-visualizar o zoom out.</em>'))
            return

        try:
            img_bgr = ler_imagem_bgr(row['path'])
            img_zoom, _ = aplicar_zoom_out_imagem_e_bboxes(img_bgr, [], int(w_zoom_out_pct.value))
            img_rgb = bgr_para_rgb(img_bgr)
            img_zoom_rgb = bgr_para_rgb(img_zoom)
        except Exception as e:
            print('Erro ao gerar preview de zoom out:', e)
            return

        fig, axs = plt.subplots(1, 2, figsize=(5.4, 2.7))
        axs[0].imshow(img_rgb)
        axs[0].set_title('Original', fontsize=9)
        axs[1].imshow(img_zoom_rgb)
        axs[1].set_title(f'Zoom out {int(w_zoom_out_pct.value)}%', fontsize=9)
        for ax in axs:
            ax.axis('off')
        plt.tight_layout()
        plt.show()


def recarregar_dataset(_=None):
    STATE['data_path'] = Path(w_data_path.value).expanduser().resolve()
    STATE['zoom_out_pct'] = int(w_zoom_out_pct.value)
    df = listar_imagens_dataset(STATE['data_path'])
    STATE['df_dataset'] = df

    _atualizar_dropdown_referencia()
    atualizar_resumo_curadoria('Dataset/seleção recarregados.')
    renderizar_preview_zoom_out()
    atualizar_todas_visualizacoes()


def on_ref_image_change(change):
    if change.get('new') is not None:
        STATE['ref_path'] = change['new']
        atualizar_todas_visualizacoes()


def on_zoom_out_pct_change(change):
    if change.get('name') == 'value':
        STATE['zoom_out_pct'] = int(change['new'])
        salvar_zoomout_config(STATE['zoom_out_pct'])
        atualizar_resumo_curadoria('Percentual de zoom out atualizado.')
        renderizar_preview_zoom_out()
        atualizar_todas_visualizacoes()


def _total_paginas_curadoria(split):
    df_split = _df_split_completo(split)
    if df_split is None or len(df_split) == 0:
        return 1
    return max(1, int(np.ceil(len(df_split) / PAGE_SIZE_CURADORIA)))


def _sincronizar_controle_pagina(split=None, manter=True):
    split = split or split_curadoria_widget.value
    total = _total_paginas_curadoria(split)
    pagina_atual = int(GRID_SELECTION_STATE.get('page', 1)) if manter else 1
    pagina_atual = max(1, min(pagina_atual, total))
    GRID_SELECTION_STATE['page'] = pagina_atual
    w_pagina_curadoria.max = total
    w_pagina_curadoria.value = pagina_atual
    lbl_total_paginas.value = f'<span style="opacity:.8;">/ {total}</span>'
    btn_pagina_anterior.disabled = pagina_atual <= 1
    btn_pagina_proxima.disabled = pagina_atual >= total
    return pagina_atual, total


def renderizar_grade_curadoria(_=None):
    split = split_curadoria_widget.value
    df_split = _df_split_completo(split)

    GRID_SELECTION_STATE['select_checkboxes'] = {}
    GRID_SELECTION_STATE['zoom_checkboxes'] = {}
    GRID_SELECTION_STATE['paths'] = []
    GRID_SELECTION_STATE['visible_paths'] = set()

    pagina, total_paginas = _sincronizar_controle_pagina(split, manter=True)
    ini = (pagina - 1) * PAGE_SIZE_CURADORIA
    fim = ini + PAGE_SIZE_CURADORIA
    df_pagina = df_split.iloc[ini:fim].copy() if df_split is not None else pd.DataFrame()

    btn_exibir_grade.disabled = True
    btn_salvar_grade.disabled = True
    status_curadoria.value = "<span style='color:#d98200; font-weight:bold;'>Carregando miniaturas da página...</span>"

    try:
        with out_grade_curadoria:
            clear_output(wait=True)

            if df_split is None or len(df_split) == 0:
                print(f"Nenhuma imagem encontrada no split {_nome_split_curadoria(split)}.")
                return

            chaves_sel = _chaves_selecionadas_split(split)
            chaves_zoom = _chaves_zoomout_split(split)
            cards = []

            for _, row in df_pagina.iterrows():
                marcada = _linha_esta_selecionada(row, chaves_sel)
                zoom_marcado = _linha_esta_zoomout(row, chaves_zoom)
                cards.append(_criar_card_curadoria(row, marcada, zoom_marcado))
                path_str = str(row['path'])
                GRID_SELECTION_STATE['paths'].append(path_str)
                GRID_SELECTION_STATE['visible_paths'].add(path_str)

            n_marcadas_visiveis = sum(cb.value for cb in GRID_SELECTION_STATE['select_checkboxes'].values())
            n_zoom_visivel = sum(cb.value for cb in GRID_SELECTION_STATE['zoom_checkboxes'].values())
            n_zoom_split = len(carregar_lista_zoomout(split))

            display(widgets.HTML(
                f"<b>Curadoria de {_nome_split_curadoria(split)}</b> | "
                f"página {pagina}/{total_paginas} | exibindo {len(df_pagina)} de {len(df_split)} imagens | "
                f"{n_marcadas_visiveis} selecionadas nesta página | "
                f"{n_zoom_visivel} com zoom nesta página | {n_zoom_split} com zoom salvo no split<br>"
                "<small>São exibidas 200 imagens por página. A miniatura permanece original; "
                "a segunda caixa marca imagens para gerar <b>cópias salvas</b> com zoom out aplicado. "
                "Apenas a prévia abaixo muda ao alterar o slider. Clique em <b>Salvar seleção</b> para persistir e gerar as cópias.</small>"
            ))

            grid = widgets.GridBox(
                cards,
                layout=widgets.Layout(
                    display='grid',
                    grid_template_columns='repeat(auto-fill, minmax(116px, 1fr))',
                    grid_gap='6px 4px',
                    width='100%',
                    overflow='visible',
                    align_items='flex-start'
                )
            )
            display(grid)

        renderizar_preview_zoom_out()

    finally:
        btn_exibir_grade.disabled = False
        btn_salvar_grade.disabled = False
        _sincronizar_controle_pagina(split, manter=True)
        status_curadoria.value = ''



def salvar_grade_curadoria(_=None):
    split = split_curadoria_widget.value

    if not GRID_SELECTION_STATE['select_checkboxes']:
        status_curadoria.value = "<span style='color:#b66;'>Exiba a grade antes de salvar alterações.</span>"
        return

    df = STATE.get('df_dataset', pd.DataFrame())
    df_split = _df_split_completo(split)
    visible_paths = set(GRID_SELECTION_STATE.get('visible_paths', set()))

    # Mantém a seleção das outras páginas e atualiza somente a página visível.
    chaves_sel_existentes = _chaves_selecionadas_split(split)
    selecionadas_anteriores = []
    for _, row in df_split.iterrows():
        path_str = str(row['path'])
        if path_str in visible_paths:
            continue
        if _linha_esta_selecionada(row, chaves_sel_existentes):
            selecionadas_anteriores.append(path_str)

    selecionadas_visiveis = [
        path for path, checkbox in GRID_SELECTION_STATE['select_checkboxes'].items()
        if checkbox.value
    ]
    selecionadas = selecionadas_anteriores + selecionadas_visiveis

    # Zoom out: agora podem existir várias imagens marcadas. Mantém outras páginas e atualiza a página visível.
    chaves_zoom_existentes = _chaves_zoomout_split(split)
    zoom_anteriores = []
    for _, row in df_split.iterrows():
        path_str = str(row['path'])
        if path_str in visible_paths:
            continue
        if _linha_esta_zoomout(row, chaves_zoom_existentes):
            zoom_anteriores.append(path_str)

    zoom_visiveis = [
        path for path, checkbox in GRID_SELECTION_STATE['zoom_checkboxes'].items()
        if checkbox.value
    ]
    zoom_marcadas = zoom_anteriores + zoom_visiveis

    salvar_json(_json_split_curadoria(split), selecionadas)
    _salvar_zoomout_split(split, zoom_marcadas)
    salvar_zoomout_config(int(w_zoom_out_pct.value))

    # Gera as cópias com zoom out no dataset derivado. As miniaturas originais da grade não são alteradas.
    status_curadoria.value = "<span style='color:#d98200;'>Salvando seleção e gerando cópias com zoom out...</span>"
    gerados = salvar_dataset_zoomout_para_selecoes(df, splits=('train', 'test'), zoom_pct=int(w_zoom_out_pct.value), sobrescrever=True)

    recarregar_dataset()
    renderizar_grade_curadoria()

    atualizar_resumo_curadoria(
        f"Seleção de {_nome_split_curadoria(split)} salva. "
        f"Total selecionado no split: {len(selecionadas)} imagens. "
        f"Imagens marcadas para zoom out neste split: {len(zoom_marcadas)}. "
        f"Cópias com zoom out geradas/atualizadas: {len(gerados)}."
    )


def marcar_todas_visiveis(_=None):
    for cb in GRID_SELECTION_STATE['select_checkboxes'].values():
        cb.value = True
    renderizar_preview_zoom_out()


def desmarcar_todas_visiveis(_=None):
    for cb in GRID_SELECTION_STATE['select_checkboxes'].values():
        cb.value = False
    renderizar_preview_zoom_out()


def limpar_zoom_visivel(_=None):
    for cb in GRID_SELECTION_STATE['zoom_checkboxes'].values():
        cb.value = False
    status_curadoria.value = "<span style='color:#d98200;'>Zoom removido da página visível. Clique em Salvar seleção para persistir.</span>"
    renderizar_preview_zoom_out()


def ao_mudar_split_curadoria(change):
    if change.get('name') == 'value':
        GRID_SELECTION_STATE['page'] = 1
        _sincronizar_controle_pagina(change.get('new'), manter=False)
        with out_grade_curadoria:
            clear_output(wait=True)
            print("Clique em 'Exibir/atualizar grade' para carregar a curadoria deste conjunto.")
        renderizar_preview_zoom_out()




btn_recarregar_dataset.on_click(recarregar_dataset)
w_ref_image.observe(on_ref_image_change, names='value')
w_zoom_out_pct.observe(on_zoom_out_pct_change, names='value')
btn_exibir_grade.on_click(renderizar_grade_curadoria)
btn_salvar_grade.on_click(salvar_grade_curadoria)
btn_marcar_todas_visiveis.on_click(marcar_todas_visiveis)
btn_desmarcar_todas_visiveis.on_click(desmarcar_todas_visiveis)
btn_limpar_zoom_visivel.on_click(limpar_zoom_visivel)
btn_pagina_anterior.on_click(lambda _: (_sincronizar_controle_pagina(split_curadoria_widget.value, manter=True), setattr(w_pagina_curadoria, 'value', max(1, int(w_pagina_curadoria.value) - 1)), GRID_SELECTION_STATE.update({'page': int(w_pagina_curadoria.value)}), renderizar_grade_curadoria()))
btn_pagina_proxima.on_click(lambda _: (_sincronizar_controle_pagina(split_curadoria_widget.value, manter=True), setattr(w_pagina_curadoria, 'value', min(int(w_pagina_curadoria.max), int(w_pagina_curadoria.value) + 1)), GRID_SELECTION_STATE.update({'page': int(w_pagina_curadoria.value)}), renderizar_grade_curadoria()))

def _on_pagina_manual_change(change):
    if change.get('name') == 'value':
        GRID_SELECTION_STATE['page'] = int(change.get('new', 1))
        with out_grade_curadoria:
            clear_output(wait=True)
            print("Clique em 'Exibir/atualizar grade' para carregar a página selecionada.")

w_pagina_curadoria.observe(_on_pagina_manual_change, names='value')
split_curadoria_widget.observe(ao_mudar_split_curadoria, names='value')

painel_curadoria = widgets.VBox([
    widgets.HTML('<b>1) Seleção manual persistente de imagens</b>'),
    widgets.HTML(
        '<small>Esta célula controla as imagens usadas em treino/teste, as imagens que terão cópias salvas com zoom out artificial e também alimenta a lista de imagem de referência. '
        'As imagens originais do dataset não são sobrescritas; as cópias com zoom out são gravadas no dataset derivado do notebook.</small>'
    ),
    w_data_path,
    widgets.HBox([btn_recarregar_dataset]),
    widgets.HBox(
        [
            split_curadoria_widget,
            btn_exibir_grade,
            btn_salvar_grade,
            btn_marcar_todas_visiveis,
            btn_desmarcar_todas_visiveis,
            btn_limpar_zoom_visivel,
            status_curadoria
        ],
        layout=widgets.Layout(align_items='center', gap='8px', flex_flow='row wrap')
    ),
    widgets.HBox(
        [btn_pagina_anterior, w_pagina_curadoria, lbl_total_paginas, btn_pagina_proxima],
        layout=widgets.Layout(align_items='center', gap='6px', flex_flow='row wrap')
    ),
    widgets.HTML('<b>Zoom out artificial para imagens marcadas</b>'),
    widgets.HTML(
        '<small>Use a segunda caixa de seleção de cada miniatura para marcar imagens que devem ficar aparentemente mais distantes. '
        'A miniatura da grade fica original; somente a prévia abaixo muda em tempo real. Ao salvar, o notebook grava uma cópia com zoom out aplicado e label YOLO ajustada.</small>'
    ),
    w_zoom_out_pct,
    out_zoom_preview,
    out_curadoria_resumo,
    widgets.HTML(
        '<small>A imagem de referência será escolhida abaixo da seção de parâmetros, '
        'usando somente as imagens marcadas nesta curadoria.</small>'
    ),
    out_grade_curadoria
])

display(painel_curadoria)

# Carrega dataset e seleção automaticamente. Não exibe listas de imagens e não treina.
recarregar_dataset()


## Ajuste interativo dos parâmetros, imagem de referência e visualizações

A célula abaixo reúne os controles de pré-processamento, Canny, morfologia, Hough, pareamento geométrico e HOG/SVM. A lista da imagem de referência continua sendo alimentada pela seleção manual feita na célula anterior.

As visualizações de pré-processamento, Hough, ROIs geométricas e HOG ficam juntas nesta mesma célula de ajuste para facilitar a comparação imediata dos efeitos de cada parâmetro.

Nesta revisão, a visualização de pré-processamento mostra explicitamente o **Mapa A** e o **Mapa B** para facilitar o ajuste da filtragem antes da morfologia.

Nesta revisão, a remoção de sólidos pós-morfologia usa também uma **largura/espessura mínima**: componentes finos são preservados como linhas, mesmo quando possuem área ou preenchimento altos.


In [ ]:
# ============================================================
# 8. Ajuste interativo de parâmetros + setups salvos + visualização da referência
# ============================================================

PARAM_WIDGETS = OrderedDict()

# Pré-processamento
PARAM_WIDGETS['clahe_ativo'] = widgets.Checkbox(value=True, description='Usar CLAHE')
PARAM_WIDGETS['clahe_clip'] = widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1, description='clip', continuous_update=False)
PARAM_WIDGETS['clahe_grid'] = widgets.IntSlider(value=8, min=2, max=16, step=1, description='grid', continuous_update=False)

PARAM_WIDGETS['bilateral_d'] = widgets.IntSlider(value=5, min=1, max=15, step=2, description='d', continuous_update=False)
PARAM_WIDGETS['bilateral_sigma_color'] = widgets.IntSlider(value=50, min=5, max=150, step=5, description='sigmaCor', continuous_update=False)
PARAM_WIDGETS['bilateral_sigma_space'] = widgets.IntSlider(value=50, min=5, max=150, step=5, description='sigmaEsp', continuous_update=False)

# Canny
PARAM_WIDGETS['canny_sigma'] = widgets.FloatSlider(value=0.33, min=0.05, max=0.90, step=0.01, description='sigma', continuous_update=False)
PARAM_WIDGETS['canny_aperture'] = widgets.Dropdown(options=[3, 5, 7], value=3, description='abertura')
PARAM_WIDGETS['canny_l2gradient'] = widgets.Checkbox(value=True, description='L2gradient')

# Filtro de pequenos componentes antes da morfologia/Hough
PARAM_WIDGETS['edge_filter_ativo'] = widgets.Checkbox(value=True, description='Filtrar traços curtos')
PARAM_WIDGETS['edge_min_area'] = widgets.IntSlider(value=12, min=0, max=200, step=1, description='área mín traço', continuous_update=False)
PARAM_WIDGETS['edge_min_extent'] = widgets.IntSlider(value=10, min=0, max=120, step=1, description='extensão mín traço', continuous_update=False)

# Morfologia
PARAM_WIDGETS['close_kernel_w'] = widgets.IntSlider(value=3, min=1, max=21, step=2, description='close W', continuous_update=False)
PARAM_WIDGETS['close_kernel_h'] = widgets.IntSlider(value=9, min=1, max=31, step=2, description='close H', continuous_update=False)
PARAM_WIDGETS['close_iter'] = widgets.IntSlider(value=1, min=0, max=5, step=1, description='close it', continuous_update=False)

PARAM_WIDGETS['open_kernel_w'] = widgets.IntSlider(value=3, min=1, max=15, step=2, description='open W', continuous_update=False)
PARAM_WIDGETS['open_kernel_h'] = widgets.IntSlider(value=3, min=1, max=15, step=2, description='open H', continuous_update=False)
PARAM_WIDGETS['open_iter'] = widgets.IntSlider(value=1, min=0, max=5, step=1, description='open it', continuous_update=False)

# Remoção de regiões sólidas geradas pelo fechamento por erosão forte
PARAM_WIDGETS['solid_filter_ativo'] = widgets.Checkbox(value=True, description='Remover sólidos por erosão')
PARAM_WIDGETS['solid_erode_kernel_w'] = widgets.IntSlider(value=9, min=1, max=41, step=2, description='erode W', continuous_update=False)
PARAM_WIDGETS['solid_erode_kernel_h'] = widgets.IntSlider(value=9, min=1, max=41, step=2, description='erode H', continuous_update=False)
PARAM_WIDGETS['solid_erode_iter'] = widgets.IntSlider(value=1, min=0, max=6, step=1, description='erode it', continuous_update=False)
PARAM_WIDGETS['solid_min_area'] = widgets.IntSlider(value=80, min=0, max=5000, step=10, description='área mín núcleo', continuous_update=False)
PARAM_WIDGETS['solid_dilate_kernel_w'] = widgets.IntSlider(value=5, min=1, max=41, step=2, description='dilata W', continuous_update=False)
PARAM_WIDGETS['solid_dilate_kernel_h'] = widgets.IntSlider(value=5, min=1, max=41, step=2, description='dilata H', continuous_update=False)
PARAM_WIDGETS['solid_dilate_iter'] = widgets.IntSlider(value=1, min=0, max=6, step=1, description='dilata it', continuous_update=False)

# Hough
PARAM_WIDGETS['hough_rho'] = widgets.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.5, description='rho', continuous_update=False)
PARAM_WIDGETS['hough_theta_deg'] = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5, description='theta°', continuous_update=False)
PARAM_WIDGETS['hough_threshold'] = widgets.IntSlider(value=35, min=5, max=150, step=5, description='threshold', continuous_update=False)
PARAM_WIDGETS['hough_min_line_length'] = widgets.IntSlider(value=45, min=5, max=300, step=5, description='minLen', continuous_update=False)
PARAM_WIDGETS['hough_max_line_gap'] = widgets.IntSlider(value=12, min=0, max=80, step=2, description='maxGap', continuous_update=False)

# Filtro geométrico
PARAM_WIDGETS['line_length_min'] = widgets.IntSlider(value=45, min=5, max=400, step=5, description='L min', continuous_update=False)
PARAM_WIDGETS['line_length_max'] = widgets.IntSlider(value=260, min=20, max=700, step=10, description='L max', continuous_update=False)
PARAM_WIDGETS['pair_angle_tol_deg'] = widgets.FloatSlider(value=10.0, min=1.0, max=30.0, step=0.5, description='ang tol', continuous_update=False)
PARAM_WIDGETS['pair_dist_min'] = widgets.IntSlider(value=20, min=5, max=200, step=5, description='dist min', continuous_update=False)
PARAM_WIDGETS['pair_dist_max'] = widgets.IntSlider(value=90, min=10, max=300, step=5, description='dist max', continuous_update=False)
PARAM_WIDGETS['pair_overlap_min'] = widgets.FloatSlider(value=0.45, min=0.05, max=1.00, step=0.05, description='overlap', continuous_update=False)
PARAM_WIDGETS['roi_aspect_min'] = widgets.FloatSlider(value=1.2, min=0.3, max=8.0, step=0.1, description='asp min', continuous_update=False)
PARAM_WIDGETS['roi_aspect_max'] = widgets.FloatSlider(value=7.0, min=1.0, max=15.0, step=0.2, description='asp max', continuous_update=False)
PARAM_WIDGETS['roi_margin_px'] = widgets.IntSlider(value=8, min=0, max=60, step=2, description='margem', continuous_update=False)
PARAM_WIDGETS['max_rois'] = widgets.IntSlider(value=60, min=1, max=300, step=1, description='max ROIs', continuous_update=False)

# HOG/SVM
PARAM_WIDGETS['hog_input'] = widgets.Dropdown(
    options=['gray', 'gray_eq', 'bilateral', 'canny_full', 'canny_hough', 'morph'],
    value='bilateral',
    description='img HOG'
)
PARAM_WIDGETS['hog_resize_w'] = widgets.IntSlider(value=64, min=32, max=160, step=8, description='HOG W', continuous_update=False)
PARAM_WIDGETS['hog_resize_h'] = widgets.IntSlider(value=128, min=64, max=256, step=8, description='HOG H', continuous_update=False)
PARAM_WIDGETS['hog_orientations'] = widgets.IntSlider(value=9, min=4, max=18, step=1, description='orient', continuous_update=False)
PARAM_WIDGETS['hog_pixels_per_cell'] = widgets.Dropdown(options=[4, 8, 16], value=8, description='cell')
PARAM_WIDGETS['hog_cells_per_block'] = widgets.Dropdown(options=[1, 2, 3], value=2, description='block')
PARAM_WIDGETS['svm_C'] = widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=2, step=0.25, description='SVM C', continuous_update=False)

# Amostragem do dataset YOLO
PARAM_WIDGETS['usar_selecao_salva'] = widgets.Checkbox(value=True, description='Usar seleção manual', disabled=True)
PARAM_WIDGETS['usar_test_real'] = widgets.Checkbox(value=True, description='Avaliar em test/')
PARAM_WIDGETS['negativos_por_positivo'] = widgets.IntSlider(value=2, min=1, max=8, step=1, description='neg/pos', continuous_update=False)
PARAM_WIDGETS['neg_iou_max'] = widgets.FloatSlider(value=0.05, min=0.0, max=0.40, step=0.01, description='IoU neg', continuous_update=False)
PARAM_WIDGETS['pos_margin_pct'] = widgets.FloatSlider(value=0.08, min=0.0, max=0.40, step=0.02, description='margem pos', continuous_update=False)
PARAM_WIDGETS['neg_attempts'] = widgets.IntSlider(value=30, min=5, max=100, step=5, description='tent neg', continuous_update=False)
PARAM_WIDGETS['max_train_images'] = widgets.IntSlider(value=0, min=0, max=3000, step=50, description='max train', continuous_update=False)
PARAM_WIDGETS['max_test_images'] = widgets.IntSlider(value=0, min=0, max=3000, step=50, description='max test', continuous_update=False)

PARAM_WIDGETS['test_size'] = widgets.FloatSlider(value=0.30, min=0.10, max=0.50, step=0.05, description='holdout', continuous_update=False)
PARAM_WIDGETS['random_state'] = widgets.IntSlider(value=42, min=0, max=999, step=1, description='seed', continuous_update=False)

# Guarda os valores originais como setup padrão do notebook.
DEFAULT_PARAM_VALUES = OrderedDict((nome, wid.value) for nome, wid in PARAM_WIDGETS.items())
DEFAULT_SETUP_ID = 'PADRAO'

# ----------------------------
# Funções para setups salvos
# ----------------------------

def valores_parametros_widgets():
    return OrderedDict((nome, wid.value) for nome, wid in PARAM_WIDGETS.items())


def _json_parametros(params):
    return json.dumps(dict(params), sort_keys=True, ensure_ascii=False, default=str)


def parametros_iguais(a, b):
    return _json_parametros(a) == _json_parametros(b)


def carregar_setups_parametricos():
    data = carregar_json(PARAM_SETUPS_PATH, default=None)
    if not isinstance(data, dict):
        data = {}
    if 'setups' not in data or not isinstance(data['setups'], dict):
        data['setups'] = {}
    return data


def salvar_setups_parametricos(data):
    data = dict(data)
    data['versao'] = 'setups_parametricos_hough_hog_svm_v1'
    data['atualizado_em'] = time.strftime('%Y-%m-%d %H:%M:%S')
    salvar_json(PARAM_SETUPS_PATH, data)


def proximo_setup_id(data):
    existentes = set(data.get('setups', {}).keys())
    i = 1
    while f'S{i:03d}' in existentes:
        i += 1
    return f'S{i:03d}'


def obter_params_setup_base(setup_id=None):
    setup_id = setup_id or STATE.get('setup_id', DEFAULT_SETUP_ID)
    if setup_id == DEFAULT_SETUP_ID:
        return DEFAULT_PARAM_VALUES

    data = carregar_setups_parametricos()
    setup = data.get('setups', {}).get(setup_id, {})
    return OrderedDict(setup.get('params', {}))


def aplicar_parametros_no_widgets(params):
    """Aplica parâmetros aos widgets sem disparar várias renderizações intermediárias."""
    STATE['loading_setup'] = True
    try:
        for nome, valor in params.items():
            if nome not in PARAM_WIDGETS:
                continue
            wid = PARAM_WIDGETS[nome]
            try:
                wid.value = valor
            except Exception:
                # Evita quebra caso o valor salvo não exista mais nas opções do widget.
                pass
        definir_params_a_partir_widgets(PARAM_WIDGETS)
    finally:
        STATE['loading_setup'] = False


def atualizar_estado_modificacao_setup():
    base = obter_params_setup_base(STATE.get('setup_id', DEFAULT_SETUP_ID))
    atual = valores_parametros_widgets()
    STATE['setup_modificado'] = not parametros_iguais(atual, base)


def obter_setup_id_para_historico():
    """Identificador salvo na tabela de resultados."""
    if STATE.get('setup_modificado', False):
        return 'MANUAL_NAO_SALVO'
    return STATE.get('setup_id', DEFAULT_SETUP_ID)


def obter_setup_nome_para_historico():
    if STATE.get('setup_modificado', False):
        base = STATE.get('setup_id', DEFAULT_SETUP_ID)
        return f'Manual não salvo, derivado de {base}'
    return STATE.get('setup_label', 'Setup padrão')


w_setup_selector = widgets.Dropdown(
    options=[('PADRAO | Setup padrão', DEFAULT_SETUP_ID)],
    value=DEFAULT_SETUP_ID,
    description='setup:',
    layout=widgets.Layout(width='420px')
)

w_setup_nome = widgets.Text(
    value='',
    placeholder='Nome opcional para o novo setup, ex.: hough_dist_20_90',
    description='nome:',
    layout=widgets.Layout(width='520px')
)

btn_setup_padrao = widgets.Button(
    description='Setup padrão',
    button_style='',
    icon='undo',
    layout=widgets.Layout(width='150px')
)

btn_salvar_setup = widgets.Button(
    description='Salvar setup',
    button_style='success',
    icon='save',
    layout=widgets.Layout(width='145px')
)

btn_excluir_setup = widgets.Button(
    description='Excluir setup',
    button_style='danger',
    icon='trash',
    layout=widgets.Layout(width='145px')
)

btn_renomear_setup = widgets.Button(
    description='Renomear setup',
    button_style='info',
    icon='edit',
    layout=widgets.Layout(width='160px')
)

btn_sobrescrever_setup = widgets.Button(
    description='Sobrescrever setup',
    button_style='warning',
    icon='refresh',
    layout=widgets.Layout(width='185px')
)

w_confirmar_sobrescrita_setup = widgets.Checkbox(
    value=False,
    description='confirmar sobrescrita',
    indent=False,
    layout=widgets.Layout(width='240px')
)

out_setup_status = widgets.Output()
out_setup_resultados = widgets.Output()


def renderizar_resultados_setup_selecionado():
    """Mostra, logo abaixo da lista de setups, os resultados já salvos para o setup atual."""
    with out_setup_resultados:
        clear_output(wait=True)
        setup_id = STATE.get('setup_id', DEFAULT_SETUP_ID)
        hist = carregar_historico()

        if hist is None or len(hist) == 0:
            display(HTML('<em>Nenhum treinamento salvo ainda.</em>'))
            return

        if 'setup_id' not in hist.columns:
            display(HTML('<em>O histórico existente ainda não possui coluna setup_id.</em>'))
            return

        df_setup = hist[hist['setup_id'].astype(str) == str(setup_id)].copy()
        if len(df_setup) == 0:
            display(HTML(f'<em>Nenhum resultado salvo para o setup {setup_id}.</em>'))
            return

        colunas = [
            'data_hora', 'setup_id', 'setup_nome', 'zoom_out_pct', 'accuracy', 'precision', 'recall', 'f1', 'tempo_s',
            'n_img_train', 'n_img_test', 'n_train', 'n_test', 'assinatura_curta'
        ]
        colunas = [c for c in colunas if c in df_setup.columns]

        display(HTML(f'<b>Resultados salvos para o setup {setup_id}</b>'))
        exibir_tabela_compacta(df_setup[colunas].tail(8), max_rows=8)


def atualizar_status_setup(mensagem=''):
    with out_setup_status:
        clear_output(wait=True)
        sid = obter_setup_id_para_historico()
        nome = obter_setup_nome_para_historico()
        marcador = ' | parâmetros alterados e ainda não salvos' if STATE.get('setup_modificado', False) else ''
        print(f'Setup atual para histórico: {sid} — {nome}{marcador}')
        if mensagem:
            print(mensagem)
        print(f'Arquivo de setups: {PARAM_SETUPS_PATH}')


def atualizar_dropdown_setups(selecionar=None):
    data = carregar_setups_parametricos()
    setups = data.get('setups', {})

    opcoes = [('PADRAO | Setup padrão', DEFAULT_SETUP_ID)]
    for setup_id in sorted(setups.keys()):
        setup = setups[setup_id]
        nome = setup.get('nome', 'sem_nome')
        data_hora = setup.get('data_hora', '')
        opcoes.append((f'{setup_id} | {nome} | {data_hora}', setup_id))

    valores = [v for _, v in opcoes]
    selecionar = selecionar if selecionar in valores else DEFAULT_SETUP_ID

    STATE['setup_dropdown_updating'] = True
    try:
        w_setup_selector.options = opcoes
        w_setup_selector.value = selecionar
    finally:
        STATE['setup_dropdown_updating'] = False


def carregar_setup_por_id(setup_id, atualizar_visualizacao=True):
    if setup_id == DEFAULT_SETUP_ID:
        aplicar_parametros_no_widgets(DEFAULT_PARAM_VALUES)
        STATE['setup_id'] = DEFAULT_SETUP_ID
        STATE['setup_label'] = 'Setup padrão'
        STATE['setup_modificado'] = False
        if atualizar_visualizacao:
            atualizar_todas_visualizacoes()
        atualizar_status_setup('Setup padrão carregado.')
        renderizar_resultados_setup_selecionado()
        return

    data = carregar_setups_parametricos()
    setup = data.get('setups', {}).get(setup_id)
    if setup is None:
        atualizar_status_setup(f'Setup {setup_id} não encontrado. O setup padrão foi mantido.')
        return

    params = OrderedDict(setup.get('params', {}))
    aplicar_parametros_no_widgets(params)
    STATE['setup_id'] = setup_id
    STATE['setup_label'] = setup.get('nome', setup_id)
    STATE['setup_modificado'] = False
    if atualizar_visualizacao:
        atualizar_todas_visualizacoes()
    atualizar_status_setup(f'Setup {setup_id} carregado.')
    renderizar_resultados_setup_selecionado()


def on_setup_selector_change(change):
    if STATE.get('setup_dropdown_updating', False):
        return
    if change.get('name') == 'value' and change.get('new') is not None:
        carregar_setup_por_id(change['new'])


def on_setup_padrao(_=None):
    atualizar_dropdown_setups(DEFAULT_SETUP_ID)
    carregar_setup_por_id(DEFAULT_SETUP_ID)



def _params_atuais_para_setup():
    return OrderedDict((k, widget_value(w)) for k, w in PARAM_WIDGETS.items())


def _setup_existente_por_nome(data, nome):
    nome_norm = str(nome).strip().lower()
    for sid, item in data.get('setups', {}).items():
        if str(item.get('nome', '')).strip().lower() == nome_norm:
            return sid, item
    return None, None


def _setup_params_diferentes(item, params):
    return not parametros_iguais(OrderedDict(item.get('params', {})), params)


def on_salvar_setup(_=None):
    data = carregar_setups_parametricos()
    params_atual = _params_atuais_para_setup()
    nome = w_setup_nome.value.strip() or f'Setup {proximo_setup_id(data)}'

    sid_existente, item_existente = _setup_existente_por_nome(data, nome)
    if sid_existente is not None:
        if _setup_params_diferentes(item_existente, params_atual):
            if not w_confirmar_sobrescrita_setup.value:
                atualizar_status_setup(
                    f"Já existe um setup chamado '{nome}' ({sid_existente}) com parâmetros diferentes. "
                    "Marque 'confirmar sobrescrita' e clique novamente em Salvar setup para substituir esse setup, "
                    "ou use outro nome para criar um novo."
                )
                return
            data['setups'][sid_existente]['params'] = dict(params_atual)
            data['setups'][sid_existente]['data_hora'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            data['setups'][sid_existente]['sobrescrito_em'] = data['setups'][sid_existente]['data_hora']
            salvar_setups_parametricos(data)
            STATE['setup_id'] = sid_existente
            STATE['setup_label'] = nome
            STATE['setup_modificado'] = False
            w_confirmar_sobrescrita_setup.value = False
            atualizar_dropdown_setups(sid_existente)
            atualizar_status_setup(f"Setup existente {sid_existente} — '{nome}' sobrescrito com os parâmetros atuais.")
            renderizar_resultados_setup_selecionado()
            return
        else:
            STATE['setup_id'] = sid_existente
            STATE['setup_label'] = nome
            STATE['setup_modificado'] = False
            atualizar_dropdown_setups(sid_existente)
            atualizar_status_setup(f"Já existia um setup chamado '{nome}' com os mesmos parâmetros. Setup {sid_existente} carregado.")
            renderizar_resultados_setup_selecionado()
            return

    novo_id = proximo_setup_id(data)
    data['setups'][novo_id] = {
        'nome': nome,
        'data_hora': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'params': dict(params_atual)
    }
    salvar_setups_parametricos(data)

    STATE['setup_id'] = novo_id
    STATE['setup_label'] = nome
    STATE['setup_modificado'] = False
    w_confirmar_sobrescrita_setup.value = False
    atualizar_dropdown_setups(novo_id)
    atualizar_status_setup(f'Setup salvo como {novo_id}.')
    renderizar_resultados_setup_selecionado()


def on_excluir_setup(_=None):
    setup_id = w_setup_selector.value
    if setup_id == DEFAULT_SETUP_ID:
        atualizar_status_setup('O setup padrão não pode ser excluído.')
        return

    data = carregar_setups_parametricos()
    setups = data.get('setups', {})
    if setup_id in setups:
        nome = setups[setup_id].get('nome', setup_id)
        del setups[setup_id]
        data['setups'] = setups
        salvar_setups_parametricos(data)
        atualizar_dropdown_setups(DEFAULT_SETUP_ID)
        carregar_setup_por_id(DEFAULT_SETUP_ID)
        atualizar_status_setup(f'Setup {setup_id} — {nome} excluído. Setup padrão carregado.')
        renderizar_resultados_setup_selecionado()
    else:
        atualizar_status_setup(f'Setup {setup_id} não encontrado.')
        renderizar_resultados_setup_selecionado()


def on_renomear_setup(_=None):
    setup_id = w_setup_selector.value
    novo_nome = w_setup_nome.value.strip()
    if setup_id == DEFAULT_SETUP_ID:
        atualizar_status_setup('O setup padrão não pode ser renomeado. Salve como um novo setup.')
        return
    if not novo_nome:
        atualizar_status_setup('Digite um novo nome no campo de nome antes de renomear.')
        return
    data = carregar_setups_parametricos()
    setups = data.get('setups', {})
    if setup_id not in setups:
        atualizar_status_setup(f'Setup {setup_id} não encontrado.')
        return
    sid_mesmo_nome, _ = _setup_existente_por_nome(data, novo_nome)
    if sid_mesmo_nome is not None and sid_mesmo_nome != setup_id:
        atualizar_status_setup(f"Já existe outro setup com o nome '{novo_nome}' ({sid_mesmo_nome}). Escolha outro nome.")
        return
    setups[setup_id]['nome'] = novo_nome
    setups[setup_id]['renomeado_em'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    salvar_setups_parametricos(data)
    STATE['setup_label'] = novo_nome
    atualizar_dropdown_setups(setup_id)
    atualizar_status_setup(f'Setup {setup_id} renomeado para {novo_nome}.')
    renderizar_resultados_setup_selecionado()


def on_sobrescrever_setup(_=None):
    setup_id = w_setup_selector.value
    if setup_id == DEFAULT_SETUP_ID:
        atualizar_status_setup('O setup padrão não pode ser sobrescrito. Salve como um novo setup.')
        return
    if not w_confirmar_sobrescrita_setup.value:
        atualizar_status_setup("Para sobrescrever o setup selecionado, marque 'confirmar sobrescrita' e clique novamente.")
        return
    data = carregar_setups_parametricos()
    setups = data.get('setups', {})
    if setup_id not in setups:
        atualizar_status_setup(f'Setup {setup_id} não encontrado.')
        return
    params_atual = _params_atuais_para_setup()
    nome = w_setup_nome.value.strip() or setups[setup_id].get('nome', setup_id)
    sid_mesmo_nome, _ = _setup_existente_por_nome(data, nome)
    if sid_mesmo_nome is not None and sid_mesmo_nome != setup_id:
        atualizar_status_setup(f"Já existe outro setup com o nome '{nome}' ({sid_mesmo_nome}). Escolha outro nome ou salve por nome com confirmação.")
        return
    setups[setup_id]['nome'] = nome
    setups[setup_id]['params'] = dict(params_atual)
    setups[setup_id]['data_hora'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    setups[setup_id]['sobrescrito_em'] = setups[setup_id]['data_hora']
    salvar_setups_parametricos(data)
    STATE['setup_id'] = setup_id
    STATE['setup_label'] = nome
    STATE['setup_modificado'] = False
    w_confirmar_sobrescrita_setup.value = False
    atualizar_dropdown_setups(setup_id)
    atualizar_status_setup(f'Setup {setup_id} sobrescrito com os parâmetros atuais.')
    renderizar_resultados_setup_selecionado()


def on_params_change(change=None):
    definir_params_a_partir_widgets(PARAM_WIDGETS)

    # Evita valores inconsistentes simples.
    if STATE['params']['line_length_max'] < STATE['params']['line_length_min']:
        STATE['params']['line_length_max'] = STATE['params']['line_length_min']

    if STATE['params']['pair_dist_max'] < STATE['params']['pair_dist_min']:
        STATE['params']['pair_dist_max'] = STATE['params']['pair_dist_min']

    if not STATE.get('loading_setup', False):
        atualizar_estado_modificacao_setup()
        atualizar_status_setup()
        atualizar_todas_visualizacoes()


for wid in PARAM_WIDGETS.values():
    wid.observe(on_params_change, names='value')

w_setup_selector.observe(on_setup_selector_change, names='value')
btn_setup_padrao.on_click(on_setup_padrao)
btn_salvar_setup.on_click(on_salvar_setup)
btn_excluir_setup.on_click(on_excluir_setup)
btn_renomear_setup.on_click(on_renomear_setup)
btn_sobrescrever_setup.on_click(on_sobrescrever_setup)

# Inicializa os parâmetros antes de registrar a visualização.
definir_params_a_partir_widgets(PARAM_WIDGETS)
atualizar_dropdown_setups(DEFAULT_SETUP_ID)
atualizar_status_setup()
renderizar_resultados_setup_selecionado()

# ----------------------------
# Textos curtos ao lado de cada parâmetro
# ----------------------------
PARAM_HELP = {
    # Pré-processamento
    'clahe_ativo': 'Ativa equalização local. Ajuda em sombras e iluminação irregular.',
    'clahe_clip': 'Controla o contraste local. Valores altos realçam bordas, mas também ruído.',
    'clahe_grid': 'Tamanho da malha do CLAHE. Menor = ajuste mais local; maior = ajuste mais suave.',
    'bilateral_d': 'Diâmetro da vizinhança do filtro. Maior suaviza mais, mas pode perder detalhes finos.',
    'bilateral_sigma_color': 'Tolerância de diferença de intensidade. Maior mistura tons mais diferentes.',
    'bilateral_sigma_space': 'Alcance espacial da suavização. Maior espalha o efeito por regiões mais largas.',

    # Canny
    'canny_sigma': 'Define os limiares automáticos pela mediana. Maior tende a aceitar mais bordas fracas.',
    'canny_aperture': 'Tamanho do operador Sobel. Maior suaviza mais o gradiente, mas pode engrossar bordas.',
    'canny_l2gradient': 'Usa cálculo de gradiente mais preciso. Costuma melhorar bordas, com pequeno custo extra.',
    'edge_filter_ativo': 'Cria um segundo mapa de bordas para Hough: remove componentes pequenos antes da morfologia, evitando blobs sólidos no fechamento.',
    'edge_min_area': 'Área mínima do componente conectado no mapa Canny para permanecer no mapa usado pela Hough.',
    'edge_min_extent': 'Maior dimensão mínima do componente. Remove traços muito curtos sem alterar o Canny completo usado para arcos.',

    # Morfologia
    'close_kernel_w': 'Largura do fechamento. Une falhas horizontais/diagonais curtas nas bordas.',
    'close_kernel_h': 'Altura do fechamento. Valores altos favorecem união de bordas verticais do cilindro.',
    'close_iter': 'Repetições do fechamento. Aumenta a conexão das bordas, mas pode colar ruídos.',
    'open_kernel_w': 'Largura da abertura. Remove ruídos pequenos e estruturas finas na horizontal.',
    'open_kernel_h': 'Altura da abertura. Remove ruídos pequenos e linhas finas na vertical.',
    'open_iter': 'Repetições da abertura. Maior limpa mais, porém pode apagar bordas úteis.',
    'solid_filter_ativo': 'Remove regiões sólidas formadas pelo fechamento usando erosão forte para preservar linhas finas conectadas.',
    'solid_erode_kernel_w': 'Largura da erosão forte. Quanto maior, mais linhas finas desaparecem antes de detectar o núcleo sólido.',
    'solid_erode_kernel_h': 'Altura da erosão forte. Aumente para exigir regiões sólidas mais espessas verticalmente.',
    'solid_erode_iter': 'Repetições da erosão forte. Aumenta a seletividade para detectar apenas corpos realmente espessos.',
    'solid_min_area': 'Área mínima do núcleo sólido após erosão. Evita subtrair pequenos resíduos que sobreviveram.',
    'solid_dilate_kernel_w': 'Largura da dilatação do núcleo sólido antes da subtração.',
    'solid_dilate_kernel_h': 'Altura da dilatação do núcleo sólido antes da subtração.',
    'solid_dilate_iter': 'Repetições da dilatação do núcleo. Controla quanto do corpo sólido será removido do mapa fechado.',

    # Hough
    'hough_rho': 'Resolução da distância no acumulador de Hough. Menor é mais preciso e mais sensível.',
    'hough_theta_deg': 'Resolução angular da Hough. Menor separa melhor inclinações próximas.',
    'hough_threshold': 'Votos mínimos para aceitar uma reta. Maior reduz falsas retas, mas perde retas fracas.',
    'hough_min_line_length': 'Comprimento mínimo do segmento detectado. Maior descarta texturas e microbordas.',
    'hough_max_line_gap': 'Lacuna máxima para unir segmentos. Maior conecta bordas interrompidas.',

    # Geometria
    'line_length_min': 'Comprimento mínimo aceito no filtro pós-Hough. Remove segmentos curtos demais.',
    'line_length_max': 'Comprimento máximo aceito. Evita retas enormes de paredes, vigas ou bordas do cenário.',
    'pair_angle_tol_deg': 'Diferença angular máxima entre duas retas. Menor exige paralelismo mais rígido.',
    'pair_dist_min': 'Distância mínima entre retas. Representa o menor diâmetro aparente esperado.',
    'pair_dist_max': 'Distância máxima entre retas. Representa o maior diâmetro aparente esperado.',
    'pair_overlap_min': 'Sobreposição mínima ao longo das retas. Maior exige que elas estejam lado a lado.',
    'roi_aspect_min': 'Proporção mínima altura/largura da ROI. Evita candidatos achatados demais.',
    'roi_aspect_max': 'Proporção máxima altura/largura da ROI. Evita candidatos estreitos demais.',
    'roi_margin_px': 'Margem adicionada ao redor da ROI. Inclui contexto, topo/base e bordas incompletas.',
    'max_rois': 'Número máximo de ROIs enviadas ao SVM. Limita custo e excesso de candidatos.',

    # HOG/SVM
    'hog_input': 'Imagem usada para extrair HOG. Bilateral preserva bordas com menos ruído; Canny usa só contornos.',
    'hog_resize_w': 'Largura fixa do recorte antes do HOG. Padroniza as amostras para o SVM.',
    'hog_resize_h': 'Altura fixa do recorte antes do HOG. Deve preservar a forma vertical do cilindro.',
    'hog_orientations': 'Número de direções do HOG. Maior detalha gradientes, mas aumenta o vetor.',
    'hog_pixels_per_cell': 'Tamanho da célula HOG. Menor captura detalhes; maior generaliza melhor.',
    'hog_cells_per_block': 'Tamanho do bloco de normalização. Maior melhora robustez a iluminação, mas suaviza detalhes.',
    'svm_C': 'Regularização do SVM. Maior ajusta mais o treino; menor tende a generalizar melhor.',

    # Amostragem/avaliação
    'usar_selecao_salva': 'Força o uso das imagens marcadas manualmente para treino e teste.',
    'usar_test_real': 'Avalia no split test/ do dataset. Desligado usa holdout a partir do treino.',
    'negativos_por_positivo': 'Quantidade de recortes negativos gerados para cada cilindro anotado.',
    'neg_iou_max': 'Sobreposição máxima permitida entre negativo e objeto real. Menor evita falsos negativos.',
    'pos_margin_pct': 'Margem ao redor da caixa positiva. Inclui contexto visual do cilindro.',
    'neg_attempts': 'Tentativas para encontrar negativos válidos por imagem. Maior melhora busca, mas demora mais.',
    'max_train_images': 'Limite de imagens de treino. Zero usa todas as selecionadas.',
    'max_test_images': 'Limite de imagens de teste. Zero usa todas as selecionadas.',
    'test_size': 'Fração usada no holdout quando não se avalia diretamente em test/.',
    'random_state': 'Semente aleatória. Mantém negativos, holdout e resultados reproduzíveis.'
}

# Mantém os sliders largos e legíveis, mesmo com explicações ao lado.
# A versão anterior deixava o widget estreito demais; aqui cada parâmetro ocupa
# uma linha quando necessário, preservando a largura útil da barra de ajuste.
for _wid in PARAM_WIDGETS.values():
    try:
        _wid.layout.width = '390px'
        _wid.layout.min_width = '360px'
    except Exception:
        pass
    try:
        _wid.style = {'description_width': '115px'}
    except Exception:
        pass


def help_html(nome):
    return widgets.HTML(
        f"<div style='font-size:11px; color:#cfcfcf; line-height:1.25; white-space:normal;'>{PARAM_HELP.get(nome, '')}</div>",
        layout=widgets.Layout(width='360px', min_width='300px')
    )


def param_item(nome, texto=None):
    """Mostra o widget com largura preservada e uma explicação curta ao lado."""
    if texto is not None:
        PARAM_HELP[nome] = texto
    return widgets.HBox(
        [PARAM_WIDGETS[nome], help_html(nome)],
        layout=widgets.Layout(
            width='780px',
            min_width='720px',
            align_items='center',
            margin='4px 0px 4px 0px'
        )
    )


def param_row(nomes):
    """Organiza parâmetros com quebra automática, priorizando legibilidade dos sliders."""
    return widgets.HBox(
        [param_item(nome) for nome in nomes],
        layout=widgets.Layout(
            flex_flow='row wrap',
            align_items='flex-start',
            gap='4px 18px',
            width='100%'
        )
    )


box_setup = widgets.VBox([
    widgets.HTML('<b>Setup paramétrico</b>'),
    widgets.HTML('<small>Use o setup padrão, carregue um setup salvo, salve uma nova combinação, renomeie ou sobrescreva setups existentes. Sobrescritas exigem confirmação. '
                 'Ao salvar, um identificador como S001 é criado e aparecerá no histórico de resultados.</small>'),
    widgets.HBox([w_setup_selector, btn_setup_padrao, btn_salvar_setup, btn_renomear_setup, btn_sobrescrever_setup, btn_excluir_setup], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    out_setup_resultados,
    widgets.HBox([w_setup_nome, w_confirmar_sobrescrita_setup], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    out_setup_status
])

box_pre = widgets.VBox([
    widgets.HTML('<b>Pré-processamento</b>'),
    param_row(['clahe_ativo', 'clahe_clip']),
    param_row(['clahe_grid', 'bilateral_d']),
    param_row(['bilateral_sigma_color', 'bilateral_sigma_space']),
])

box_canny = widgets.VBox([
    widgets.HTML('<b>Canny adaptativo</b>'),
    param_row(['canny_sigma', 'canny_aperture']),
    param_row(['canny_l2gradient']),
])

box_edge_filter = widgets.VBox([
    widgets.HTML('<b>Filtro de traços curtos antes da morfologia</b>'),
    widgets.HTML('<small>Remove apenas componentes pequenos do Canny antes da morfologia. A remoção de regiões sólidas/blobs gerados pelo fechamento fica na aba Morfologia.</small>'),
    param_row(['edge_filter_ativo', 'edge_min_area']),
    param_row(['edge_min_extent']),
])

box_morph = widgets.VBox([
    widgets.HTML('<b>Morfologia</b>'),
    param_row(['close_kernel_w', 'close_kernel_h']),
    param_row(['close_iter', 'open_kernel_w']),
    param_row(['open_kernel_h', 'open_iter']),
    widgets.HTML('<b>Remoção de regiões sólidas após o fechamento</b>'),
    widgets.HTML('<small>Salva o mapa fechado, aplica erosão forte para apagar linhas finas, dilata o núcleo sólido e subtrai essa máscara do mapa fechado. Isso evita apagar linhas úteis conectadas ao sólido.</small>'),
    param_row(['solid_filter_ativo', 'solid_min_area']),
    param_row(['solid_erode_kernel_w', 'solid_erode_kernel_h']),
    param_row(['solid_erode_iter', 'solid_dilate_kernel_w']),
    param_row(['solid_dilate_kernel_h', 'solid_dilate_iter']),
])

box_hough = widgets.VBox([
    widgets.HTML('<b>Hough probabilística</b>'),
    param_row(['hough_rho', 'hough_theta_deg']),
    param_row(['hough_threshold', 'hough_min_line_length']),
    param_row(['hough_max_line_gap']),
])

box_geo = widgets.VBox([
    widgets.HTML('<b>Pareamento geométrico</b>'),
    param_row(['line_length_min', 'line_length_max']),
    param_row(['pair_angle_tol_deg', 'pair_dist_min']),
    param_row(['pair_dist_max', 'pair_overlap_min']),
    param_row(['roi_aspect_min', 'roi_aspect_max']),
    param_row(['roi_margin_px', 'max_rois']),
])

box_hog_svm = widgets.VBox([
    widgets.HTML('<b>HOG + SVM e amostragem YOLO</b>'),
    param_row(['hog_input', 'hog_resize_w']),
    param_row(['hog_resize_h', 'hog_orientations']),
    param_row(['hog_pixels_per_cell', 'hog_cells_per_block']),
    param_row(['svm_C', 'test_size']),
    param_row(['random_state']),
    widgets.HTML('<b>Amostras de treino a partir das caixas YOLO</b>'),
    param_row(['usar_selecao_salva', 'usar_test_real']),
    param_row(['negativos_por_positivo', 'neg_iou_max']),
    param_row(['pos_margin_pct', 'neg_attempts']),
    param_row(['max_train_images', 'max_test_images']),
])

accordion = widgets.Accordion(children=[
    box_pre,
    box_canny,
    box_edge_filter,
    box_morph,
    box_hough,
    box_geo,
    box_hog_svm
])

for idx, title in enumerate([
    'Pré-processamento',
    'Canny',
    'Filtro pré-Hough',
    'Morfologia',
    'Hough',
    'Filtro geométrico',
    'HOG/SVM'
]):
    accordion.set_title(idx, title)

out_filtros_ref = widgets.Output()
out_hough = widgets.Output()
out_rois = widgets.Output()

w_hog_roi = widgets.Dropdown(
    description='ROI HOG',
    options=[('Nenhuma ROI candidata', None)],
    value=None,
    layout=widgets.Layout(width='90%'),
    style={'description_width': '80px'}
)
out_hog_vis = widgets.Output()

box_ref = widgets.VBox([
    widgets.HTML('<b>Imagem de referência</b>'),
    widgets.HTML(
        '<small>Somente imagens marcadas na seleção manual aparecem nesta lista. '
        'A imagem escolhida alimenta todas as visualizações abaixo.</small>'
    ),
    w_ref_image
])

box_visualizacoes = widgets.VBox([
    widgets.HTML('<b>Visualizações da imagem de referência para ajuste dos parâmetros</b>'),
    widgets.HTML('<small>As imagens abaixo são atualizadas automaticamente ao alterar parâmetros, trocar setup ou escolher outra referência.</small>'),

    widgets.HTML('<hr><b>1. Pré-processamento aplicado</b>'),
    out_filtros_ref,

    widgets.HTML('<hr><b>2. Hough probabilística — segmentos detectados</b>'),
    widgets.HTML('<small>Mostra como os parâmetros de Hough alteram a quantidade, o comprimento e a fragmentação das retas encontradas.</small>'),
    out_hough,

    widgets.HTML('<hr><b>3. Pareamento geométrico — pares paralelos e ROIs candidatas</b>'),
    widgets.HTML('<small>Mostra quais pares de retas passam pelos filtros geométricos e quais regiões serão enviadas ao HOG/SVM.</small>'),
    out_rois,

    widgets.HTML('<hr><b>4. HOG aplicado à ROI candidata</b>'),
    widgets.HTML('<small>Escolha uma ROI candidata para ver o recorte usado pelo descritor e a visualização dos gradientes.</small>'),
    w_hog_roi,
    out_hog_vis
])

painel_parametros = widgets.VBox([
    widgets.HTML('<b>2) Ajuste interativo de parâmetros</b>'),
    widgets.HTML('<small>Ao alterar qualquer parâmetro, trocar setup ou trocar a imagem de referência, todas as visualizações abaixo são recalculadas automaticamente.</small>'),
    box_setup,
    accordion,
    box_ref,
    box_visualizacoes
])

display(painel_parametros)


def render_filtros_referencia():
    pre = obter_preprocessamento_referencia()

    if pre is None:
        print('Nenhuma imagem de referência selecionada. Marque imagens na seleção manual e escolha uma referência.')
        return

    info = pre.get('edge_filter_info', {})
    solid_info = pre.get('solid_filter_info', {})

    fig, axs = plt.subplots(3, 3, figsize=(16, 10))
    axs = axs.ravel()

    axs[0].imshow(pre['rgb'])
    axs[0].set_title('Original')

    axs[1].imshow(pre['gray_eq'], cmap='gray')
    axs[1].set_title('Cinza / CLAHE')

    axs[2].imshow(pre['bilateral'], cmap='gray')
    axs[2].set_title('Bilateral')

    axs[3].imshow(pre['canny_full'], cmap='gray')
    axs[3].set_title(f"Mapa A: Canny completo\nT1={pre['canny_lower']} | T2={pre['canny_upper']}")

    axs[4].imshow(pre['canny_hough'], cmap='gray')
    if info.get('enabled', False):
        titulo_filtro = (
            f"Mapa B: Canny p/ Hough\n"
            f"mantidos={info.get('n_kept', 0)} | removidos={info.get('n_removed', 0)}"
        )
    else:
        titulo_filtro = 'Mapa B: Canny p/ Hough\nfiltro desativado'
    axs[4].set_title(titulo_filtro)

    axs[5].imshow(pre['morph_close'], cmap='gray')
    axs[5].set_title('Mapa fechado salvo\nsobre o mapa B')

    axs[6].imshow(pre.get('solid_core', np.zeros_like(pre['morph'])), cmap='gray')
    if solid_info.get('enabled', False):
        axs[6].set_title(
            f"Núcleo sólido por erosão\n"
            f"núcleos={solid_info.get('n_core_kept', 0)} | área>={solid_info.get('core_min_area', 0)}"
        )
    else:
        axs[6].set_title('Núcleo sólido\nfiltro desativado')

    axs[7].imshow(pre.get('morph_no_solids', pre['morph']), cmap='gray')
    if solid_info.get('enabled', False):
        axs[7].set_title(
            f"Após subtrair sólidos\n"
            f"px máscara={solid_info.get('mask_pixels', 0)} | erode={solid_info.get('erode_kernel', '-') }"
        )
    else:
        axs[7].set_title('Após subtrair sólidos\nfiltro desativado')

    axs[8].imshow(pre['morph'], cmap='gray')
    axs[8].set_title('Morfologia final\npara Hough')

    for ax in axs:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

def render_hough():
    pre = obter_preprocessamento_referencia()

    if pre is None:
        print('Nenhuma imagem de referência selecionada.')
        return

    p = STATE['params']
    fonte = p.get('hough_source_any', 'morph')
    mapa_hough = pre.get(fonte, pre['morph'])
    lines = detectar_linhas_hough(mapa_hough, p)
    img_lines = desenhar_linhas(pre['rgb'], lines, max_lines=120)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(img_lines)
    ax.set_title(f'HoughLinesP — {len(lines)} segmentos detectados | fonte={fonte}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def render_rois():
    pre = obter_preprocessamento_referencia()

    if pre is None:
        print('Nenhuma imagem de referência selecionada.')
        return

    lines = detectar_linhas_hough(pre['morph'], STATE['params'])
    candidatos = encontrar_rois_por_pares_de_linhas(lines, pre['morph'].shape, STATE['params'])
    img_rois = desenhar_rois(pre['rgb'], candidatos, mostrar_pares=True)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(img_rois)
    ax.set_title(f'ROIs por pares paralelos — {len(candidatos)} candidatas')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    if len(candidatos) > 0:
        df_cand = pd.DataFrame([
            {
                'roi': i + 1,
                'bbox': c['bbox'],
                'dist_px': round(c['distance'], 1),
                'ang_diff': round(c['angle_diff'], 1),
                'overlap': round(c['overlap'], 2),
                'aspect': round(c['aspect'], 2),
                'area': c['area']
            }
            for i, c in enumerate(candidatos)
        ])
        exibir_tabela_compacta(df_cand, max_rows=12)
    else:
        print('Nenhum par de retas passou pelos filtros geométricos.')


def atualizar_opcoes_roi_hog(candidatos):
    """Atualiza o dropdown de ROIs sem disparar recursão de renderização."""
    antigo = w_hog_roi.value

    if not candidatos:
        novas_opcoes = [('Nenhuma ROI candidata', None)]
        novo_valor = None
    else:
        novas_opcoes = []
        for i, c in enumerate(candidatos):
            x1, y1, x2, y2 = c['bbox']
            rotulo = (
                f'ROI {i+1} | bbox=({x1},{y1},{x2},{y2}) | '
                f'dist={c["distance"]:.1f}px | ang={c["angle_diff"]:.1f}° | ov={c["overlap"]:.2f}'
            )
            novas_opcoes.append((rotulo, i))

        valores = [v for _, v in novas_opcoes]
        novo_valor = antigo if antigo in valores else valores[0]

    STATE['hog_roi_dropdown_updating'] = True
    try:
        w_hog_roi.options = novas_opcoes
        w_hog_roi.value = novo_valor
    finally:
        STATE['hog_roi_dropdown_updating'] = False

    return novo_valor


def render_hog_visualizacao():
    pre = obter_preprocessamento_referencia()
    img_bgr_ref = obter_imagem_referencia()

    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada.')
        return

    lines = detectar_linhas_hough(pre['morph'], STATE['params'])
    candidatos = encontrar_rois_por_pares_de_linhas(lines, pre['morph'].shape, STATE['params'])
    idx = atualizar_opcoes_roi_hog(candidatos)

    if idx is None or len(candidatos) == 0:
        print('Nenhuma ROI candidata foi gerada para esta imagem com os parâmetros atuais.')
        print('Ajuste Hough, comprimento das linhas, distância entre pares, sobreposição ou proporção da ROI.')
        return

    idx = int(idx)
    if idx < 0 or idx >= len(candidatos):
        idx = 0

    c = candidatos[idx]
    x1, y1, x2, y2 = c['bbox']
    crop_bgr = img_bgr_ref[y1:y2, x1:x2].copy()

    if crop_bgr.size == 0:
        print('A ROI selecionada gerou um recorte vazio.')
        return

    try:
        hog_input_resized, hog_image, features = extrair_hog_visualizacao_de_imagem(crop_bgr, STATE['params'])
    except Exception as e:
        print('Não foi possível gerar a visualização HOG:', e)
        traceback.print_exc(limit=1)
        return

    img_contexto = pre['rgb'].copy()
    cv2.rectangle(img_contexto, (x1, y1), (x2, y2), (255, 120, 0), 2)
    cv2.putText(
        img_contexto,
        f'ROI {idx+1}',
        (x1, max(15, y1 - 6)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 120, 0),
        2,
        cv2.LINE_AA
    )
    for line in [c['line1'], c['line2']]:
        lx1, ly1, lx2, ly2 = line
        cv2.line(img_contexto, (lx1, ly1), (lx2, ly2), (0, 255, 0), 2)

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    fig, axs = plt.subplots(1, 4, figsize=(16, 4.6))

    axs[0].imshow(img_contexto)
    axs[0].set_title('ROI selecionada\nno contexto')

    axs[1].imshow(crop_rgb)
    axs[1].set_title('Recorte original\nda ROI')

    axs[2].imshow(hog_input_resized, cmap='gray')
    axs[2].set_title(f"Entrada do HOG\n{STATE['params']['hog_input']} | {hog_input_resized.shape[1]}×{hog_input_resized.shape[0]}")

    axs[3].imshow(hog_image, cmap='gray')
    axs[3].set_title(f'Visualização HOG\nvetor = {len(features)} atributos')

    for ax in axs:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    df_info = pd.DataFrame([{
        'roi': idx + 1,
        'bbox': c['bbox'],
        'dist_px': round(c['distance'], 1),
        'ang_diff': round(c['angle_diff'], 1),
        'overlap': round(c['overlap'], 2),
        'aspect': round(c['aspect'], 2),
        'area': c['area'],
        'hog_input': STATE['params']['hog_input'],
        'hog_size': f"{STATE['params']['hog_resize_w']}x{STATE['params']['hog_resize_h']}",
        'orient': STATE['params']['hog_orientations'],
        'px_cell': STATE['params']['hog_pixels_per_cell'],
        'cells_block': STATE['params']['hog_cells_per_block'],
        'n_atributos': len(features)
    }])
    exibir_tabela_compacta(df_info, max_rows=1)


def on_hog_roi_change(change):
    if STATE.get('hog_roi_dropdown_updating', False):
        return
    if change.get('name') == 'value':
        atualizar_renderer('hog_visualizacao')


w_hog_roi.observe(on_hog_roi_change, names='value')

registrar_renderer('filtros_referencia', out_filtros_ref, render_filtros_referencia)
registrar_renderer('hough', out_hough, render_hough)
registrar_renderer('rois', out_rois, render_rois)
registrar_renderer('hog_visualizacao', out_hog_vis, render_hog_visualizacao)


## Refinamento da bounding box, arcos de tampa e IoU do detector completo

Esta revisão adiciona três recursos discutidos após a última versão do notebook:

1. **Refinamento paramétrico da bounding box**: a ROI gerada pelo par de linhas verticais pode ser expandida em X/Y e ajustada por proporção esperada antes da validação pelo SVM.
2. **Score de arcos de tampa**: após detectar pares de linhas verticais, o notebook volta ao Canny completo dentro da ROI e procura contornos curvos nas faixas superior/inferior conectados às laterais dentro de uma tolerância de gap. Esse score entra como bônus, não como regra obrigatória.
3. **Métricas do detector completo por IoU**: além das métricas do HOG/SVM nos recortes, o histórico passa a registrar métricas de localização das caixas previstas contra as caixas YOLO reais.


In [ ]:
# ============================================================
# 11B. Refinamento de bounding boxes, arcos de tampa e avaliação por IoU
# ============================================================
# Esta célula fica após o ajuste de parâmetros para reaproveitar os helpers de UI
# e antes do treinamento, pois as métricas IoU do detector completo são registradas
# no histórico de treinamento.

# ------------------------------------------------------------
# Widgets adicionais de refinamento e avaliação
# ------------------------------------------------------------
if 'PARAM_WIDGETS' not in globals():
    raise RuntimeError('Execute primeiro a célula de ajuste de parâmetros.')

PARAM_WIDGETS['bbox_refine_enabled'] = widgets.Checkbox(
    value=True, description='ativar refino', indent=False,
    layout=widgets.Layout(width='210px')
)
PARAM_WIDGETS['bbox_refine_mode'] = widgets.Dropdown(
    options=[('expansão fixa salva', 'parametrica'), ('busca local por score', 'score_local')],
    value='score_local', description='modo:',
    layout=widgets.Layout(width='280px'), style={'description_width': '55px'}
)
PARAM_WIDGETS['ref_expand_x_pct'] = widgets.FloatSlider(
    value=20.0, min=0.0, max=100.0, step=2.5,
    description='exp X %', readout_format='.1f', continuous_update=False
)
PARAM_WIDGETS['ref_expand_y_pct'] = widgets.FloatSlider(
    value=45.0, min=0.0, max=160.0, step=2.5,
    description='exp Y %', readout_format='.1f', continuous_update=False
)
PARAM_WIDGETS['ref_aspect_min'] = widgets.FloatSlider(
    value=1.6, min=0.5, max=8.0, step=0.1,
    description='asp ref min', readout_format='.1f', continuous_update=False
)
PARAM_WIDGETS['ref_aspect_max'] = widgets.FloatSlider(
    value=8.0, min=1.0, max=15.0, step=0.2,
    description='asp ref max', readout_format='.1f', continuous_update=False
)
PARAM_WIDGETS['ref_search_steps'] = widgets.IntSlider(
    value=5, min=2, max=9, step=1,
    description='passos', continuous_update=False
)

PARAM_WIDGETS['cap_score_enabled'] = widgets.Checkbox(
    value=True, description='usar arcos', indent=False,
    layout=widgets.Layout(width='180px')
)
PARAM_WIDGETS['svm_usar_features_arcos'] = widgets.Checkbox(
    value=True, description='HOG+arcos no SVM', indent=False,
    layout=widgets.Layout(width='230px')
)
PARAM_WIDGETS['cap_band_pct'] = widgets.FloatSlider(
    value=25.0, min=8.0, max=45.0, step=1.0,
    description='faixa %', readout_format='.0f', continuous_update=False
)
PARAM_WIDGETS['cap_gap_px'] = widgets.IntSlider(
    value=10, min=0, max=60, step=1,
    description='gap px', continuous_update=False
)
PARAM_WIDGETS['cap_span_min'] = widgets.FloatSlider(
    value=0.35, min=0.10, max=0.95, step=0.05,
    description='span min', readout_format='.2f', continuous_update=False
)
PARAM_WIDGETS['cap_curve_min'] = widgets.FloatSlider(
    value=0.10, min=0.00, max=0.80, step=0.02,
    description='curva min', readout_format='.2f', continuous_update=False
)
PARAM_WIDGETS['cap_min_points'] = widgets.IntSlider(
    value=8, min=3, max=80, step=1,
    description='pts min', continuous_update=False
)
PARAM_WIDGETS['score_weight_svm'] = widgets.FloatSlider(
    value=1.00, min=0.0, max=3.0, step=0.05,
    description='peso SVM', readout_format='.2f', continuous_update=False
)
PARAM_WIDGETS['score_weight_cap'] = widgets.FloatSlider(
    value=0.35, min=0.0, max=2.0, step=0.05,
    description='peso arco', readout_format='.2f', continuous_update=False
)
PARAM_WIDGETS['score_weight_geom'] = widgets.FloatSlider(
    value=0.10, min=0.0, max=1.0, step=0.05,
    description='peso geom', readout_format='.2f', continuous_update=False
)
PARAM_WIDGETS['rect_penalty_weight'] = widgets.FloatSlider(
    value=0.20, min=0.0, max=2.0, step=0.05,
    description='penal ret', readout_format='.2f', continuous_update=False
)

PARAM_WIDGETS['det_score_min'] = widgets.FloatSlider(
    value=0.0, min=-3.0, max=8.0, step=0.10,
    description='score det', readout_format='.2f', continuous_update=False
)
PARAM_WIDGETS['det_nms_iou'] = widgets.FloatSlider(
    value=0.30, min=0.05, max=0.90, step=0.05,
    description='NMS det', readout_format='.2f', continuous_update=False
)
PARAM_WIDGETS['det_iou_thr'] = widgets.FloatSlider(
    value=0.50, min=0.10, max=0.90, step=0.05,
    description='IoU aval', readout_format='.2f', continuous_update=False
)
PARAM_WIDGETS['det_max_det'] = widgets.IntSlider(
    value=8, min=1, max=50, step=1,
    description='máx det', continuous_update=False
)
PARAM_WIDGETS['det_eval_max_images'] = widgets.IntSlider(
    value=120, min=10, max=1000, step=10,
    description='imgs aval', continuous_update=False
)

# Conecta os novos widgets à atualização de estado/visualizações.
for _w in list(PARAM_WIDGETS.values()):
    try:
        _w.unobserve(on_params_change, names='value')
    except Exception:
        pass
    try:
        _w.observe(on_params_change, names='value')
    except Exception:
        pass

# Atualiza STATE['params'] incluindo os widgets recém-criados.
definir_params_a_partir_widgets(PARAM_WIDGETS)

# Layout dos novos controles. Usa os helpers definidos na célula de parâmetros.
_box_refino = widgets.VBox([
    widgets.HTML('<h4 style="margin:4px 0 2px 0;">Refinamento da bounding box</h4>'),
    widgets.HTML('<small>A ROI parcial gerada pela Hough pode ser expandida antes do SVM. No modo de busca local, várias expansões são testadas e a melhor nota combinada é escolhida.</small>'),
    widgets.HBox([
        PARAM_WIDGETS['bbox_refine_enabled'],
        PARAM_WIDGETS['bbox_refine_mode'],
        param_item('ref_expand_x_pct', 'Aumenta a largura da caixa candidata. Ajuda quando a Hough pegou só parte das laterais.'),
        param_item('ref_expand_y_pct', 'Aumenta a altura da caixa candidata. Ajuda quando as linhas verticais foram detectadas apenas parcialmente.')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        param_item('ref_aspect_min', 'Força altura/largura mínima após refino. Útil para cilindros verticais de média/longa distância.'),
        param_item('ref_aspect_max', 'Limita caixas altas demais após expansão.'),
        param_item('ref_search_steps', 'Quantidade de expansões testadas no modo busca local por score.')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px'))
])

_box_arcos = widgets.VBox([
    widgets.HTML('<h4 style="margin:4px 0 2px 0;">Arcos de tampa conectados às laterais</h4>'),
    widgets.HTML('<small>As linhas verticais continuam vindo da imagem morfologicamente filtrada. Para os arcos, o notebook volta ao Canny completo dentro da ROI, preservando bordas horizontais/curvas.</small>'),
    widgets.HBox([
        PARAM_WIDGETS['cap_score_enabled'],
        PARAM_WIDGETS['svm_usar_features_arcos'],
        param_item('cap_band_pct', 'Faixa superior/inferior analisada para procurar tampas curvas.'),
        param_item('cap_gap_px', 'Tolerância para considerar que o arco está conectado às laterais verticais.')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        param_item('cap_span_min', 'Largura mínima do contorno/arco em relação à largura da ROI.'),
        param_item('cap_curve_min', 'Curvatura mínima para reduzir bônus de linhas retas de caixas.'),
        param_item('cap_min_points', 'Número mínimo de pontos do contorno analisado.')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        param_item('score_weight_svm', 'Peso do score do SVM na escolha da ROI refinada.'),
        param_item('score_weight_cap', 'Bônus para arcos de tampa detectados.'),
        param_item('score_weight_geom', 'Bônus para coerência geométrica do par de linhas.'),
        param_item('rect_penalty_weight', 'Penaliza candidatos com topo/base muito retos, semelhantes a caixas.')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px'))
])

_box_det_eval = widgets.VBox([
    widgets.HTML('<h4 style="margin:4px 0 2px 0;">Avaliação do detector completo por IoU</h4>'),
    widgets.HTML('<small>Esses parâmetros controlam a avaliação imagem inteira → Hough → ROI refinada → HOG/SVM → NMS → comparação com YOLO.</small>'),
    widgets.HBox([
        param_item('det_score_min', 'Score final mínimo para aceitar uma predição.'),
        param_item('det_nms_iou', 'Sobreposição usada para suprimir caixas duplicadas.'),
        param_item('det_iou_thr', 'IoU usada para contar TP/FP/FN na tabela de histórico.'),
        param_item('det_max_det', 'Máximo de caixas finais por imagem.'),
        param_item('det_eval_max_images', 'Limite de imagens usadas na avaliação completa após o treino.')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px'))
])

display(widgets.VBox([_box_refino, _box_arcos, _box_det_eval], layout=widgets.Layout(gap='10px')))

# ------------------------------------------------------------
# Funções geométricas e de arcos
# ------------------------------------------------------------
def _pget(p, nome, default):
    return p[nome] if nome in p else default


def _bbox_valida(bbox):
    try:
        vals = [float(v) for v in bbox]
        return all(np.isfinite(v) for v in vals) and vals[2] > vals[0] and vals[3] > vals[1]
    except Exception:
        return False


def expandir_bbox_xy_pct(bbox, pct_x, pct_y, shape_hw):
    h, w = shape_hw
    x1, y1, x2, y2 = map(float, bbox)
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    mx = bw * float(pct_x) / 100.0 / 2.0
    my = bh * float(pct_y) / 100.0 / 2.0
    return clip_bbox(x1 - mx, y1 - my, x2 + mx, y2 + my, w, h)


def ajustar_bbox_aspecto(bbox, shape_hw, aspect_min=None, aspect_max=None):
    h_img, w_img = shape_hw
    x1, y1, x2, y2 = map(float, bbox)
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    aspect = bh / bw

    if aspect_min is not None and aspect < float(aspect_min):
        bh = bw * float(aspect_min)
    if aspect_max is not None and aspect > float(aspect_max):
        bw = bh / float(aspect_max)

    return clip_bbox(cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2, w_img, h_img)


def _pontuacao_contorno_arco(cnt, roi_w, band_h, p):
    pts = cnt.reshape(-1, 2).astype(float)
    min_pts = int(_pget(p, 'cap_min_points', 8))
    if len(pts) < min_pts or roi_w <= 1 or band_h <= 1:
        return None

    xs = pts[:, 0]
    ys = pts[:, 1]
    span_px = float(xs.max() - xs.min() + 1.0)
    span_ratio = span_px / max(1.0, roi_w)
    if span_ratio < float(_pget(p, 'cap_span_min', 0.35)):
        return None

    gap_max = max(1.0, float(_pget(p, 'cap_gap_px', 10)))
    left_gap = float(xs.min())
    right_gap = float((roi_w - 1) - xs.max())
    conn_left = max(0.0, 1.0 - left_gap / gap_max)
    conn_right = max(0.0, 1.0 - right_gap / gap_max)
    connect_score = (conn_left + conn_right) / 2.0

    amp_ratio = float(ys.max() - ys.min() + 1.0) / max(1.0, band_h)
    lin_err = 0.0
    quad_gain = 0.0
    if len(np.unique(xs.astype(int))) >= 4:
        try:
            c1 = np.polyfit(xs, ys, 1)
            y1_hat = np.polyval(c1, xs)
            lin_err = float(np.sqrt(np.mean((ys - y1_hat) ** 2))) / max(1.0, band_h)
            c2 = np.polyfit(xs, ys, 2)
            y2_hat = np.polyval(c2, xs)
            quad_err = float(np.sqrt(np.mean((ys - y2_hat) ** 2))) / max(1.0, band_h)
            quad_gain = max(0.0, lin_err - quad_err)
        except Exception:
            pass

    # A curvatura é estimada por variação vertical e ganho de ajuste quadrático.
    # Linhas horizontais retas terão amplitude e erro linear baixos.
    curva_score = min(1.0, max(amp_ratio * 1.25, quad_gain * 5.0, lin_err * 2.5))
    if curva_score < float(_pget(p, 'cap_curve_min', 0.10)):
        curva_score *= 0.35

    span_score = min(1.0, span_ratio / max(1e-6, float(_pget(p, 'cap_span_min', 0.35))))
    score = 0.30 * span_score + 0.25 * connect_score + 0.45 * curva_score

    # Penalidade de retangularidade: contorno largo, conectado e quase reto.
    straight_like = 0.0
    if span_ratio >= float(_pget(p, 'cap_span_min', 0.35)) and connect_score > 0.45 and lin_err < 0.035 and amp_ratio < 0.18:
        straight_like = min(1.0, 0.5 * span_score + 0.5 * connect_score)

    return {
        'score': float(np.clip(score, 0.0, 1.0)),
        'span_ratio': span_ratio,
        'connect_score': connect_score,
        'curva_score': curva_score,
        'straight_like': straight_like,
        'contour': pts.astype(int),
    }


def score_arcos_tampa(pre, bbox, p):
    """Calcula scores de arcos superior/inferior usando o Canny completo da ROI.

    A morfologia vertical pode remover horizontais para a Hough, mas aqui a análise
    volta ao Canny completo para procurar fechamento curvo das tampas.
    """
    edges = pre.get('canny')
    if edges is None:
        return {'top_score': 0.0, 'bottom_score': 0.0, 'cap_score': 0.0, 'rect_penalty': 0.0, 'top_contour': None, 'bottom_contour': None}

    h_img, w_img = edges.shape[:2]
    x1, y1, x2, y2 = clip_bbox(*bbox, w_img, h_img)
    roi_w = max(1, x2 - x1)
    roi_h = max(1, y2 - y1)
    if roi_w < 4 or roi_h < 4:
        return {'top_score': 0.0, 'bottom_score': 0.0, 'cap_score': 0.0, 'rect_penalty': 0.0, 'top_contour': None, 'bottom_contour': None}

    band_h = int(max(4, min(roi_h, round(roi_h * float(_pget(p, 'cap_band_pct', 25.0)) / 100.0))))
    roi_edges = edges[y1:y2, x1:x2]

    def analisar_banda(band, y_offset):
        contours, _ = cv2.findContours(band.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        best = None
        for cnt in contours:
            s = _pontuacao_contorno_arco(cnt, roi_w, band_h, p)
            if s is None:
                continue
            if best is None or s['score'] > best['score']:
                best = s
        if best is None:
            return 0.0, 0.0, None
        pts_global = best['contour'].copy()
        pts_global[:, 0] += x1
        pts_global[:, 1] += y1 + y_offset
        return best['score'], best['straight_like'], pts_global

    top_score, top_straight, top_contour = analisar_banda(roi_edges[:band_h, :], 0)
    bottom_score, bottom_straight, bottom_contour = analisar_banda(roi_edges[roi_h-band_h:roi_h, :], roi_h - band_h)

    cap_score = max(float(top_score), float(bottom_score))
    rect_penalty = max(float(top_straight), float(bottom_straight))
    return {
        'top_score': float(top_score),
        'bottom_score': float(bottom_score),
        'cap_score': float(cap_score),
        'rect_penalty': float(rect_penalty),
        'top_contour': top_contour,
        'bottom_contour': bottom_contour,
    }


def features_arcos_crop(img_bgr, p):
    """Extrai pequenas características geométricas de tampa para concatenar ao HOG."""
    if img_bgr is None or img_bgr.size == 0:
        return np.zeros(5, dtype=np.float32)
    pre = aplicar_preprocessamento(img_bgr, p)
    h, w = pre['gray'].shape[:2]
    s = score_arcos_tampa(pre, (0, 0, w, h), p)
    aspect = float(h) / max(1.0, float(w))
    # Aspecto é compactado para evitar escala numérica exagerada perto do HOG.
    aspect_compacto = min(1.0, aspect / 8.0)
    return np.array([
        s['top_score'], s['bottom_score'], s['cap_score'], s['rect_penalty'], aspect_compacto
    ], dtype=np.float32)


# Redefine a extração de features usada por treino e inferência.
def extrair_hog_de_imagem(img_bgr, p):
    """Extrai HOG e, opcionalmente, concatena características de arcos de tampa."""
    pre = aplicar_preprocessamento(img_bgr, p)
    img_hog = selecionar_imagem_hog(pre, p)

    w = int(p['hog_resize_w'])
    h = int(p['hog_resize_h'])
    img_resized = cv2.resize(img_hog, (w, h), interpolation=cv2.INTER_AREA)

    features = hog(
        img_resized,
        orientations=int(p['hog_orientations']),
        pixels_per_cell=(int(p['hog_pixels_per_cell']), int(p['hog_pixels_per_cell'])),
        cells_per_block=(int(p['hog_cells_per_block']), int(p['hog_cells_per_block'])),
        block_norm='L2-Hys',
        transform_sqrt=True,
        feature_vector=True
    ).astype(np.float32)

    if bool(_pget(p, 'svm_usar_features_arcos', True)):
        geo = features_arcos_crop(img_bgr, p)
        features = np.concatenate([features, geo]).astype(np.float32)

    return features


def refinar_bbox_candidato(img_bgr, pre, candidato, clf, p):
    """Gera a bbox refinada de um candidato Hough/geométrico.

    Modo parametrico: aplica apenas a expansão X/Y salva no setup.
    Modo score_local: testa várias expansões e escolhe a melhor nota combinada.
    """
    h_img, w_img = pre['gray'].shape[:2]
    bbox0 = candidato.get('bbox')
    if not bool(_pget(p, 'bbox_refine_enabled', True)):
        cap = score_arcos_tampa(pre, bbox0, p) if bool(_pget(p, 'cap_score_enabled', True)) else {'cap_score':0,'rect_penalty':0,'top_score':0,'bottom_score':0,'top_contour':None,'bottom_contour':None}
        return bbox0, cap, {'refine_mode': 'off', 'tested': 1, 'score_final': None, 'score_svm': None}

    base_x = float(_pget(p, 'ref_expand_x_pct', 20.0))
    base_y = float(_pget(p, 'ref_expand_y_pct', 45.0))
    aspect_min = float(_pget(p, 'ref_aspect_min', 1.6))
    aspect_max = float(_pget(p, 'ref_aspect_max', 8.0))
    mode = _pget(p, 'bbox_refine_mode', 'parametrica')

    if mode == 'score_local' and clf is not None:
        steps = int(_pget(p, 'ref_search_steps', 5))
        fatores = np.linspace(0.0, 1.0, max(2, steps))
    else:
        fatores = np.array([1.0], dtype=float)

    melhor = None
    for f in fatores:
        bx = expandir_bbox_xy_pct(bbox0, base_x * float(f), base_y * float(f), (h_img, w_img))
        bx = ajustar_bbox_aspecto(bx, (h_img, w_img), aspect_min, aspect_max)
        x1, y1, x2, y2 = map(int, bx)
        crop = img_bgr[y1:y2, x1:x2].copy()
        if crop.size == 0:
            continue

        cap = score_arcos_tampa(pre, bx, p) if bool(_pget(p, 'cap_score_enabled', True)) else {'cap_score':0,'rect_penalty':0,'top_score':0,'bottom_score':0,'top_contour':None,'bottom_contour':None}
        svm_score = 0.0
        pred = None
        if clf is not None:
            try:
                feat = extrair_hog_de_imagem(crop, p).reshape(1, -1)
                pred = int(clf.predict(feat)[0])
                try:
                    svm_score = float(clf.decision_function(feat)[0])
                except Exception:
                    svm_score = float(pred)
            except Exception:
                svm_score = -999.0

        geom_score = 0.5 * float(candidato.get('overlap', 0.0)) + 0.5 * (1.0 - min(1.0, float(candidato.get('angle_diff', 0.0)) / max(1e-6, float(_pget(p, 'pair_angle_tol_deg', 10.0)))))
        score_final = (
            float(_pget(p, 'score_weight_svm', 1.0)) * svm_score +
            float(_pget(p, 'score_weight_cap', 0.35)) * float(cap.get('cap_score', 0.0)) +
            float(_pget(p, 'score_weight_geom', 0.10)) * geom_score -
            float(_pget(p, 'rect_penalty_weight', 0.20)) * float(cap.get('rect_penalty', 0.0))
        )

        item = {
            'bbox': bx,
            'cap': cap,
            'score_final': score_final,
            'score_svm': svm_score,
            'pred': pred,
            'factor': float(f),
            'geom_score': geom_score,
        }
        if melhor is None or item['score_final'] > melhor['score_final']:
            melhor = item

    if melhor is None:
        cap = score_arcos_tampa(pre, bbox0, p) if bool(_pget(p, 'cap_score_enabled', True)) else {'cap_score':0,'rect_penalty':0,'top_score':0,'bottom_score':0,'top_contour':None,'bottom_contour':None}
        return bbox0, cap, {'refine_mode': mode, 'tested': 0, 'score_final': None, 'score_svm': None}

    return melhor['bbox'], melhor['cap'], {
        'refine_mode': mode,
        'tested': int(len(fatores)),
        'factor': melhor['factor'],
        'score_final': melhor['score_final'],
        'score_svm': melhor['score_svm'],
        'pred_refine': melhor['pred'],
        'geom_score': melhor['geom_score'],
    }


def nms_bboxes_refinado(resultados_pos, iou_thr=0.30, max_det=8):
    if not resultados_pos:
        return []
    items = []
    for r in resultados_pos:
        if not _bbox_valida(r.get('bbox')):
            continue
        score = r.get('score_final_float', r.get('score_float', 0.0))
        if score is None:
            score = 0.0
        items.append((float(score), r))
    items.sort(key=lambda x: x[0], reverse=True)

    keep = []
    for _, r in items:
        if len(keep) >= int(max_det):
            break
        if all(iou_bbox(r['bbox'], k['bbox']) <= float(iou_thr) for k in keep):
            keep.append(r)
    return keep


def detectar_candidatos_cilindro(img_bgr, clf=None, p=None, score_min=None, nms_iou=None, max_det=None, aplicar_nms=True):
    """Aplica a etapa completa: Hough vertical → ROI → refino → arcos → HOG/SVM → NMS."""
    if p is None:
        p = STATE['params']
    score_min = float(_pget(p, 'det_score_min', 0.0) if score_min is None else score_min)
    nms_iou = float(_pget(p, 'det_nms_iou', 0.30) if nms_iou is None else nms_iou)
    max_det = int(_pget(p, 'det_max_det', 8) if max_det is None else max_det)

    pre = aplicar_preprocessamento(img_bgr, p)
    # Hough continua usando a imagem morfologicamente filtrada, favorecendo verticais.
    lines = detectar_linhas_hough(pre['morph'], p)
    candidatos = encontrar_rois_por_pares_de_linhas(lines, pre['morph'].shape, p)

    resultados = []
    positivos = []
    erros = 0
    for idx, c in enumerate(candidatos, start=1):
        try:
            bbox_ref, cap, ref_info = refinar_bbox_candidato(img_bgr, pre, c, clf, p)
            x1, y1, x2, y2 = map(int, bbox_ref)
            crop = img_bgr[y1:y2, x1:x2].copy()
            if crop.size == 0:
                continue

            pred = None
            score_svm = ref_info.get('score_svm')
            score_final = ref_info.get('score_final')
            if clf is not None:
                # No modo parametrico, o score ainda não foi calculado dentro do refino.
                if score_svm is None:
                    feat = extrair_hog_de_imagem(crop, p).reshape(1, -1)
                    pred = int(clf.predict(feat)[0])
                    try:
                        score_svm = float(clf.decision_function(feat)[0])
                    except Exception:
                        score_svm = float(pred)
                    geom_score = 0.5 * float(c.get('overlap', 0.0)) + 0.5 * (1.0 - min(1.0, float(c.get('angle_diff', 0.0)) / max(1e-6, float(_pget(p, 'pair_angle_tol_deg', 10.0)))))
                    score_final = (
                        float(_pget(p, 'score_weight_svm', 1.0)) * score_svm +
                        float(_pget(p, 'score_weight_cap', 0.35)) * float(cap.get('cap_score', 0.0)) +
                        float(_pget(p, 'score_weight_geom', 0.10)) * geom_score -
                        float(_pget(p, 'rect_penalty_weight', 0.20)) * float(cap.get('rect_penalty', 0.0))
                    )
                else:
                    # Usa predição já obtida durante a busca local, ou recalcula se necessário.
                    pred = ref_info.get('pred_refine')
                    if pred is None:
                        feat = extrair_hog_de_imagem(crop, p).reshape(1, -1)
                        pred = int(clf.predict(feat)[0])
            else:
                pred = None
                score_svm = None
                score_final = None

            r = {
                'roi': idx,
                'pred': 'cilindro' if pred == 1 else ('negativo' if pred == 0 else 'sem_modelo'),
                'bbox_original': c.get('bbox'),
                'bbox': tuple(map(int, bbox_ref)),
                'score': None if score_svm is None else round(float(score_svm), 3),
                'score_float': score_svm,
                'score_final': None if score_final is None else round(float(score_final), 3),
                'score_final_float': score_final,
                'cap_top': round(float(cap.get('top_score', 0.0)), 3),
                'cap_bottom': round(float(cap.get('bottom_score', 0.0)), 3),
                'cap_score': round(float(cap.get('cap_score', 0.0)), 3),
                'rect_penalty': round(float(cap.get('rect_penalty', 0.0)), 3),
                'top_contour': cap.get('top_contour'),
                'bottom_contour': cap.get('bottom_contour'),
                'refine_mode': ref_info.get('refine_mode'),
                'refine_factor': ref_info.get('factor'),
                'dist_px': round(float(c.get('distance', 0.0)), 1),
                'overlap': round(float(c.get('overlap', 0.0)), 2),
                'ang_diff': round(float(c.get('angle_diff', 0.0)), 2),
            }
            resultados.append(r)
            if clf is not None and pred == 1 and (score_final is None or float(score_final) >= score_min):
                positivos.append(r)
        except Exception as e:
            erros += 1
            resultados.append({'roi': idx, 'pred': 'erro', 'bbox_original': c.get('bbox'), 'bbox': c.get('bbox'), 'erro': str(e)[:120]})

    positivos_finais = nms_bboxes_refinado(positivos, iou_thr=nms_iou, max_det=max_det) if aplicar_nms else positivos
    ids_finais = {r['roi'] for r in positivos_finais}
    for r in resultados:
        r['nms_keep'] = bool(r.get('roi') in ids_finais)
        r['pred_visual'] = 'suprimido' if r.get('pred') == 'cilindro' and not r['nms_keep'] else r.get('pred')

    info = {
        'pred_raw': int(len(positivos)),
        'pred_final': int(len(positivos_finais)),
        'erro_rois': int(erros),
        'score_min': score_min,
        'nms_iou': nms_iou,
        'max_det': max_det,
    }
    return {
        'pre': pre,
        'lines': lines,
        'candidatos': candidatos,
        'resultados': resultados,
        'positivos_finais': positivos_finais,
        'info': info,
    }


def associar_predicoes_gt(pred_boxes, gt_boxes, iou_thr=0.50):
    """Associa predições a GT por IoU de forma gulosa."""
    pred_boxes = [tuple(map(float, b)) for b in pred_boxes if _bbox_valida(b)]
    gt_boxes = [tuple(map(float, b)) for b in gt_boxes if _bbox_valida(b)]
    pares = []
    for i, pb in enumerate(pred_boxes):
        for j, gb in enumerate(gt_boxes):
            pares.append((iou_bbox(pb, gb), i, j))
    pares.sort(reverse=True, key=lambda x: x[0])

    usados_p = set()
    usados_g = set()
    ious_tp = []
    for iou, i, j in pares:
        if iou < float(iou_thr):
            break
        if i in usados_p or j in usados_g:
            continue
        usados_p.add(i)
        usados_g.add(j)
        ious_tp.append(float(iou))

    tp = len(ious_tp)
    fp = max(0, len(pred_boxes) - tp)
    fn = max(0, len(gt_boxes) - tp)
    return {'tp': tp, 'fp': fp, 'fn': fn, 'ious_tp': ious_tp, 'n_pred': len(pred_boxes), 'n_gt': len(gt_boxes)}


def _metricas_det_agregadas(contadores, iou_thr):
    tp = int(contadores.get('tp', 0)); fp = int(contadores.get('fp', 0)); fn = int(contadores.get('fn', 0))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    ious = contadores.get('ious_tp', [])
    return {
        f'precision_det_{int(iou_thr*100)}': prec,
        f'recall_det_{int(iou_thr*100)}': rec,
        f'f1_det_{int(iou_thr*100)}': f1,
        f'tp_det_{int(iou_thr*100)}': tp,
        f'fp_det_{int(iou_thr*100)}': fp,
        f'fn_det_{int(iou_thr*100)}': fn,
        f'mean_iou_tp_{int(iou_thr*100)}': float(np.mean(ious)) if ious else 0.0,
    }


def avaliar_detector_completo(df, clf, p, status_callback=None):
    """Avalia o detector completo em imagens selecionadas de teste, comparando com YOLO por IoU."""
    if df is None or len(df) == 0 or clf is None:
        return {}

    usar_selecao = bool(_pget(p, 'usar_selecao_salva', True))
    df_eval = aplicar_selecao_salva(df, 'test', usar_selecao=usar_selecao)
    if df_eval is None or len(df_eval) == 0:
        df_eval = aplicar_selecao_salva(df, 'train', usar_selecao=usar_selecao)
    if df_eval is None or len(df_eval) == 0:
        return {}

    max_imgs = int(_pget(p, 'det_eval_max_images', 120))
    if max_imgs > 0 and len(df_eval) > max_imgs:
        df_eval = df_eval.sample(max_imgs, random_state=int(_pget(p, 'random_state', 42))).reset_index(drop=True)
    else:
        df_eval = df_eval.reset_index(drop=True)

    thr_main = float(_pget(p, 'det_iou_thr', 0.50))
    thrs = sorted(set([0.30, 0.50, round(thr_main, 2)]))
    cont = {thr: {'tp': 0, 'fp': 0, 'fn': 0, 'ious_tp': []} for thr in thrs}
    n_gt_total = 0
    n_pred_total = 0
    best_ious_all = []
    erros = 0

    for k, row in df_eval.iterrows():
        try:
            img_bgr, boxes_gt, _ = carregar_imagem_e_bboxes_row(row)
            det = detectar_candidatos_cilindro(
                img_bgr, clf=clf, p=p,
                score_min=float(_pget(p, 'det_score_min', 0.0)),
                nms_iou=float(_pget(p, 'det_nms_iou', 0.30)),
                max_det=int(_pget(p, 'det_max_det', 8)),
                aplicar_nms=True
            )
            pred_boxes = [r['bbox'] for r in det['positivos_finais'] if _bbox_valida(r.get('bbox'))]
            gt_boxes = [b for b in boxes_gt if _bbox_valida(b)]
            n_gt_total += len(gt_boxes)
            n_pred_total += len(pred_boxes)
            for pb in pred_boxes:
                best_ious_all.append(max([iou_bbox(pb, gb) for gb in gt_boxes], default=0.0))
            for thr in thrs:
                m = associar_predicoes_gt(pred_boxes, gt_boxes, iou_thr=thr)
                cont[thr]['tp'] += m['tp']; cont[thr]['fp'] += m['fp']; cont[thr]['fn'] += m['fn']; cont[thr]['ious_tp'].extend(m['ious_tp'])
        except Exception:
            erros += 1
        if status_callback is not None and (k + 1) % 25 == 0:
            status_callback(f'Avaliação detector completo: {k+1}/{len(df_eval)} imagens...')

    out = {
        'det_eval_imgs': int(len(df_eval)),
        'n_gt_det': int(n_gt_total),
        'n_pred_det': int(n_pred_total),
        'det_erros': int(erros),
        'det_iou_thr': float(thr_main),
        'mean_iou_best': float(np.mean(best_ious_all)) if best_ious_all else 0.0,
        'median_iou_best': float(np.median(best_ious_all)) if best_ious_all else 0.0,
    }
    for thr in thrs:
        out.update(_metricas_det_agregadas(cont[thr], thr))
    # aliases para a IoU principal escolhida no widget
    key = int(thr_main * 100)
    for prefix in ['precision_det', 'recall_det', 'f1_det', 'tp_det', 'fp_det', 'fn_det', 'mean_iou_tp']:
        out[prefix] = out.get(f'{prefix}_{key}', 0.0)
    return out


# ------------------------------------------------------------
# Visualizações atualizadas: ROI original, ROI refinada e arcos encontrados
# ------------------------------------------------------------
COLOR_LINE_CANDIDATE = (255, 0, 0)  # vermelho: retas usadas para formar a ROI candidata

def desenhar_refinamento_rois(img_rgb, candidatos, pre, p, max_draw=30):
    img = img_rgb.copy()
    dummy_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    for i, c in enumerate(candidatos[:max_draw], start=1):
        if not _bbox_valida(c.get('bbox')):
            continue
        # Desenha explicitamente as retas usadas para formar a ROI candidata.
        for line in [c.get('line1'), c.get('line2')]:
            if line is None:
                continue
            try:
                lx1, ly1, lx2, ly2 = map(int, line)
                cv2.line(img, (lx1, ly1), (lx2, ly2), COLOR_LINE_CANDIDATE, 2)
            except Exception:
                pass

        x1, y1, x2, y2 = map(int, c['bbox'])
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 120, 0), 1)
        try:
            bbox_ref, cap, _ = refinar_bbox_candidato(dummy_bgr, pre, c, None, p)
            rx1, ry1, rx2, ry2 = map(int, bbox_ref)
            cv2.rectangle(img, (rx1, ry1), (rx2, ry2), (0, 255, 80), 2)
            for contour in [cap.get('top_contour'), cap.get('bottom_contour')]:
                if contour is not None and len(contour) >= 2:
                    cv2.polylines(img, [contour.reshape(-1, 1, 2)], False, (0, 220, 255), 2)
            cv2.putText(img, f'R{i} cap={cap.get("cap_score",0):.2f}', (rx1, max(14, ry1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 255, 80), 1, cv2.LINE_AA)
        except Exception:
            cv2.putText(img, f'R{i}', (x1, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 120, 0), 1, cv2.LINE_AA)
    return img


def render_rois_refinadas():
    pre = obter_preprocessamento_referencia()
    if pre is None:
        print('Nenhuma imagem de referência selecionada.')
        return
    lines = detectar_linhas_hough(pre['morph'], STATE['params'])
    candidatos = encontrar_rois_por_pares_de_linhas(lines, pre['morph'].shape, STATE['params'])
    img_rois = desenhar_refinamento_rois(pre['rgb'], candidatos, pre, STATE['params'])

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(img_rois)
    ax.set_title(f'ROIs geométricas + refino + arcos de tampa — {len(candidatos)} candidatas')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    if len(candidatos) > 0:
        rows = []
        dummy_bgr = cv2.cvtColor(pre['rgb'], cv2.COLOR_RGB2BGR)
        for i, c in enumerate(candidatos[:30], start=1):
            bbox_ref, cap, info = refinar_bbox_candidato(dummy_bgr, pre, c, None, STATE['params'])
            rows.append({
                'roi': i,
                'bbox_hough': c['bbox'],
                'bbox_refinada': bbox_ref,
                'dist_px': round(c['distance'], 1),
                'ang_diff': round(c['angle_diff'], 1),
                'overlap': round(c['overlap'], 2),
                'aspect': round(c['aspect'], 2),
                'cap_top': round(cap.get('top_score', 0), 3),
                'cap_bottom': round(cap.get('bottom_score', 0), 3),
                'cap_score': round(cap.get('cap_score', 0), 3),
                'penal_ret': round(cap.get('rect_penalty', 0), 3),
            })
        exibir_tabela_compacta(pd.DataFrame(rows), max_rows=12)
    else:
        print('Nenhum par de retas passou pelos filtros geométricos.')


def render_hog_visualizacao_refinada():
    pre = obter_preprocessamento_referencia()
    img_bgr_ref = obter_imagem_referencia()
    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada.')
        return
    lines = detectar_linhas_hough(pre['morph'], STATE['params'])
    candidatos = encontrar_rois_por_pares_de_linhas(lines, pre['morph'].shape, STATE['params'])
    idx = atualizar_opcoes_roi_hog(candidatos)
    if idx is None or len(candidatos) == 0:
        print('Nenhuma ROI candidata foi gerada para esta imagem com os parâmetros atuais.')
        return
    idx = int(idx)
    if idx < 0 or idx >= len(candidatos):
        idx = 0
    c = candidatos[idx]
    bbox_ref, cap, ref_info = refinar_bbox_candidato(img_bgr_ref, pre, c, None, STATE['params'])
    x1, y1, x2, y2 = map(int, bbox_ref)
    crop_bgr = img_bgr_ref[y1:y2, x1:x2].copy()
    if crop_bgr.size == 0:
        print('A ROI refinada selecionada gerou um recorte vazio.')
        return
    try:
        hog_input_resized, hog_image, features_hog = extrair_hog_visualizacao_de_imagem(crop_bgr, STATE['params'])
        features_total = extrair_hog_de_imagem(crop_bgr, STATE['params'])
    except Exception as e:
        print('Não foi possível gerar a visualização HOG:', e)
        traceback.print_exc(limit=1)
        return

    img_contexto = pre['rgb'].copy()
    ox1, oy1, ox2, oy2 = map(int, c['bbox'])
    cv2.rectangle(img_contexto, (ox1, oy1), (ox2, oy2), (255, 120, 0), 1)
    cv2.rectangle(img_contexto, (x1, y1), (x2, y2), (0, 255, 80), 2)
    for line in [c.get('line1'), c.get('line2')]:
        if line is None:
            continue
        lx1, ly1, lx2, ly2 = map(int, line)
        cv2.line(img_contexto, (lx1, ly1), (lx2, ly2), COLOR_LINE_CANDIDATE, 2)
    for contour in [cap.get('top_contour'), cap.get('bottom_contour')]:
        if contour is not None and len(contour) >= 2:
            cv2.polylines(img_contexto, [contour.reshape(-1, 1, 2)], False, (0, 220, 255), 2)

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    fig, axs = plt.subplots(1, 4, figsize=(16, 4.6))
    axs[0].imshow(img_contexto); axs[0].set_title('ROI original/refinada\nretas em vermelho + arcos')
    axs[1].imshow(crop_rgb); axs[1].set_title('Recorte refinado')
    axs[2].imshow(hog_input_resized, cmap='gray'); axs[2].set_title(f"Entrada do HOG\n{STATE['params']['hog_input']}")
    axs[3].imshow(hog_image, cmap='gray'); axs[3].set_title(f'HOG\nbase={len(features_hog)} | total={len(features_total)}')
    for ax in axs: ax.axis('off')
    plt.tight_layout(); plt.show()

    df_info = pd.DataFrame([{
        'roi': idx + 1,
        'bbox_hough': c['bbox'],
        'bbox_refinada': bbox_ref,
        'cap_top': round(cap.get('top_score', 0), 3),
        'cap_bottom': round(cap.get('bottom_score', 0), 3),
        'cap_score': round(cap.get('cap_score', 0), 3),
        'penal_ret': round(cap.get('rect_penalty', 0), 3),
        'n_hog_base': len(features_hog),
        'n_features_total': len(features_total),
        'features_arcos_svm': bool(STATE['params'].get('svm_usar_features_arcos', True)),
    }])
    exibir_tabela_compacta(df_info, max_rows=1)

# Substitui as funções registradas anteriormente por versões com refino/arcos.
if 'STATE' in globals() and 'renderers' in STATE:
    if 'rois' in STATE['renderers']:
        STATE['renderers']['rois']['fn'] = render_rois_refinadas
    if 'hog_visualizacao' in STATE['renderers']:
        STATE['renderers']['hog_visualizacao']['fn'] = render_hog_visualizacao_refinada
    atualizar_renderer('rois')
    atualizar_renderer('hog_visualizacao')


## Detector orientado por linhas, arcos e expansão guiada pelo SVM

Esta célula substitui a geração antiga baseada apenas em pares verticais por uma versão mais geral:

```text
Hough em qualquer direção
→ união de segmentos colineares separados por GAP
→ retas remanescentes como base
→ busca de arcos/tampas apenas nas proximidades dessas retas
→ geração de caixas orientadas
→ expansão paralela à reta até arcos/limite
→ expansão perpendicular até reta paralela/limite
→ score combinado: SVM + arcos + geometria - penalidade retangular
→ NMS + IoU na validação
```

O treinamento continua usando as imagens selecionadas manualmente, incluindo as versões com zoom out salvas quando marcadas. A diferença é que a aplicação/validação do detector passa a usar a estratégia orientada por linhas e arcos.


In [ ]:

# ============================================================
# 11C. Detector orientado por linhas + arcos + expansão guiada pelo SVM
# ============================================================
# Esta célula deve ficar antes do treinamento. Ela preserva a seleção manual
# e as imagens com zoom out já salvas, mas altera a geração/validação das ROIs.

if 'PARAM_WIDGETS' not in globals():
    raise RuntimeError('Execute primeiro a célula de ajuste de parâmetros.')

# Guarda versões anteriores para permitir comparação.
if 'detectar_candidatos_cilindro_legado_pares' not in globals() and 'detectar_candidatos_cilindro' in globals():
    detectar_candidatos_cilindro_legado_pares = detectar_candidatos_cilindro
if 'render_rois_refinadas_legado' not in globals() and 'render_rois_refinadas' in globals():
    render_rois_refinadas_legado = render_rois_refinadas
if 'render_hog_visualizacao_refinada_legado' not in globals() and 'render_hog_visualizacao_refinada' in globals():
    render_hog_visualizacao_refinada_legado = render_hog_visualizacao_refinada


def _add_param_widget(nome, widget, ajuda=''):
    """Adiciona widget novo sem duplicar se a célula for reexecutada."""
    if nome not in PARAM_WIDGETS:
        PARAM_WIDGETS[nome] = widget
        try:
            PARAM_WIDGETS[nome].observe(on_params_change, names='value')
        except Exception:
            pass
    if 'PARAM_HELP' in globals():
        PARAM_HELP[nome] = ajuda
    return PARAM_WIDGETS[nome]


_add_param_widget(
    'detector_strategy',
    widgets.Dropdown(
        options=[
            ('linhas + arcos orientados', 'linhas_arcos_orientado'),
            ('pares verticais legados', 'pares_verticais')
        ],
        value='linhas_arcos_orientado',
        description='estratégia:',
        layout=widgets.Layout(width='390px')
    ),
    'Define se o detector usa a estratégia nova por linhas/arcos orientados ou a versão anterior por pares verticais.'
)

_add_param_widget(
    'hough_source_any',
    widgets.Dropdown(
        options=[
            ('Mapa A: Canny completo', 'canny_full'),
            ('Mapa B: Canny filtrado p/ Hough', 'canny_hough'),
            ('fechamento sobre mapa B', 'morph_close'),
            ('morfologia final sobre mapa B', 'morph')
        ],
        value='morph',
        description='Hough em:',
        layout=widgets.Layout(width='390px')
    ),
    'Fonte de bordas para a Hough em qualquer direção. O padrão usa a morfologia final sobre o mapa B; os arcos continuam sendo buscados no mapa A completo.'
)

_add_param_widget(
    'merge_collinear_enabled',
    widgets.Checkbox(value=True, description='unir colineares', indent=False, layout=widgets.Layout(width='210px')),
    'Une segmentos de reta quase colineares separados por pequenos gaps.'
)
_add_param_widget(
    'merge_gap_px',
    widgets.IntSlider(value=18, min=0, max=80, step=1, description='gap colinear', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Gap máximo, em pixels, para unir segmentos colineares.'
)
_add_param_widget(
    'merge_angle_tol_deg',
    widgets.FloatSlider(value=5.0, min=0.5, max=20.0, step=0.5, description='tol ângulo', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Diferença angular máxima para considerar duas retas colineares/paralelas.'
)
_add_param_widget(
    'merge_perp_tol_px',
    widgets.IntSlider(value=8, min=1, max=40, step=1, description='tol perp', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Distância transversal máxima para unir segmentos aproximadamente sobre a mesma reta.'
)

_add_param_widget(
    'oriented_parallel_expand_pct',
    widgets.IntSlider(value=60, min=0, max=180, step=5, description='exp paralelo %', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Expansão máxima no eixo paralelo à reta. Serve para completar topo/base até curvas ou limite.'
)
_add_param_widget(
    'oriented_parallel_step_px',
    widgets.IntSlider(value=6, min=2, max=24, step=1, description='passo paralelo', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Passo da busca paralela às retas, em pixels.'
)
_add_param_widget(
    'oriented_cap_stop_score',
    widgets.FloatSlider(value=0.18, min=0.0, max=1.0, step=0.02, description='stop arco', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Score mínimo de bordas/arco para parar a expansão paralela.'
)
_add_param_widget(
    'oriented_perp_extra_pct',
    widgets.IntSlider(value=12, min=0, max=80, step=2, description='extra perp %', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Expansão extra perpendicular após encontrar a reta paralela oposta.'
)
_add_param_widget(
    'oriented_min_cap_edges',
    widgets.IntSlider(value=6, min=1, max=80, step=1, description='min bordas arco', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Quantidade mínima de pixels de borda em uma faixa para considerar arco/tampa.'
)
_add_param_widget(
    'oriented_arc_margin_px',
    widgets.IntSlider(value=8, min=2, max=40, step=1, description='margem arco', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Margem ao redor da caixa orientada para procurar arcos próximos às extremidades.'
)
_add_param_widget(
    'oriented_use_score_search',
    widgets.Checkbox(value=True, description='usar busca por score', indent=False, layout=widgets.Layout(width='230px')),
    'Testa várias expansões e escolhe a melhor por score SVM + arcos + geometria.'
)


_add_param_widget(
    'oriented_accept_gap_pairs',
    widgets.Checkbox(value=True, description='aceitar par com gap', indent=False, layout=widgets.Layout(width='210px')),
    'Aceita duas retas paralelas mesmo quando a sobreposição longitudinal é baixa, desde que o gap ao longo do eixo seja pequeno.'
)
_add_param_widget(
    'oriented_pair_axis_gap_px',
    widgets.IntSlider(value=45, min=0, max=180, step=5, description='gap eixo', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Gap longitudinal máximo entre duas retas paralelas para ainda formar uma ROI. Ajuda em cilindros inclinados com bordas detectadas em trechos diferentes.'
)
_add_param_widget(
    'oriented_use_union_interval',
    widgets.Checkbox(value=True, description='unir extensão do par', indent=False, layout=widgets.Layout(width='220px')),
    'Usa a união longitudinal dos segmentos paralelos para formar a ROI, em vez de usar apenas a faixa de sobreposição.'
)
_add_param_widget(
    'single_line_candidates_enabled',
    widgets.Checkbox(value=True, description='candidatos por reta única', indent=False, layout=widgets.Layout(width='240px')),
    'Gera candidatos também a partir de uma única reta forte, expandindo perpendicularmente para os dois lados. Útil quando uma das laterais do cilindro está fraca.'
)
_add_param_widget(
    'single_line_width_px',
    widgets.IntSlider(value=60, min=10, max=260, step=5, description='largura reta única', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Largura inicial, em pixels, das ROIs criadas a partir de uma reta única.'
)
_add_param_widget(
    'single_line_max',
    widgets.IntSlider(value=20, min=0, max=120, step=5, description='máx reta única', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Número máximo de retas filtradas que podem gerar candidatos por reta única.'
)

_add_param_widget(
    'pair_rank_len_weight',
    widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.05, description='peso comp.', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Peso do comprimento das linhas no ranqueamento de pares orientados. Ajuda a priorizar pares fortes e longos.'
)
_add_param_widget(
    'pair_rank_overlap_weight',
    widgets.FloatSlider(value=0.30, min=0.0, max=1.0, step=0.05, description='peso overlap', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Peso da sobreposição longitudinal no ranqueamento de pares orientados.'
)
_add_param_widget(
    'pair_rank_dist_weight',
    widgets.FloatSlider(value=0.20, min=0.0, max=1.0, step=0.05, description='peso dist.', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Peso da distância transversal plausível no ranqueamento de pares orientados.'
)
_add_param_widget(
    'pair_rank_angle_weight',
    widgets.FloatSlider(value=0.15, min=0.0, max=1.0, step=0.05, description='peso âng.', continuous_update=False, layout=widgets.Layout(width='390px')),
    'Peso da similaridade angular no ranqueamento de pares orientados.'
)

# Garante que a busca local por score seja o padrão quando o refino antigo existir.
try:
    PARAM_WIDGETS['bbox_refine_mode'].value = 'score_local'
except Exception:
    pass

definir_params_a_partir_widgets(PARAM_WIDGETS)


# ------------------------------------------------------------
# Setup específico do refinamento/expansão orientada
# ------------------------------------------------------------
# Observação: esses parâmetros também ficam dentro do setup paramétrico geral,
# mas este bloco permite salvar/carregar apenas a configuração de refino/expansão,
# facilitando comparar ajustes locais sem alterar todo o setup de pré-processamento.

if 'REFINE_SETUPS_PATH' not in globals():
    if 'USER_REFINE_SETUPS_PATH' in globals():
        REFINE_SETUPS_PATH = USER_REFINE_SETUPS_PATH
    else:
        REFINE_SETUPS_PATH = CLASSICAL_DIR / 'setups_refinamento_expansao.json'

REFINE_SETUP_DEFAULT_ID = 'REFINO_ATUAL'
REFINE_PARAM_KEYS = [
    'detector_strategy', 'hough_source_any',
    'merge_collinear_enabled', 'merge_gap_px', 'merge_angle_tol_deg', 'merge_perp_tol_px',
    'oriented_accept_gap_pairs', 'oriented_pair_axis_gap_px', 'oriented_use_union_interval',
    'single_line_candidates_enabled', 'single_line_width_px', 'single_line_max',
    'pair_rank_len_weight', 'pair_rank_overlap_weight', 'pair_rank_dist_weight', 'pair_rank_angle_weight',
    'oriented_use_score_search', 'oriented_parallel_expand_pct', 'oriented_parallel_step_px',
    'oriented_cap_stop_score', 'oriented_perp_extra_pct', 'oriented_min_cap_edges',
    'oriented_arc_margin_px'
]

def _carregar_json_refino(path, default):
    try:
        return carregar_json(path, default=default)
    except TypeError:
        return carregar_json(path, default)
    except Exception:
        if Path(path).exists():
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except Exception:
                return default
        return default


def _salvar_json_refino(path, data):
    try:
        salvar_json(path, data)
    except Exception:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)


def carregar_setups_refino():
    data = _carregar_json_refino(REFINE_SETUPS_PATH, default=None)
    if not isinstance(data, dict):
        data = {'version': 1, 'setups': {}}
    data.setdefault('version', 1)
    data.setdefault('setups', {})
    return data


def salvar_setups_refino(data):
    data.setdefault('version', 1)
    data.setdefault('setups', {})
    _salvar_json_refino(REFINE_SETUPS_PATH, data)


def proximo_refine_setup_id(data):
    existentes = list(data.get('setups', {}).keys())
    nums = []
    for sid in existentes:
        if isinstance(sid, str) and sid.startswith('R'):
            try:
                nums.append(int(sid[1:]))
            except Exception:
                pass
    return f'R{(max(nums) + 1) if nums else 1:03d}'


def capturar_params_refino():
    return {k: PARAM_WIDGETS[k].value for k in REFINE_PARAM_KEYS if k in PARAM_WIDGETS}


def aplicar_params_refino(params):
    for k, v in params.items():
        if k in PARAM_WIDGETS:
            try:
                PARAM_WIDGETS[k].value = v
            except Exception:
                pass
    try:
        definir_params_a_partir_widgets(PARAM_WIDGETS)
    except Exception:
        pass


w_refine_setup_selector = widgets.Dropdown(
    options=[('REFINO_ATUAL | parâmetros atuais/manuais', REFINE_SETUP_DEFAULT_ID)],
    value=REFINE_SETUP_DEFAULT_ID,
    description='setup refino:',
    layout=widgets.Layout(width='430px')
)
w_refine_setup_nome = widgets.Text(
    value='',
    placeholder='Nome do setup de refino/expansão',
    description='nome:',
    layout=widgets.Layout(width='430px')
)
btn_salvar_refine_setup = widgets.Button(
    description='Salvar refino',
    button_style='success',
    icon='save',
    layout=widgets.Layout(width='145px')
)
btn_excluir_refine_setup = widgets.Button(
    description='Excluir refino',
    button_style='danger',
    icon='trash',
    layout=widgets.Layout(width='145px')
)
btn_refine_atual = widgets.Button(
    description='Refino atual',
    button_style='info',
    icon='sliders',
    layout=widgets.Layout(width='135px')
)
out_refine_setup_status = widgets.Output()


def atualizar_status_refino(msg):
    with out_refine_setup_status:
        clear_output(wait=True)
        print(msg)
        print(f'Arquivo de setups de refino: {REFINE_SETUPS_PATH}')


def atualizar_dropdown_refino(selecionar=None):
    data = carregar_setups_refino()
    opcoes = [('REFINO_ATUAL | parâmetros atuais/manuais', REFINE_SETUP_DEFAULT_ID)]
    for sid in sorted(data.get('setups', {}).keys()):
        item = data['setups'][sid]
        nome = item.get('nome', sid)
        data_hora = item.get('data_hora', '')
        opcoes.append((f'{sid} | {nome} | {data_hora}', sid))
    valores = [v for _, v in opcoes]
    selecionar = selecionar if selecionar in valores else REFINE_SETUP_DEFAULT_ID
    w_refine_setup_selector.options = opcoes
    w_refine_setup_selector.value = selecionar


def carregar_refino_por_id(setup_id):
    if setup_id == REFINE_SETUP_DEFAULT_ID:
        STATE['refine_setup_id'] = REFINE_SETUP_DEFAULT_ID
        STATE['refine_setup_nome'] = 'parâmetros atuais/manuais'
        atualizar_status_refino('Usando parâmetros atuais/manuais de refino.')
        return
    data = carregar_setups_refino()
    item = data.get('setups', {}).get(setup_id)
    if not item:
        atualizar_status_refino(f'Setup de refino {setup_id} não encontrado.')
        return
    aplicar_params_refino(item.get('params', {}))
    STATE['refine_setup_id'] = setup_id
    STATE['refine_setup_nome'] = item.get('nome', setup_id)
    atualizar_status_refino(f'Setup de refino {setup_id} carregado.')
    try:
        atualizar_visualizacoes_parametros()
    except Exception:
        pass


def on_refine_selector_change(change):
    if change.get('name') == 'value' and change.get('new'):
        carregar_refino_por_id(change['new'])


def on_salvar_refine_setup(_):
    data = carregar_setups_refino()
    sid = proximo_refine_setup_id(data)
    nome = w_refine_setup_nome.value.strip() or f'Refino {sid}'
    data['setups'][sid] = {
        'nome': nome,
        'data_hora': time.strftime('%Y-%m-%d %H:%M:%S'),
        'params': capturar_params_refino()
    }
    salvar_setups_refino(data)
    STATE['refine_setup_id'] = sid
    STATE['refine_setup_nome'] = nome
    atualizar_dropdown_refino(sid)
    atualizar_status_refino(f'Setup de refino salvo como {sid}.')


def on_excluir_refine_setup(_):
    sid = w_refine_setup_selector.value
    if sid == REFINE_SETUP_DEFAULT_ID:
        atualizar_status_refino('O refino atual/manual não pode ser excluído.')
        return
    data = carregar_setups_refino()
    setups = data.get('setups', {})
    if sid in setups:
        nome = setups[sid].get('nome', sid)
        del setups[sid]
        salvar_setups_refino(data)
        STATE['refine_setup_id'] = REFINE_SETUP_DEFAULT_ID
        STATE['refine_setup_nome'] = 'parâmetros atuais/manuais'
        atualizar_dropdown_refino(REFINE_SETUP_DEFAULT_ID)
        atualizar_status_refino(f'Setup de refino {sid} — {nome} excluído.')
    else:
        atualizar_status_refino(f'Setup de refino {sid} não encontrado.')


def on_refine_atual(_):
    STATE['refine_setup_id'] = REFINE_SETUP_DEFAULT_ID
    STATE['refine_setup_nome'] = 'parâmetros atuais/manuais'
    atualizar_dropdown_refino(REFINE_SETUP_DEFAULT_ID)
    atualizar_status_refino('Mantidos os parâmetros atuais/manuais de refino.')


w_refine_setup_selector.observe(on_refine_selector_change, names='value')
btn_salvar_refine_setup.on_click(on_salvar_refine_setup)
btn_excluir_refine_setup.on_click(on_excluir_refine_setup)
btn_refine_atual.on_click(on_refine_atual)
atualizar_dropdown_refino(STATE.get('refine_setup_id', REFINE_SETUP_DEFAULT_ID))

_box_refine_setup = widgets.VBox([
    widgets.HTML('<b>Setups de refino/expansão orientada</b>'),
    widgets.HTML('<small>Salva apenas os parâmetros da estratégia de linhas/arcos, ranqueamento, pares inclinados, reta única e expansão orientada. O setup paramétrico geral continua salvando todos os parâmetros do notebook.</small>'),
    widgets.HBox([w_refine_setup_selector, btn_refine_atual, btn_salvar_refine_setup, btn_excluir_refine_setup], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([w_refine_setup_nome], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    out_refine_setup_status
], layout=widgets.Layout(border='1px solid #444', padding='8px', margin='6px 0'))

_box_linhas_arcos = widgets.VBox([
    widgets.HTML('<h4 style="margin:4px 0 2px 0;">Estratégia nova: linhas Hough + arcos + expansão orientada</h4>'),
    widgets.HTML('<small>Hough pode detectar retas em qualquer direção. O mapa A mantém o Canny completo para arcos/gradientes; o mapa B limpa as bordas para Hough. Segmentos colineares são unidos por GAP. Pares inclinados podem ser aceitos por sobreposição ou por gap longitudinal, e também há fallback por reta única.</small>'),
    _box_refine_setup,
    widgets.HBox([
        param_item('detector_strategy'),
        param_item('hough_source_any')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        PARAM_WIDGETS['merge_collinear_enabled'],
        param_item('merge_gap_px'),
        param_item('merge_angle_tol_deg'),
        param_item('merge_perp_tol_px')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        PARAM_WIDGETS['oriented_accept_gap_pairs'],
        PARAM_WIDGETS['oriented_use_union_interval'],
        param_item('oriented_pair_axis_gap_px')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        PARAM_WIDGETS['single_line_candidates_enabled'],
        param_item('single_line_width_px'),
        param_item('single_line_max')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        param_item('pair_rank_len_weight'),
        param_item('pair_rank_overlap_weight'),
        param_item('pair_rank_dist_weight'),
        param_item('pair_rank_angle_weight')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        PARAM_WIDGETS['oriented_use_score_search'],
        param_item('oriented_parallel_expand_pct'),
        param_item('oriented_parallel_step_px'),
        param_item('oriented_cap_stop_score')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([
        param_item('oriented_perp_extra_pct'),
        param_item('oriented_min_cap_edges'),
        param_item('oriented_arc_margin_px')
    ], layout=widgets.Layout(flex_flow='row wrap', gap='8px'))
])

display(_box_linhas_arcos)


# ------------------------------------------------------------
# Geometria de linhas e caixas orientadas
# ------------------------------------------------------------
def _line_info(line):
    x1, y1, x2, y2 = map(float, line)
    dx = x2 - x1
    dy = y2 - y1
    L = float(math.hypot(dx, dy))
    if L < 1e-6:
        return None
    ux, uy = dx / L, dy / L
    # Normaliza orientação para ângulos equivalentes em [0, 180).
    angle = (math.degrees(math.atan2(uy, ux)) + 180.0) % 180.0
    cx, cy = (x1 + x2) * 0.5, (y1 + y2) * 0.5
    return {
        'line': tuple(map(int, line)),
        'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
        'length': L,
        'u': np.array([ux, uy], dtype=float),
        'v': np.array([-uy, ux], dtype=float),
        'center': np.array([cx, cy], dtype=float),
        'angle': angle
    }


def _angle_diff_180(a, b):
    d = abs(float(a) - float(b)) % 180.0
    return min(d, 180.0 - d)


def _interval_gap(a1, a2, b1, b2):
    if a2 < b1:
        return b1 - a2
    if b2 < a1:
        return a1 - b2
    return 0.0


def _projecoes_linha_em_base(line, origin, u, v):
    pts = np.array([[line[0], line[1]], [line[2], line[3]]], dtype=float)
    rel = pts - origin.reshape(1, 2)
    ts = rel @ u
    ss = rel @ v
    return float(np.min(ts)), float(np.max(ts)), float(np.mean(ss))


def unir_segmentos_colineares(lines, p):
    """Une segmentos de reta quase colineares separados por gap pequeno.

    Revisão v23:
    - os intervalos projetados são sempre ordenados antes de calcular gap;
    - a orientação da reta é comparada módulo 180°, evitando rejeitar segmentos equivalentes com sentido oposto;
    - isso reduz falhas de união em cilindros inclinados.
    """
    if not bool(_pget(p, 'merge_collinear_enabled', True)):
        return [tuple(map(int, l)) for l in lines]

    infos = [_line_info(l) for l in lines]
    infos = [i for i in infos if i is not None and i['length'] >= float(_pget(p, 'line_length_min', 0))]
    if not infos:
        return []

    gap_max = float(_pget(p, 'merge_gap_px', 18))
    angle_tol = float(_pget(p, 'merge_angle_tol_deg', 5.0))
    perp_tol = float(_pget(p, 'merge_perp_tol_px', 8.0))

    infos = sorted(infos, key=lambda d: d['length'], reverse=True)
    usados = set()
    merged = []

    for i, base in enumerate(infos):
        if i in usados:
            continue

        origin = base['center']
        u = base['u'].copy()
        v = base['v'].copy()
        t_vals = []
        s_vals = []
        grupo = []

        t1, t2, s0 = _projecoes_linha_em_base(base['line'], origin, u, v)
        intervalo_atual = [min(t1, t2), max(t1, t2)]

        changed = True
        while changed:
            changed = False
            for j, other in enumerate(infos):
                if j in usados or j == i:
                    continue
                if _angle_diff_180(base['angle'], other['angle']) > angle_tol:
                    continue
                ot1, ot2, os = _projecoes_linha_em_base(other['line'], origin, u, v)
                omin, omax = min(ot1, ot2), max(ot1, ot2)
                if abs(os) > perp_tol:
                    continue
                if _interval_gap(intervalo_atual[0], intervalo_atual[1], omin, omax) > gap_max:
                    continue
                usados.add(j)
                grupo.append(other)
                intervalo_atual[0] = min(intervalo_atual[0], omin)
                intervalo_atual[1] = max(intervalo_atual[1], omax)
                changed = True

        usados.add(i)
        grupo.append(base)

        for g in grupo:
            gt1, gt2, gs = _projecoes_linha_em_base(g['line'], origin, u, v)
            t_vals.extend([gt1, gt2])
            s_vals.append(gs)

        tmin, tmax = float(np.min(t_vals)), float(np.max(t_vals))
        smed = float(np.mean(s_vals)) if s_vals else 0.0
        p1 = origin + u * tmin + v * smed
        p2 = origin + u * tmax + v * smed
        merged.append(tuple(map(int, [round(p1[0]), round(p1[1]), round(p2[0]), round(p2[1])])))

    return merged


def filtrar_linhas_remanescentes(lines, p):
    """Filtra linhas por comprimento. A orientação fica livre por padrão."""
    out = []
    Lmin = float(_pget(p, 'line_length_min', 0))
    Lmax = float(_pget(p, 'line_length_max', 999999))
    for line in lines:
        info = _line_info(line)
        if info is None:
            continue
        if info['length'] < Lmin or info['length'] > Lmax:
            continue
        out.append(tuple(map(int, line)))
    return out


def _bbox_axis_from_points(pts, shape, margin=0):
    h, w = shape[:2]
    pts = np.asarray(pts, dtype=float)
    x1 = int(max(0, math.floor(np.min(pts[:, 0]) - margin)))
    y1 = int(max(0, math.floor(np.min(pts[:, 1]) - margin)))
    x2 = int(min(w, math.ceil(np.max(pts[:, 0]) + margin)))
    y2 = int(min(h, math.ceil(np.max(pts[:, 1]) + margin)))
    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None
    return (x1, y1, x2, y2)


def _oriented_corners(origin, u, v, tmin, tmax, smin, smax):
    pts = [
        origin + u * tmin + v * smin,
        origin + u * tmax + v * smin,
        origin + u * tmax + v * smax,
        origin + u * tmin + v * smax,
    ]
    return np.asarray(pts, dtype=np.float32)


def _edge_points(edges):
    ys, xs = np.nonzero(edges)
    if len(xs) == 0:
        return np.empty((0, 2), dtype=float)
    return np.column_stack([xs, ys]).astype(float)


def _cap_score_orientado(edge_pts, origin, u, v, tmin, tmax, smin, smax, p, lado='top'):
    """Pontua presença de arco/tampa em faixa próxima à extremidade tmin/tmax."""
    if edge_pts is None or len(edge_pts) == 0:
        return {'score': 0.0, 'n': 0, 'span': 0.0, 'pts': None}

    margin = float(_pget(p, 'oriented_arc_margin_px', 8))
    min_edges = int(_pget(p, 'oriented_min_cap_edges', 6))
    rel = edge_pts - origin.reshape(1, 2)
    ts = rel @ u
    ss = rel @ v

    # Analisa uma faixa próxima à extremidade paralela.
    if lado == 'top':
        mask_t = (ts >= tmax - margin) & (ts <= tmax + margin * 2.0)
    else:
        mask_t = (ts >= tmin - margin * 2.0) & (ts <= tmin + margin)

    mask_s = (ss >= smin - margin) & (ss <= smax + margin)
    pts = edge_pts[mask_t & mask_s]
    if len(pts) < min_edges:
        return {'score': 0.0, 'n': int(len(pts)), 'span': 0.0, 'pts': pts}

    rel2 = pts - origin.reshape(1, 2)
    ts2 = rel2 @ u
    ss2 = rel2 @ v
    width = max(1e-6, abs(smax - smin))
    span = float((np.max(ss2) - np.min(ss2)) / width)
    dens = min(1.0, len(pts) / max(1.0, width))
    # Não exige curvatura rigorosa: span + densidade já servem como evidência leve de tampa/canto/arco.
    score = float(np.clip(0.65 * min(1.0, span) + 0.35 * min(1.0, dens), 0.0, 1.0))
    return {'score': score, 'n': int(len(pts)), 'span': span, 'pts': pts}



def _score_par_orientado(a, b, dist, dist_min, dist_max, overlap, gap_axis, angle_diff, pair_tol, p):
    """Score geométrico para ranquear pares orientados antes do SVM.

    O objetivo não é classificar cilindro, mas evitar que pares inclinados fortes fiquem fora
    dos primeiros candidatos apenas porque outras regiões maiores produziram ROIs maiores.
    """
    len_ref = max(1.0, float(_pget(p, 'line_length_min', 30)))
    len_score = min(1.0, min(float(a['length']), float(b['length'])) / len_ref)
    len_balance = min(float(a['length']), float(b['length'])) / max(1e-6, max(float(a['length']), float(b['length'])))
    len_score = 0.65 * len_score + 0.35 * len_balance

    angle_score = 1.0 - min(1.0, float(angle_diff) / max(1e-6, float(pair_tol)))

    # Distância transversal é melhor quando está dentro da faixa e mais distante das bordas do intervalo.
    if dist_max <= dist_min:
        dist_score = 1.0
    else:
        center = 0.5 * (dist_min + dist_max)
        half = 0.5 * (dist_max - dist_min)
        dist_score = 1.0 - min(1.0, abs(float(dist) - center) / max(1e-6, half))
        # Não zera totalmente perto das bordas, pois cilindros reais podem estar no limite da escala.
        dist_score = 0.35 + 0.65 * dist_score

    # Overlap é útil, mas pares com pequeno gap longitudinal também podem ser reais.
    gap_score = 1.0 / (1.0 + max(0.0, float(gap_axis)) / max(1.0, float(_pget(p, 'oriented_pair_axis_gap_px', 45))))
    axis_score = max(float(overlap), 0.65 * gap_score)

    w_len = float(_pget(p, 'pair_rank_len_weight', 0.35))
    w_ov = float(_pget(p, 'pair_rank_overlap_weight', 0.30))
    w_dist = float(_pget(p, 'pair_rank_dist_weight', 0.20))
    w_ang = float(_pget(p, 'pair_rank_angle_weight', 0.15))
    w_sum = max(1e-6, w_len + w_ov + w_dist + w_ang)
    score = (w_len * len_score + w_ov * axis_score + w_dist * dist_score + w_ang * angle_score) / w_sum
    return float(score), {
        'len_score': float(len_score),
        'axis_score': float(axis_score),
        'dist_score': float(dist_score),
        'angle_score': float(angle_score)
    }


def gerar_pares_orientados(lines, img_shape, p, return_debug=False):
    """
    Gera candidatos orientados a partir de pares de retas paralelas em qualquer direção.

    Revisão v23:
    - paralelismo é comparado módulo 180°;
    - distância transversal e overlap são calculados no sistema local da reta base (u = eixo, v = normal);
    - os pares são ranqueados por um score geométrico, não apenas por área/overlap;
    - isso evita perder cilindros inclinados fortes quando há muitos candidatos no fundo.
    """
    infos = [_line_info(l) for l in lines]
    infos = [i for i in infos if i is not None]
    candidatos = []

    dbg = {
        'n_linhas_para_pares': len(infos),
        'n_pares_testados': 0,
        'n_pares_ang_ok': 0,
        'n_pares_dist_ok': 0,
        'n_pares_overlap_ok': 0,
        'n_pares_gap_ok': 0,
        'n_pares_rejeitados_overlap_gap': 0,
        'n_candidatos_par': 0,
        'n_pares_fortes': 0,
    }

    pair_tol = float(_pget(p, 'pair_angle_tol_deg', 5.0))
    dist_min = float(_pget(p, 'pair_dist_min', 10))
    dist_max = float(_pget(p, 'pair_dist_max', 120))
    overlap_min = float(_pget(p, 'pair_overlap_min', 0.2))
    axis_gap_max = float(_pget(p, 'oriented_pair_axis_gap_px', 45))
    accept_gap_pairs = bool(_pget(p, 'oriented_accept_gap_pairs', True))
    use_union_interval = bool(_pget(p, 'oriented_use_union_interval', True))
    margin = int(_pget(p, 'roi_margin_px', 4))
    max_rois = int(_pget(p, 'max_rois', 80))

    for i in range(len(infos)):
        a = infos[i]
        origin = a['center']
        u = a['u']
        v = a['v']
        at1, at2, as0 = _projecoes_linha_em_base(a['line'], origin, u, v)
        amin, amax = min(at1, at2), max(at1, at2)

        for j in range(i + 1, len(infos)):
            dbg['n_pares_testados'] += 1
            b = infos[j]
            adiff = _angle_diff_180(a['angle'], b['angle'])
            if adiff > pair_tol:
                continue
            dbg['n_pares_ang_ok'] += 1

            bt1, bt2, bs0 = _projecoes_linha_em_base(b['line'], origin, u, v)
            bmin, bmax = min(bt1, bt2), max(bt1, bt2)
            dist = abs(bs0 - as0)
            if dist < dist_min or dist > dist_max:
                continue
            dbg['n_pares_dist_ok'] += 1

            ov = razao_sobreposicao((amin, amax), (bmin, bmax))
            gap_axis = _interval_gap(amin, amax, bmin, bmax)
            overlap_ok = ov >= overlap_min
            gap_ok = accept_gap_pairs and (gap_axis <= axis_gap_max)
            if overlap_ok:
                dbg['n_pares_overlap_ok'] += 1
            if gap_ok and not overlap_ok:
                dbg['n_pares_gap_ok'] += 1
            if not (overlap_ok or gap_ok):
                dbg['n_pares_rejeitados_overlap_gap'] += 1
                continue

            if use_union_interval or gap_ok:
                tmin = min(amin, bmin)
                tmax = max(amax, bmax)
            else:
                tmin = max(amin, bmin)
                tmax = min(amax, bmax)
                if tmax <= tmin + 2:
                    tmin = min(amin, bmin)
                    tmax = max(amax, bmax)

            smin, smax = sorted([as0, bs0])
            perp_extra = (smax - smin) * float(_pget(p, 'oriented_perp_extra_pct', 12)) / 100.0
            smin2 = smin - perp_extra
            smax2 = smax + perp_extra

            corners = _oriented_corners(origin, u, v, tmin, tmax, smin2, smax2)
            bbox = _bbox_axis_from_points(corners, img_shape, margin=margin)
            if bbox is None:
                continue

            x1, y1, x2, y2 = bbox
            bw, bh = max(1, x2 - x1), max(1, y2 - y1)
            aspect = bh / bw
            score_pair, partes_score = _score_par_orientado(a, b, dist, dist_min, dist_max, ov, gap_axis, adiff, pair_tol, p)
            if score_pair >= 0.70:
                dbg['n_pares_fortes'] += 1

            candidatos.append({
                'line1': a['line'], 'line2': b['line'],
                'line_base': a['line'],
                'line_oposta': b['line'],
                'bbox': bbox,
                'oriented_corners': corners,
                'origin': origin,
                'u': u,
                'v': v,
                'tmin': float(tmin), 'tmax': float(tmax),
                'smin': float(smin2), 'smax': float(smax2),
                'distance': float(dist),
                'overlap': float(ov),
                'gap_axis': float(gap_axis),
                'aceito_por_gap': bool(gap_ok and not overlap_ok),
                'angle_diff': float(adiff),
                'angle': float(a['angle']),
                'area': int(bw * bh),
                'aspect': float(aspect),
                'score_pair': float(score_pair),
                'pair_len_score': partes_score['len_score'],
                'pair_axis_score': partes_score['axis_score'],
                'pair_dist_score': partes_score['dist_score'],
                'pair_angle_score': partes_score['angle_score'],
                'tipo': 'par_gap_orientado' if (gap_ok and not overlap_ok) else 'par_orientado'
            })

    candidatos = sorted(
        candidatos,
        key=lambda c: (float(c.get('score_pair', 0.0)), float(c.get('overlap', 0.0)), float(c.get('area', 0.0))),
        reverse=True
    )[:max_rois]
    dbg['n_candidatos_par'] = len(candidatos)
    return (candidatos, dbg) if return_debug else candidatos


def gerar_candidatos_reta_unica(lines, img_shape, p):
    """
    Fallback: gera candidatos a partir de uma única reta forte.

    Útil quando uma lateral do cilindro é clara, mas a lateral oposta ficou fraca ou foi descartada.
    Cada reta gera até duas caixas orientadas, uma para cada lado perpendicular.
    """
    if not bool(_pget(p, 'single_line_candidates_enabled', True)):
        return []

    infos = [_line_info(l) for l in lines]
    infos = [i for i in infos if i is not None]
    infos = sorted(infos, key=lambda x: x['length'], reverse=True)
    max_lines = int(_pget(p, 'single_line_max', 20))
    width = float(_pget(p, 'single_line_width_px', 60))
    margin = int(_pget(p, 'roi_margin_px', 4))
    max_rois = int(_pget(p, 'max_rois', 80))

    candidatos = []
    for info in infos[:max_lines]:
        origin = info['center']
        u = info['u']
        v = info['v']
        t1, t2, s0 = _projecoes_linha_em_base(info['line'], origin, u, v)
        tmin, tmax = min(t1, t2), max(t1, t2)

        # Assume que a reta pode estar em uma lateral; testa os dois lados.
        for lado, smin, smax in [('+normal', s0, s0 + width), ('-normal', s0 - width, s0)]:
            corners = _oriented_corners(origin, u, v, tmin, tmax, smin, smax)
            bbox = _bbox_axis_from_points(corners, img_shape, margin=margin)
            if bbox is None:
                continue
            x1, y1, x2, y2 = bbox
            candidatos.append({
                'line1': info['line'], 'line2': None,
                'line_base': info['line'],
                'line_oposta': None,
                'bbox': bbox,
                'oriented_corners': corners,
                'origin': origin,
                'u': u,
                'v': v,
                'tmin': float(tmin), 'tmax': float(tmax),
                'smin': float(smin), 'smax': float(smax),
                'distance': float(width),
                'overlap': 1.0,
                'gap_axis': 0.0,
                'aceito_por_gap': False,
                'angle_diff': 0.0,
                'angle': float(info['angle']),
                'area': int((x2 - x1) * (y2 - y1)),
                'aspect': float((y2 - y1) / max(1, (x2 - x1))),
                'tipo': 'reta_unica_' + lado
            })

    return candidatos[:max_rois]


def expandir_candidato_orientado_por_score(img_bgr, pre, candidato, clf, p):
    """Expande candidato ao longo da reta, testa fatores e escolhe por score combinado."""
    h, w = pre['gray'].shape[:2]
    edge_pts = _edge_points(pre.get('canny_full', pre['canny']))
    origin = np.asarray(candidato['origin'], dtype=float)
    u = np.asarray(candidato['u'], dtype=float)
    v = np.asarray(candidato['v'], dtype=float)
    tmin0, tmax0 = float(candidato['tmin']), float(candidato['tmax'])
    smin, smax = float(candidato['smin']), float(candidato['smax'])

    base_len = max(1.0, tmax0 - tmin0)
    max_extra = base_len * float(_pget(p, 'oriented_parallel_expand_pct', 60)) / 100.0
    steps = int(_pget(p, 'ref_search_steps', 5)) if bool(_pget(p, 'oriented_use_score_search', True)) else 1
    fatores = np.linspace(0.0, 1.0, max(1, steps))
    margin = int(_pget(p, 'roi_margin_px', 4))

    melhor = None
    for f in fatores:
        extra = max_extra * float(f)
        tmin = tmin0 - extra
        tmax = tmax0 + extra

        # Se há evidência de tampa/arco antes do limite, encurta até ela.
        if max_extra > 1:
            step = max(1.0, float(_pget(p, 'oriented_parallel_step_px', 6)))
            stop_score = float(_pget(p, 'oriented_cap_stop_score', 0.18))

            t_cur = tmax0
            while t_cur < tmax0 + extra:
                sc = _cap_score_orientado(edge_pts, origin, u, v, tmin0, t_cur, smin, smax, p, lado='top')
                if sc['score'] >= stop_score:
                    tmax = t_cur
                    break
                t_cur += step

            t_cur = tmin0
            while t_cur > tmin0 - extra:
                sc = _cap_score_orientado(edge_pts, origin, u, v, t_cur, tmax0, smin, smax, p, lado='bottom')
                if sc['score'] >= stop_score:
                    tmin = t_cur
                    break
                t_cur -= step

        corners = _oriented_corners(origin, u, v, tmin, tmax, smin, smax)
        bbox = _bbox_axis_from_points(corners, (h, w), margin=margin)
        if bbox is None or not _bbox_valida(bbox):
            continue
        x1, y1, x2, y2 = map(int, bbox)
        crop = img_bgr[y1:y2, x1:x2].copy()
        if crop.size == 0:
            continue

        cap_top = _cap_score_orientado(edge_pts, origin, u, v, tmin, tmax, smin, smax, p, lado='top')
        cap_bottom = _cap_score_orientado(edge_pts, origin, u, v, tmin, tmax, smin, smax, p, lado='bottom')
        cap_score = max(float(cap_top['score']), float(cap_bottom['score']))

        pred = None
        svm_score = 0.0
        if clf is not None:
            try:
                feat = extrair_hog_de_imagem(crop, p).reshape(1, -1)
                pred = int(clf.predict(feat)[0])
                try:
                    svm_score = float(clf.decision_function(feat)[0])
                except Exception:
                    svm_score = float(pred)
            except Exception:
                svm_score = -999.0

        geom_score = 0.5 * float(candidato.get('overlap', 0.0)) + 0.5 * (
            1.0 - min(1.0, float(candidato.get('angle_diff', 0.0)) / max(1e-6, float(_pget(p, 'pair_angle_tol_deg', 10.0))))
        )
        # Penaliza caixas excessivamente retangulares com tampa reta/fraca.
        bw = max(1, x2 - x1)
        bh = max(1, y2 - y1)
        aspect_axis = bh / bw
        rect_penalty = 0.0
        if cap_score < 0.08 and (aspect_axis < 1.2 or aspect_axis > 10.0):
            rect_penalty = 0.5

        score_final = (
            float(_pget(p, 'score_weight_svm', 1.0)) * svm_score +
            float(_pget(p, 'score_weight_cap', 0.35)) * cap_score +
            float(_pget(p, 'score_weight_geom', 0.10)) * geom_score -
            float(_pget(p, 'rect_penalty_weight', 0.20)) * rect_penalty
        )

        item = {
            'bbox': tuple(map(int, bbox)),
            'oriented_corners': corners,
            'score_svm': svm_score,
            'score_final': score_final,
            'pred': pred,
            'factor': float(f),
            'cap_score': cap_score,
            'cap_top': float(cap_top['score']),
            'cap_bottom': float(cap_bottom['score']),
            'cap_top_pts': cap_top.get('pts'),
            'cap_bottom_pts': cap_bottom.get('pts'),
            'geom_score': geom_score,
            'rect_penalty': rect_penalty,
            'tmin': float(tmin), 'tmax': float(tmax),
            'smin': float(smin), 'smax': float(smax)
        }
        if melhor is None or item['score_final'] > melhor['score_final']:
            melhor = item

    if melhor is None:
        bbox = candidato.get('bbox')
        corners = candidato.get('oriented_corners')
        return bbox, {
            'cap_score': 0.0, 'top_score': 0.0, 'bottom_score': 0.0,
            'rect_penalty': 0.0, 'top_contour': None, 'bottom_contour': None
        }, {
            'refine_mode': 'orientado_sem_refino',
            'tested': 0,
            'score_final': None,
            'score_svm': None,
            'pred_refine': None,
            'oriented_corners': corners
        }

    # Converte pontos de arco em polilinhas simples para visualização.
    def pts_para_contorno(pts):
        if pts is None or len(pts) < 2:
            return None
        pts = np.asarray(pts, dtype=np.int32)
        # Ordena aproximadamente pelo eixo perpendicular, para desenhar uma faixa contínua.
        rel = pts.astype(float) - origin.reshape(1, 2)
        order = np.argsort(rel @ v)
        return pts[order].reshape(-1, 1, 2)

    cap = {
        'cap_score': melhor['cap_score'],
        'top_score': melhor['cap_top'],
        'bottom_score': melhor['cap_bottom'],
        'rect_penalty': melhor['rect_penalty'],
        'top_contour': pts_para_contorno(melhor.get('cap_top_pts')),
        'bottom_contour': pts_para_contorno(melhor.get('cap_bottom_pts')),
    }
    ref_info = {
        'refine_mode': 'linhas_arcos_orientado_score',
        'tested': int(len(fatores)),
        'factor': melhor['factor'],
        'score_final': melhor['score_final'],
        'score_svm': melhor['score_svm'],
        'pred_refine': melhor['pred'],
        'geom_score': melhor['geom_score'],
        'oriented_corners': melhor['oriented_corners'],
        'tmin': melhor['tmin'], 'tmax': melhor['tmax'],
        'smin': melhor['smin'], 'smax': melhor['smax']
    }
    return melhor['bbox'], cap, ref_info


def candidatos_linhas_arcos_orientados(img_bgr, pre, p):
    """Executa Hough em fonte escolhida, une colineares e gera candidatos orientados."""
    fonte = _pget(p, 'hough_source_any', 'morph')
    edges = pre.get(fonte, pre.get('morph', pre['canny']))
    lines_raw = detectar_linhas_hough(edges, p)
    lines_merged = unir_segmentos_colineares(lines_raw, p)
    lines_filtradas = filtrar_linhas_remanescentes(lines_merged, p)

    candidatos_pares, debug_pares = gerar_pares_orientados(lines_filtradas, pre['gray'].shape, p, return_debug=True)
    candidatos_reta = gerar_candidatos_reta_unica(lines_filtradas, pre['gray'].shape, p)

    candidatos = candidatos_pares + candidatos_reta
    candidatos = sorted(
        candidatos,
        key=lambda c: (
            1 if str(c.get('tipo', '')).startswith('par') else 0,
            float(c.get('score_pair', 0.0)),
            float(c.get('overlap', 0.0)),
            float(c.get('area', 0.0))
        ),
        reverse=True
    )[:int(_pget(p, 'max_rois', 80))]

    info = {
        'lines_raw': lines_raw,
        'lines_merged': lines_merged,
        'lines_filtradas': lines_filtradas,
        'n_lines_raw': len(lines_raw),
        'n_lines_merged': len(lines_merged),
        'n_lines_filtradas': len(lines_filtradas),
        'n_candidatos_par': len(candidatos_pares),
        'n_candidatos_reta_unica': len(candidatos_reta),
        'n_candidatos_total': len(candidatos),
    }
    info.update(debug_pares)
    return candidatos, info


def detectar_candidatos_cilindro(img_bgr, clf=None, p=None, score_min=None, nms_iou=None, max_det=None, aplicar_nms=True):
    """Aplica a etapa completa. Estratégia nova: Hough qualquer direção → arcos → expansão orientada → SVM → NMS."""
    if p is None:
        p = STATE['params']

    if _pget(p, 'detector_strategy', 'linhas_arcos_orientado') == 'pares_verticais':
        return detectar_candidatos_cilindro_legado_pares(img_bgr, clf=clf, p=p, score_min=score_min, nms_iou=nms_iou, max_det=max_det, aplicar_nms=aplicar_nms)

    score_min = float(_pget(p, 'det_score_min', 0.0) if score_min is None else score_min)
    nms_iou = float(_pget(p, 'det_nms_iou', 0.30) if nms_iou is None else nms_iou)
    max_det = int(_pget(p, 'det_max_det', 8) if max_det is None else max_det)

    pre = aplicar_preprocessamento(img_bgr, p)
    candidatos, info_linhas = candidatos_linhas_arcos_orientados(img_bgr, pre, p)

    resultados = []
    positivos = []
    erros = 0

    for idx, c in enumerate(candidatos, start=1):
        try:
            bbox_ref, cap, ref_info = expandir_candidato_orientado_por_score(img_bgr, pre, c, clf, p)
            if bbox_ref is None or not _bbox_valida(bbox_ref):
                continue
            x1, y1, x2, y2 = map(int, bbox_ref)
            crop = img_bgr[y1:y2, x1:x2].copy()
            if crop.size == 0:
                continue

            score_svm = ref_info.get('score_svm')
            score_final = ref_info.get('score_final')
            pred = ref_info.get('pred_refine')

            if clf is not None and (score_svm is None or pred is None):
                feat = extrair_hog_de_imagem(crop, p).reshape(1, -1)
                pred = int(clf.predict(feat)[0])
                try:
                    score_svm = float(clf.decision_function(feat)[0])
                except Exception:
                    score_svm = float(pred)
                score_final = (
                    float(_pget(p, 'score_weight_svm', 1.0)) * score_svm +
                    float(_pget(p, 'score_weight_cap', 0.35)) * float(cap.get('cap_score', 0.0)) +
                    float(_pget(p, 'score_weight_geom', 0.10)) * float(ref_info.get('geom_score', 0.0)) -
                    float(_pget(p, 'rect_penalty_weight', 0.20)) * float(cap.get('rect_penalty', 0.0))
                )

            r = {
                'roi': idx,
                'pred': 'cilindro' if pred == 1 else ('negativo' if pred == 0 else 'sem_modelo'),
                'bbox_original': c.get('bbox'),
                'bbox': tuple(map(int, bbox_ref)),
                'oriented_corners': ref_info.get('oriented_corners', c.get('oriented_corners')),
                'score': None if score_svm is None else round(float(score_svm), 3),
                'score_float': score_svm,
                'score_final': None if score_final is None else round(float(score_final), 3),
                'score_final_float': score_final,
                'cap_top': round(float(cap.get('top_score', 0.0)), 3),
                'cap_bottom': round(float(cap.get('bottom_score', 0.0)), 3),
                'cap_score': round(float(cap.get('cap_score', 0.0)), 3),
                'rect_penalty': round(float(cap.get('rect_penalty', 0.0)), 3),
                'top_contour': cap.get('top_contour'),
                'bottom_contour': cap.get('bottom_contour'),
                'refine_mode': ref_info.get('refine_mode'),
                'refine_factor': ref_info.get('factor'),
                'dist_px': round(float(c.get('distance', 0.0)), 1),
                'score par': round(float(c.get('score_pair', 0)), 2),
            'overlap': round(float(c.get('overlap', 0.0)), 2),
                'ang_diff': round(float(c.get('angle_diff', 0.0)), 2),
                'angle': round(float(c.get('angle', 0.0)), 1),
                'tipo': c.get('tipo', 'orientado'),
            }
            resultados.append(r)

            if clf is not None and pred == 1 and (score_final is None or float(score_final) >= score_min):
                positivos.append(r)
        except Exception as e:
            erros += 1
            resultados.append({'roi': idx, 'pred': 'erro', 'bbox_original': c.get('bbox'), 'bbox': c.get('bbox'), 'erro': str(e)[:120]})

    positivos_finais = nms_bboxes_refinado(positivos, iou_thr=nms_iou, max_det=max_det) if aplicar_nms else positivos
    ids_finais = {r['roi'] for r in positivos_finais}
    for r in resultados:
        r['nms_keep'] = bool(r.get('roi') in ids_finais)
        r['pred_visual'] = 'suprimido' if r.get('pred') == 'cilindro' and not r['nms_keep'] else r.get('pred')

    info = {
        'pred_raw': int(len(positivos)),
        'pred_final': int(len(positivos_finais)),
        'erro_rois': int(erros),
        'score_min': score_min,
        'nms_iou': nms_iou,
        'detector_strategy': 'linhas_arcos_orientado',
    }
    info.update(info_linhas)

    return {
        'pre': pre,
        'candidatos': candidatos,
        'resultados': resultados,
        'positivos_finais': positivos_finais,
        'info': info
    }


# ------------------------------------------------------------
# Cores de diagnóstico em RGB para Matplotlib/OpenCV sobre imagem RGB
# ------------------------------------------------------------
COLOR_HOUGH_RAW = (150, 150, 150)        # cinza: linhas brutas
COLOR_HOUGH_FILTERED = (0, 200, 255)    # ciano: linhas filtradas
COLOR_CANDIDATE_LINE = (255, 0, 0)      # vermelho: linhas que geraram candidato
COLOR_SINGLE_LINE = (255, 0, 0)         # vermelho: candidatos por reta única
COLOR_ROI_INITIAL = (255, 140, 0)       # laranja: ROI inicial
COLOR_ROI_REFINED = (0, 255, 80)             # verde: ROI refinada / predição final
COLOR_ARC_TOP = (0, 220, 255)           # ciano: arco/tampa superior
COLOR_ARC_BOTTOM = (0, 180, 255)        # ciano-azulado: arco/tampa inferior
COLOR_GT = (255, 220, 0)                # amarelo: bounding box YOLO/GT
COLOR_NEG = COLOR_ROI_INITIAL               # laranja claro: rejeitados/negativos

# ------------------------------------------------------------
# Visualizações integradas da nova estratégia
# ------------------------------------------------------------
def _desenhar_poligono_orientado(img_rgb, pts, color=(0, 255, 255), thickness=1):
    if pts is None:
        return img_rgb
    pts = np.asarray(pts, dtype=np.int32).reshape(-1, 1, 2)
    cv2.polylines(img_rgb, [pts], True, color, thickness)
    return img_rgb


def render_rois_refinadas():
    pre = obter_preprocessamento_referencia()
    img_bgr_ref = obter_imagem_referencia()
    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada.')
        return

    p = STATE['params']
    if _pget(p, 'detector_strategy', 'linhas_arcos_orientado') == 'pares_verticais':
        return render_rois_refinadas_legado()

    det = detectar_candidatos_cilindro(img_bgr_ref, clf=None, p=p, aplicar_nms=False)
    candidatos = det['candidatos']
    info = det['info']
    idx = atualizar_opcoes_roi_hog(candidatos)
    img = pre['rgb'].copy()

    # Linhas brutas/mescladas/filtradas.
    for line in info.get('lines_raw', [])[:300]:
        x1, y1, x2, y2 = map(int, line)
        cv2.line(img, (x1, y1), (x2, y2), COLOR_HOUGH_RAW, 1)
    for line in info.get('lines_filtradas', [])[:250]:
        x1, y1, x2, y2 = map(int, line)
        cv2.line(img, (x1, y1), (x2, y2), COLOR_HOUGH_FILTERED, 1)

    # Candidatos orientados.
    for k, c in enumerate(candidatos[:int(_pget(p, 'max_rois', 80))], start=1):
        line_color = COLOR_SINGLE_LINE if str(c.get('tipo', '')).startswith('reta_unica') else COLOR_CANDIDATE_LINE
        for line_key in ['line_base', 'line_oposta']:
            line = c.get(line_key)
            if line is None:
                continue
            x1l, y1l, x2l, y2l = map(int, line)
            cv2.line(img, (x1l, y1l), (x2l, y2l), line_color, 2)
        _desenhar_poligono_orientado(img, c.get('oriented_corners'), color=COLOR_ROI_INITIAL, thickness=1)
        if _bbox_valida(c.get('bbox')):
            x1, y1, x2, y2 = map(int, c['bbox'])
            cv2.rectangle(img, (x1, y1), (x2, y2), COLOR_ROI_INITIAL, 1)
            cv2.putText(img, str(k), (x1, max(12, y1 - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.38, COLOR_ROI_INITIAL, 1, cv2.LINE_AA)
        # Mostra também os arcos/tampas usados no refino/score da ROI.
        try:
            bbox_ref, cap, ref_info = expandir_candidato_orientado_por_score(img_bgr_ref, pre, c, None, p)
            for contour, color in [(cap.get('top_contour'), COLOR_ARC_TOP), (cap.get('bottom_contour'), COLOR_ARC_BOTTOM)]:
                if contour is not None and len(contour) >= 2:
                    cv2.polylines(img, [contour.reshape(-1, 1, 2)], False, color, 2)
        except Exception:
            pass

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img)
    ax.set_title(
        f"Linhas/arcos orientados — retas vermelhas + arcos ciano | Hough raw={info.get('n_lines_raw',0)} | "
        f"unidas={info.get('n_lines_merged',0)} | filtradas={info.get('n_lines_filtradas',0)} | "
        f"ROIs={len(candidatos)} | pares={info.get('n_candidatos_par',0)} | reta única={info.get('n_candidatos_reta_unica',0)}"
    )
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    display(HTML('<small><b>Legenda:</b> cinza = Hough bruta; ciano-claro = linhas filtradas; vermelho = retas usadas na ROI; laranja = ROI inicial; verde = ROI refinada; ciano forte = arcos/tampas detectados.</small>'))
    debug_rows = [{
        'pares testados': info.get('n_pares_testados', 0),
        'ang ok': info.get('n_pares_ang_ok', 0),
        'dist ok': info.get('n_pares_dist_ok', 0),
        'overlap ok': info.get('n_pares_overlap_ok', 0),
        'aceitos por gap': info.get('n_pares_gap_ok', 0),
        'rejeitados overlap/gap': info.get('n_pares_rejeitados_overlap_gap', 0),
        'cand par': info.get('n_candidatos_par', 0),
        'cand reta única': info.get('n_candidatos_reta_unica', 0),
        'pares fortes': info.get('n_pares_fortes', 0)
    }]
    exibir_tabela_compacta(pd.DataFrame(debug_rows), max_rows=1)

    rows = []
    for k, c in enumerate(candidatos[:12], start=1):
        rows.append({
            'roi': k,
            'tipo': c.get('tipo'),
            'ângulo': round(float(c.get('angle', 0)), 1),
            'dist_px': round(float(c.get('distance', 0)), 1),
            'score par': round(float(c.get('score_pair', 0)), 2),
            'overlap': round(float(c.get('overlap', 0)), 2),
            'ang_diff': round(float(c.get('angle_diff', 0)), 2),
            'gap_eixo': round(float(c.get('gap_axis', 0)), 1),
            'por_gap': bool(c.get('aceito_por_gap', False)),
            'bbox': c.get('bbox')
        })
    if rows:
        exibir_tabela_compacta(pd.DataFrame(rows), max_rows=12)
    else:
        print('Nenhuma ROI orientada foi gerada. Ajuste Hough, união colinear ou filtros geométricos.')


def render_hog_visualizacao_refinada():
    pre = obter_preprocessamento_referencia()
    img_bgr_ref = obter_imagem_referencia()
    if pre is None or img_bgr_ref is None:
        print('Nenhuma imagem de referência selecionada.')
        return

    p = STATE['params']
    if _pget(p, 'detector_strategy', 'linhas_arcos_orientado') == 'pares_verticais':
        return render_hog_visualizacao_refinada_legado()

    det = detectar_candidatos_cilindro(img_bgr_ref, clf=None, p=p, aplicar_nms=False)
    candidatos = det['candidatos']
    idx = atualizar_opcoes_roi_hog(candidatos)
    if idx is None or len(candidatos) == 0:
        print('Nenhuma ROI candidata foi gerada para esta imagem com os parâmetros atuais.')
        return
    idx = int(np.clip(int(idx), 0, len(candidatos)-1))
    c = candidatos[idx]

    bbox_ref, cap, ref_info = expandir_candidato_orientado_por_score(img_bgr_ref, pre, c, None, p)
    if bbox_ref is None or not _bbox_valida(bbox_ref):
        print('A ROI orientada selecionada gerou uma caixa inválida.')
        return

    x1, y1, x2, y2 = map(int, bbox_ref)
    crop_bgr = img_bgr_ref[y1:y2, x1:x2].copy()
    if crop_bgr.size == 0:
        print('A ROI refinada selecionada gerou um recorte vazio.')
        return

    try:
        hog_input_resized, hog_image, features_hog = extrair_hog_visualizacao_de_imagem(crop_bgr, p)
        features_total = extrair_hog_de_imagem(crop_bgr, p)
    except Exception as e:
        print('Não foi possível gerar a visualização HOG:', e)
        traceback.print_exc(limit=1)
        return

    img_contexto = pre['rgb'].copy()
    line_color = COLOR_SINGLE_LINE if str(c.get('tipo', '')).startswith('reta_unica') else COLOR_CANDIDATE_LINE
    for line_key in ['line_base', 'line_oposta']:
        line = c.get(line_key)
        if line is not None:
            x1l, y1l, x2l, y2l = map(int, line)
            cv2.line(img_contexto, (x1l, y1l), (x2l, y2l), line_color, 2)
    if _bbox_valida(c.get('bbox')):
        ox1, oy1, ox2, oy2 = map(int, c['bbox'])
        cv2.rectangle(img_contexto, (ox1, oy1), (ox2, oy2), COLOR_ROI_INITIAL, 1)
    _desenhar_poligono_orientado(img_contexto, c.get('oriented_corners'), color=COLOR_ROI_INITIAL, thickness=1)
    _desenhar_poligono_orientado(img_contexto, ref_info.get('oriented_corners'), color=COLOR_ROI_REFINED, thickness=2)
    cv2.rectangle(img_contexto, (x1, y1), (x2, y2), COLOR_ROI_REFINED, 2)

    for contour, color in [(cap.get('top_contour'), COLOR_ARC_TOP), (cap.get('bottom_contour'), COLOR_ARC_BOTTOM)]:
        if contour is not None and len(contour) >= 2:
            cv2.polylines(img_contexto, [contour.reshape(-1, 1, 2)], False, color, 2)

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    fig, axs = plt.subplots(1, 4, figsize=(16, 4.6))
    axs[0].imshow(img_contexto); axs[0].set_title('ROI orientada/refinada\nretas vermelhas + arcos ciano')
    axs[1].imshow(crop_rgb); axs[1].set_title('Recorte usado no SVM')
    axs[2].imshow(hog_input_resized, cmap='gray'); axs[2].set_title(f"Entrada HOG\n{p['hog_input']}")
    axs[3].imshow(hog_image, cmap='gray'); axs[3].set_title(f'HOG\nbase={len(features_hog)} | total={len(features_total)}')
    for ax in axs: ax.axis('off')
    plt.tight_layout(); plt.show()

    df_info = pd.DataFrame([{
        'roi': idx + 1,
        'bbox_inicial': c.get('bbox'),
        'bbox_refinada': bbox_ref,
        'ângulo_base': round(float(c.get('angle', 0)), 1),
        'cap_top': round(float(cap.get('top_score', 0)), 3),
        'cap_bottom': round(float(cap.get('bottom_score', 0)), 3),
        'cap_score': round(float(cap.get('cap_score', 0)), 3),
        'factor': ref_info.get('factor'),
        'modo_refino': ref_info.get('refine_mode'),
        'n_hog_base': len(features_hog),
        'n_features_total': len(features_total),
    }])
    exibir_tabela_compacta(df_info, max_rows=1)


# Atualiza renderers já existentes para usar a nova versão.
if 'STATE' in globals() and 'renderers' in STATE:
    if 'rois' in STATE['renderers']:
        STATE['renderers']['rois']['fn'] = render_rois_refinadas
    if 'hog_visualizacao' in STATE['renderers']:
        STATE['renderers']['hog_visualizacao']['fn'] = render_hog_visualizacao_refinada
    atualizar_renderer('rois')
    atualizar_renderer('hog_visualizacao')

# Sincroniza parâmetros após adicionar widgets novos.
definir_params_a_partir_widgets(PARAM_WIDGETS)


## Treinamento HOG + SVM com bloqueio por botão

Esta célula **não treina automaticamente** quando você executa todas as células do notebook.

O treinamento só ocorre ao clicar no botão **Confirmar setup e treinar se necessário**.

Ao clicar, o notebook calcula uma assinatura considerando:

- imagens selecionadas nos JSONs de treino/teste;
- rótulos inferidos;
- tamanho e modificação dos arquivos;
- todos os parâmetros ajustados acima.

Se a assinatura for igual à do último modelo salvo, o treinamento é pulado e o histórico é apenas exibido.

In [ ]:
# ============================================================
# 12. Treinamento protegido por botão + histórico cumulativo
# ============================================================

btn_treinar = widgets.Button(
    description='Confirmar setup e treinar se necessário',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='330px')
)

btn_mostrar_historico = widgets.Button(
    description='Mostrar histórico',
    button_style='',
    icon='table',
    layout=widgets.Layout(width='180px')
)

out_train_status = widgets.Output()
out_history = widgets.Output()

display(widgets.HBox([btn_treinar, btn_mostrar_historico]))
display(out_train_status)
display(out_history)


def mostrar_historico():
    with out_history:
        clear_output(wait=True)

        hist = carregar_historico()
        if hist.empty:
            print('Ainda não há treinamentos registrados.')
            return

        cols_pref = [
            'data_hora', 'setup_id', 'setup_nome', 'refine_setup_id', 'refine_setup_nome', 'assinatura_curta', 'modo_eval',
            'n_img_train', 'n_img_test',
            'n_train', 'n_test', 'pos_train', 'neg_train', 'pos_test', 'neg_test',
            'accuracy', 'precision', 'recall', 'f1',
            'det_eval_imgs', 'n_gt_det', 'n_pred_det', 'det_iou_thr',
            'mean_iou_best', 'median_iou_best', 'mean_iou_tp',
            'precision_det', 'recall_det', 'f1_det', 'tp_det', 'fp_det', 'fn_det',
            'precision_det_30', 'recall_det_30', 'f1_det_30',
            'precision_det_50', 'recall_det_50', 'f1_det_50',
            'tempo_s',
            'hog_input', 'hog_size',
            'bilateral_d', 'sigma_color', 'sigma_space',
            'canny_sigma', 'edge_filter', 'edge_min_area', 'edge_min_extent',
            'close_kernel', 'open_kernel', 'solid_filter', 'solid_core_min_area', 'solid_erode_kernel', 'solid_dilate_kernel',
            'hough_min_len', 'pair_dist', 'pair_angle_tol',
            'refine_mode', 'expand_xy', 'cap_weight', 'cap_gap', 'cap_feat_svm',
            'neg_por_pos', 'pos_margin', 'svm_C'
        ]

        cols = [c for c in cols_pref if c in hist.columns]
        exibir_tabela_compacta(hist[cols], max_rows=15)

        print(f'\nHistórico completo salvo em:\n{HISTORY_PATH}')


def treinar_se_necessario(_=None):
    with out_train_status:
        clear_output(wait=True)

        df = STATE.get('df_dataset', pd.DataFrame())
        p = dict(STATE['params'])

        if df is None or len(df) == 0:
            print('Dataset vazio. Recarregue a pasta de imagens antes de treinar.')
            return

        n_obj_train = int(df.loc[df['split'] == 'train', 'n_obj'].sum())
        n_obj_test = int(df.loc[df['split'] == 'test', 'n_obj'].sum())

        if n_obj_train < 2:
            print('Dataset de treino insuficiente: menos de 2 objetos anotados em train/labels.')
            print(resumo_dataset(df))
            return

        assinatura = assinatura_treinamento(df, p)
        assinatura_curta = assinatura[:12]

        meta = carregar_json(METADATA_PATH, default={})

        setup_id_hist = obter_setup_id_para_historico()
        setup_nome_hist = obter_setup_nome_para_historico()

        print('Setup atual:', setup_id_hist, '-', setup_nome_hist)
        print('Assinatura atual:', assinatura_curta)
        print('Modelo salvo:', MODEL_PATH.exists())
        print('Objetos anotados em train:', n_obj_train)
        print('Objetos anotados em test :', n_obj_test)

        if MODEL_PATH.exists() and meta.get('assinatura') == assinatura:
            print('\nNenhuma alteração detectada nos parâmetros/dataset.')
            print('Treinamento pulado. O modelo salvo continua válido para este setup.')
            mostrar_historico()
            if 'renderizar_resultados_setup_selecionado' in globals():
                renderizar_resultados_setup_selecionado()
            return

        print('\nAlteração detectada ou modelo inexistente.')
        print('Gerando amostras YOLO e treinando HOG + SVM...')

        t0 = time.time()

        def status_callback(txt):
            print(txt)

        try:
            X_train, X_test, y_train, y_test, stats, erros, info = preparar_dados_treino_teste(
                df, p, status_callback=status_callback
            )

            clf = make_pipeline(
                StandardScaler(),
                LinearSVC(
                    C=float(p['svm_C']),
                    class_weight='balanced',
                    max_iter=10000
                )
            )

            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, zero_division=0)
            rec = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

            tempo_s = time.time() - t0

            joblib.dump(clf, MODEL_PATH)

            # Avaliação do detector completo em imagem inteira.
            # Esta etapa mede localização de caixas via IoU, separada das métricas
            # de classificação HOG/SVM dos recortes.
            det_metrics = {}
            try:
                print('\nAvaliando detector completo por IoU...')
                det_metrics = avaliar_detector_completo(df, clf, p, status_callback=status_callback)
                if det_metrics:
                    print(
                        'Detector completo: '
                        f"imgs={det_metrics.get('det_eval_imgs', 0)} | "
                        f"GT={det_metrics.get('n_gt_det', 0)} | "
                        f"Pred={det_metrics.get('n_pred_det', 0)} | "
                        f"IoU médio(best)={det_metrics.get('mean_iou_best', 0):.3f} | "
                        f"F1@{det_metrics.get('det_iou_thr', 0.5):.2f}={det_metrics.get('f1_det', 0):.3f}"
                    )
            except Exception as e_det:
                print('Aviso: falha na avaliação do detector completo por IoU:', e_det)
                det_metrics = {}

            metadata = {
                'assinatura': assinatura,
                'assinatura_curta': assinatura_curta,
                'data_hora': time.strftime('%Y-%m-%d %H:%M:%S'),
                'model_path': str(MODEL_PATH),
                'history_path': str(HISTORY_PATH),
                'params': p,
                'setup_id': setup_id_hist,
                'setup_nome': setup_nome_hist,
                'setup_modificado': bool(STATE.get('setup_modificado', False)),
                'refine_setup_id': STATE.get('refine_setup_id', 'REFINO_ATUAL'),
                'refine_setup_nome': STATE.get('refine_setup_nome', 'parâmetros atuais/manuais'),
                'zoom_out_pct': int(obter_zoom_out_pct()),
                'zoomout_train_count': len(carregar_chaves_zoomout('train')),
                'zoomout_test_count': len(carregar_chaves_zoomout('test')),
                'dataset_path': str(STATE['data_path']),
                'stats': stats,
                'detector_eval': det_metrics
            }
            salvar_json(METADATA_PATH, metadata)

            row = {
                'data_hora': metadata['data_hora'],
                'setup_id': setup_id_hist,
                'setup_nome': setup_nome_hist,
                'refine_setup_id': STATE.get('refine_setup_id', 'REFINO_ATUAL'),
                'refine_setup_nome': STATE.get('refine_setup_nome', 'parâmetros atuais/manuais'),
                'zoom_out_pct': int(obter_zoom_out_pct()),
                'zoomout_train_count': len(carregar_chaves_zoomout('train')),
                'zoomout_test_count': len(carregar_chaves_zoomout('test')),
                'assinatura': assinatura,
                'assinatura_curta': assinatura_curta,
                'modo_eval': stats['modo_eval'],
                'n_img_train': stats['n_img_train'],
                'n_img_test': stats['n_img_test'],
                'n_total': stats['n_total'],
                'n_train': stats['n_train'],
                'n_test': stats['n_test'],
                'pos_train': stats['n_pos_train'],
                'neg_train': stats['n_neg_train'],
                'pos_test': stats['n_pos_test'],
                'neg_test': stats['n_neg_test'],
                'accuracy': abreviar_float(acc),
                'precision': abreviar_float(prec),
                'recall': abreviar_float(rec),
                'f1': abreviar_float(f1),
                'det_eval_imgs': int(det_metrics.get('det_eval_imgs', 0)),
                'n_gt_det': int(det_metrics.get('n_gt_det', 0)),
                'n_pred_det': int(det_metrics.get('n_pred_det', 0)),
                'det_iou_thr': abreviar_float(det_metrics.get('det_iou_thr', p.get('det_iou_thr', 0.5)), 2),
                'mean_iou_best': abreviar_float(det_metrics.get('mean_iou_best', 0.0), 4),
                'median_iou_best': abreviar_float(det_metrics.get('median_iou_best', 0.0), 4),
                'mean_iou_tp': abreviar_float(det_metrics.get('mean_iou_tp', 0.0), 4),
                'precision_det': abreviar_float(det_metrics.get('precision_det', 0.0), 4),
                'recall_det': abreviar_float(det_metrics.get('recall_det', 0.0), 4),
                'f1_det': abreviar_float(det_metrics.get('f1_det', 0.0), 4),
                'tp_det': int(det_metrics.get('tp_det', 0)),
                'fp_det': int(det_metrics.get('fp_det', 0)),
                'fn_det': int(det_metrics.get('fn_det', 0)),
                'precision_det_30': abreviar_float(det_metrics.get('precision_det_30', 0.0), 4),
                'recall_det_30': abreviar_float(det_metrics.get('recall_det_30', 0.0), 4),
                'f1_det_30': abreviar_float(det_metrics.get('f1_det_30', 0.0), 4),
                'precision_det_50': abreviar_float(det_metrics.get('precision_det_50', 0.0), 4),
                'recall_det_50': abreviar_float(det_metrics.get('recall_det_50', 0.0), 4),
                'f1_det_50': abreviar_float(det_metrics.get('f1_det_50', 0.0), 4),
                'tempo_s': abreviar_float(tempo_s, 3),
                'cm_tn': int(cm[0, 0]),
                'cm_fp': int(cm[0, 1]),
                'cm_fn': int(cm[1, 0]),
                'cm_tp': int(cm[1, 1]),
                'hog_input': p['hog_input'],
                'hog_size': f"{p['hog_resize_w']}x{p['hog_resize_h']}",
                'hog_orientations': int(p['hog_orientations']),
                'hog_cell': int(p['hog_pixels_per_cell']),
                'hog_block': int(p['hog_cells_per_block']),
                'bilateral_d': int(p['bilateral_d']),
                'sigma_color': int(p['bilateral_sigma_color']),
                'sigma_space': int(p['bilateral_sigma_space']),
                'canny_sigma': abreviar_float(p['canny_sigma'], 2),
                'edge_filter': bool(p.get('edge_filter_ativo', True)),
                'edge_min_area': int(p.get('edge_min_area', 0)),
                'edge_min_extent': int(p.get('edge_min_extent', 0)),
                'close_kernel': f"{p['close_kernel_w']}x{p['close_kernel_h']}:{p['close_iter']}",
                'open_kernel': f"{p['open_kernel_w']}x{p['open_kernel_h']}:{p['open_iter']}",
                'solid_filter': bool(p.get('solid_filter_ativo', True)),
                'solid_core_min_area': int(p.get('solid_min_area', 0)),
                'solid_erode_kernel': f"{p.get('solid_erode_kernel_w', 0)}x{p.get('solid_erode_kernel_h', 0)}:{p.get('solid_erode_iter', 0)}",
                'solid_dilate_kernel': f"{p.get('solid_dilate_kernel_w', 0)}x{p.get('solid_dilate_kernel_h', 0)}:{p.get('solid_dilate_iter', 0)}",
                'hough_min_len': int(p['hough_min_line_length']),
                'hough_gap': int(p['hough_max_line_gap']),
                'pair_dist': f"{p['pair_dist_min']}-{p['pair_dist_max']}",
                'pair_angle_tol': abreviar_float(p['pair_angle_tol_deg'], 1),
                'detector_strategy': p.get('detector_strategy', 'pares_verticais'),
                'hough_source_any': p.get('hough_source_any', 'morph'),
                'merge_collinear': bool(p.get('merge_collinear_enabled', False)),
                'merge_gap': int(p.get('merge_gap_px', 0)),
                'merge_angle_tol': abreviar_float(p.get('merge_angle_tol_deg', 0), 1),
                'oriented_parallel_exp': int(p.get('oriented_parallel_expand_pct', 0)),
                'oriented_perp_extra': int(p.get('oriented_perp_extra_pct', 0)),
                'refine_mode': p.get('bbox_refine_mode', 'parametrica') if p.get('bbox_refine_enabled', True) else 'off',
                'expand_xy': f"{abreviar_float(p.get('ref_expand_x_pct', 0),1)}-{abreviar_float(p.get('ref_expand_y_pct', 0),1)}",
                'cap_weight': abreviar_float(p.get('score_weight_cap', 0.0), 2),
                'cap_gap': int(p.get('cap_gap_px', 0)),
                'cap_feat_svm': bool(p.get('svm_usar_features_arcos', False)),
                'neg_por_pos': int(p['negativos_por_positivo']),
                'neg_iou_max': abreviar_float(p['neg_iou_max'], 2),
                'pos_margin': abreviar_float(p['pos_margin_pct'], 2),
                'svm_C': abreviar_float(p['svm_C'], 4),
                'dataset_path': str(STATE['data_path'])
            }

            salvar_linha_historico(row)

            print('\nTreinamento concluído.')
            print(f"Modo de avaliação: {stats['modo_eval']}")
            print(f'Tempo: {tempo_s:.3f} s')
            print(f'Acurácia: {acc:.4f} | Precisão: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}')
            if det_metrics:
                print(
                    f"Detector completo: IoU médio(best)={det_metrics.get('mean_iou_best', 0):.4f} | "
                    f"Precision_det={det_metrics.get('precision_det', 0):.4f} | "
                    f"Recall_det={det_metrics.get('recall_det', 0):.4f} | "
                    f"F1_det={det_metrics.get('f1_det', 0):.4f}"
                )
            print('\nAmostras:')
            print(f"Treino: {stats['n_train']}  | Pos: {stats['n_pos_train']}  | Neg: {stats['n_neg_train']}")
            print(f"Teste : {stats['n_test']}  | Pos: {stats['n_pos_test']}  | Neg: {stats['n_neg_test']}")
            print('\nMatriz de confusão [linhas: real 0/1; colunas: predito 0/1]:')
            print(cm)

            if erros:
                print(f'\nAtenção: {len(erros)} imagens falharam durante a extração de features.')
                for epath, emsg in erros[:5]:
                    print('-', Path(epath).name, '=>', emsg[:120])

            mostrar_historico()
            if 'renderizar_resultados_setup_selecionado' in globals():
                renderizar_resultados_setup_selecionado()

        except Exception as e:
            print('Erro durante o treinamento:')
            print(e)
            traceback.print_exc(limit=2)


btn_treinar.on_click(treinar_se_necessario)
btn_mostrar_historico.on_click(lambda _: mostrar_historico())

# Mostra histórico inicial, sem treinar.
mostrar_historico()


## Validação visual na imagem de referência e em 15 imagens aleatórias

Esta seção aplica o modelo salvo às ROIs candidatas geradas por Hough. As métricas da tabela de treinamento avaliam o HOG/SVM sobre recortes; a validação visual abaixo mostra a etapa completa de detecção, incluindo score mínimo e supressão de caixas sobrepostas por NMS.


In [ ]:
# ============================================================
# 13. Aplicação do modelo salvo na imagem de referência + grade aleatória de validação
# ============================================================

out_model_ref = widgets.Output()
out_grade_validacao = widgets.Output()

# Cores de validação visual em RGB. Se a célula do detector orientado já foi executada,
# estas variáveis já existem; os fallbacks abaixo evitam erro ao reexecutar apenas esta célula.
COLOR_GT = globals().get('COLOR_GT', (255, 220, 0))
COLOR_ROI_INITIAL = globals().get('COLOR_ROI_INITIAL', (255, 140, 0))
COLOR_ROI_REFINED = globals().get('COLOR_ROI_REFINED', (0, 255, 80))
COLOR_ARC_TOP = globals().get('COLOR_ARC_TOP', (0, 220, 255))
COLOR_ARC_BOTTOM = globals().get('COLOR_ARC_BOTTOM', (0, 180, 255))
COLOR_NEG = globals().get('COLOR_NEG', (255, 120, 0))
COLOR_LINE_CANDIDATE = globals().get('COLOR_LINE_CANDIDATE', (255, 0, 0))


btn_novas_15_validacao = widgets.Button(
    description='Selecionar outros 15 resultados aleatórios',
    button_style='info',
    icon='random',
    layout=widgets.Layout(width='310px')
)

w_validacao_split = widgets.Dropdown(
    options=[('Teste selecionado', 'test'), ('Treino selecionado', 'train'), ('Treino + teste selecionados', 'ambos')],
    value='test',
    description='amostra:',
    layout=widgets.Layout(width='310px'),
    style={'description_width': '75px'}
)

# Controles exclusivos da validação visual. Eles não alteram o treinamento;
# servem para inspecionar melhor as predições geradas pelas ROIs Hough+SVM.
w_vis_score_min = widgets.FloatSlider(
    value=0.0, min=-2.0, max=5.0, step=0.10,
    description='score mín.:',
    readout_format='.2f',
    layout=widgets.Layout(width='260px'),
    style={'description_width': '75px'}
)

w_vis_nms_iou = widgets.FloatSlider(
    value=0.30, min=0.05, max=0.90, step=0.05,
    description='NMS IoU:',
    readout_format='.2f',
    layout=widgets.Layout(width='250px'),
    style={'description_width': '70px'}
)

w_vis_max_det = widgets.IntSlider(
    value=8, min=1, max=30, step=1,
    description='máx. det.:',
    layout=widgets.Layout(width='230px'),
    style={'description_width': '70px'}
)

painel_validacao = widgets.VBox([
    widgets.HTML('<b>Validação visual</b>'),
    widgets.HTML('<small>A grade avalia a etapa completa: Hough em qualquer direção → linhas/arcos orientados → refino → HOG/SVM → NMS. Amarelo = YOLO/GT; vermelho = retas usadas na ROI; laranja = ROI inicial; verde = predição refinada; ciano = arcos/tampas.</small>'),
    out_model_ref,
    widgets.HBox([w_validacao_split, btn_novas_15_validacao], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    widgets.HBox([w_vis_score_min, w_vis_nms_iou, w_vis_max_det], layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    out_grade_validacao
])

display(painel_validacao)


def _modelo_valido_para_assinatura_atual(verbose=True):
    if not MODEL_PATH.exists():
        if verbose:
            print('Ainda não há modelo treinado salvo.')
        return None, False

    meta = carregar_json(METADATA_PATH, default={})
    assinatura_atual = assinatura_treinamento(STATE['df_dataset'], dict(STATE['params']))
    modelo_desatualizado = meta.get('assinatura') != assinatura_atual

    if modelo_desatualizado:
        if verbose:
            print('Aviso: o modelo salvo foi treinado com outra assinatura de parâmetros/dataset/zoom out.')
            print("Clique em 'Confirmar setup e treinar se necessário' para atualizar o modelo antes da validação visual.")
        return None, False

    try:
        clf = joblib.load(MODEL_PATH)
        return clf, True
    except Exception as e:
        if verbose:
            print('Não foi possível carregar o modelo salvo:', e)
        return None, False


def _bbox_valida(bbox):
    try:
        vals = [float(v) for v in bbox]
        return all(np.isfinite(v) for v in vals) and vals[2] > vals[0] and vals[3] > vals[1]
    except Exception:
        return False


def _desenhar_gt_yolo(img_rgb, boxes, label='GT'):
    out = img_rgb.copy()
    for idx, bbox in enumerate(boxes, start=1):
        if not _bbox_valida(bbox):
            continue
        x1, y1, x2, y2 = map(int, bbox)
        cv2.rectangle(out, (x1, y1), (x2, y2), COLOR_GT, 1)
        cv2.putText(out, f'{label}{idx}', (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.35, COLOR_GT, 1, cv2.LINE_AA)
    return out


def nms_bboxes(resultados_pos, iou_thr=0.30, max_det=8):
    """Aplica Non-Maximum Suppression nas predições positivas do SVM."""
    if not resultados_pos:
        return []

    dados = []
    for r in resultados_pos:
        bbox = r.get('bbox')
        if not _bbox_valida(bbox):
            continue
        score = r.get('score_float')
        if score is None or not np.isfinite(score):
            score = 0.0
        dados.append((float(score), r))

    if not dados:
        return []

    dados = sorted(dados, key=lambda t: t[0], reverse=True)
    keep = []
    for score, r in dados:
        if len(keep) >= int(max_det):
            break
        if all(iou_bbox(r['bbox'], k['bbox']) <= float(iou_thr) for k in keep):
            keep.append(r)
    return keep


def _melhor_iou_com_gt(bbox, boxes_gt):
    if not boxes_gt or not _bbox_valida(bbox):
        return 0.0
    vals = [iou_bbox(bbox, gt) for gt in boxes_gt if _bbox_valida(gt)]
    return float(max(vals)) if vals else 0.0


def aplicar_modelo_em_row(row, clf=None, desenhar_negativos=False, desenhar_gt=True, aplicar_nms=True,
                          score_min=None, nms_iou=None, max_det=None):
    """Aplica a pipeline completa com refino da ROI, arcos de tampa e NMS."""
    img_bgr, boxes_gt, zoom_info = carregar_imagem_e_bboxes_row(row)
    p = STATE['params']
    score_min = float(w_vis_score_min.value if score_min is None else score_min)
    nms_iou = float(w_vis_nms_iou.value if nms_iou is None else nms_iou)
    max_det = int(w_vis_max_det.value if max_det is None else max_det)

    det = detectar_candidatos_cilindro(
        img_bgr, clf=clf, p=p,
        score_min=score_min, nms_iou=nms_iou, max_det=max_det, aplicar_nms=aplicar_nms
    )
    pre = det['pre']
    candidatos = det['candidatos']
    resultados = det['resultados']
    positivos_finais = det['positivos_finais']
    info = det['info']

    img_out = pre['rgb'].copy()
    if desenhar_gt and boxes_gt:
        img_out = _desenhar_gt_yolo(img_out, boxes_gt, label='GT')

    if clf is None:
        for idx, c in enumerate(candidatos, start=1):
            if not _bbox_valida(c.get('bbox')):
                continue
            x1, y1, x2, y2 = map(int, c['bbox'])
            for line in [c.get('line1'), c.get('line2')]:
                if line is None:
                    continue
                lx1, ly1, lx2, ly2 = map(int, line)
                cv2.line(img_out, (lx1, ly1), (lx2, ly2), COLOR_LINE_CANDIDATE, 2)
            cv2.rectangle(img_out, (x1, y1), (x2, y2), COLOR_ROI_INITIAL, 1)
            try:
                if 'origin' in c and 'expandir_candidato_orientado_por_score' in globals():
                    bbox_ref, cap, _ = expandir_candidato_orientado_por_score(img_bgr, pre, c, None, p)
                else:
                    bbox_ref, cap, _ = refinar_bbox_candidato(img_bgr, pre, c, None, p)
                rx1, ry1, rx2, ry2 = map(int, bbox_ref)
                cv2.rectangle(img_out, (rx1, ry1), (rx2, ry2), COLOR_NEG, 1)
                for contour, color in [(cap.get('top_contour'), COLOR_ARC_TOP), (cap.get('bottom_contour'), COLOR_ARC_BOTTOM)]:
                    if contour is not None and len(contour) >= 2:
                        cv2.polylines(img_out, [contour.reshape(-1, 1, 2)], False, color, 1)
            except Exception:
                pass
            cv2.putText(img_out, f'ROI {idx}', (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.35, COLOR_ROI_INITIAL, 1, cv2.LINE_AA)
        info.update({'pred_raw': 0, 'pred_final': 0, 'erro_rois': det['info'].get('erro_rois', 0)})
        return img_out, resultados, candidatos, boxes_gt, zoom_info, info

    # Desenha negativos quando pedido, para diagnóstico na referência.
    if desenhar_negativos:
        for r in resultados:
            if r.get('pred') != 'negativo':
                continue
            bx = r.get('bbox')
            if not _bbox_valida(bx):
                continue
            x1, y1, x2, y2 = map(int, bx)
            for line in [r.get('line1'), r.get('line2')]:
                if line is None:
                    continue
                lx1, ly1, lx2, ly2 = map(int, line)
                cv2.line(img_out, (lx1, ly1), (lx2, ly2), COLOR_LINE_CANDIDATE, 2)
            cv2.rectangle(img_out, (x1, y1), (x2, y2), COLOR_ROI_INITIAL, 1)
            cv2.putText(img_out, f"neg {r.get('roi')} {r.get('score_final', '')}", (x1, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.34, COLOR_ROI_INITIAL, 1, cv2.LINE_AA)

    for r in positivos_finais:
        x1, y1, x2, y2 = map(int, r['bbox'])
        for line in [r.get('line1'), r.get('line2')]:
            if line is None:
                continue
            lx1, ly1, lx2, ly2 = map(int, line)
            cv2.line(img_out, (lx1, ly1), (lx2, ly2), COLOR_LINE_CANDIDATE, 2)
        # Caixa original da Hough em laranja claro, refinada em verde.
        if _bbox_valida(r.get('bbox_original')):
            ox1, oy1, ox2, oy2 = map(int, r['bbox_original'])
            cv2.rectangle(img_out, (ox1, oy1), (ox2, oy2), COLOR_ROI_INITIAL, 1)
        cv2.rectangle(img_out, (x1, y1), (x2, y2), COLOR_ROI_REFINED, 2)
        for contour, color in [(r.get('top_contour'), COLOR_ARC_TOP), (r.get('bottom_contour'), COLOR_ARC_BOTTOM)]:
            if contour is not None and len(contour) >= 2:
                cv2.polylines(img_out, [contour.reshape(-1, 1, 2)], False, color, 2)
        texto = f'CIL {r["roi"]}'
        if r.get('score_final') is not None:
            texto += f' sf={r["score_final"]:.2f}'
        if r.get('cap_score') is not None:
            texto += f' cap={r["cap_score"]:.2f}'
        cv2.putText(img_out, texto, (x1, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.38, COLOR_ROI_REFINED, 1, cv2.LINE_AA)

    # Acrescenta IoU das predições finais para exibição.
    pred_boxes = [r['bbox'] for r in positivos_finais if _bbox_valida(r.get('bbox'))]
    info['mean_best_iou_pred'] = round(float(np.mean([_melhor_iou_com_gt(b, boxes_gt) for b in pred_boxes])) if pred_boxes else 0.0, 3)
    info['det_match_iou50'] = associar_predicoes_gt(pred_boxes, boxes_gt, iou_thr=0.50)

    for r in resultados:
        if _bbox_valida(r.get('bbox')):
            r['best_iou_gt'] = round(_melhor_iou_com_gt(r['bbox'], boxes_gt), 3)
        # Evita jogar arrays de contorno dentro da tabela.
        r.pop('top_contour', None)
        r.pop('bottom_contour', None)
        r.pop('oriented_corners', None)
        r.pop('score_float', None)
        r.pop('score_final_float', None)

    return img_out, resultados, candidatos, boxes_gt, zoom_info, info

def render_modelo_referencia():
    pre = obter_preprocessamento_referencia()

    if pre is None:
        print('Nenhuma imagem de referência selecionada.')
        return

    clf, modelo_ok = _modelo_valido_para_assinatura_atual(verbose=True)
    df = STATE.get('df_dataset', pd.DataFrame())
    ref_path = STATE.get('ref_path')

    if df is None or len(df) == 0 or ref_path is None:
        print('Dataset ou imagem de referência indisponível.')
        return

    row_df = df[df['path'].astype(str) == str(ref_path)]
    if len(row_df) == 0:
        print('Imagem de referência não encontrada no dataframe do dataset.')
        return

    row = row_df.iloc[0]
    img_out, resultados, candidatos, boxes_gt, zoom_info, info = aplicar_modelo_em_row(
        row, clf if modelo_ok else None, desenhar_negativos=True, desenhar_gt=True
    )

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(img_out)
    if modelo_ok:
        ax.set_title(f"Modelo na referência — ROIs={len(candidatos)} | pred bruto={info['pred_raw']} | pós-NMS={info['pred_final']}")
    else:
        ax.set_title(f'Modelo indisponível/desatualizado — ROIs geométricas — {len(candidatos)} candidatas')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    if zoom_info.get('zoom_out_aplicado'):
        print(f"Zoom out aplicado nesta imagem: {zoom_info.get('zoom_out_pct', 0)}%")

    if resultados:
        cols = ['roi', 'pred', 'pred_visual', 'nms_keep', 'score', 'best_iou_gt', 'bbox', 'dist_px', 'overlap', 'erro']
        df_res = pd.DataFrame(resultados)
        cols = [c for c in cols if c in df_res.columns]
        exibir_tabela_compacta(df_res[cols], max_rows=20)
    else:
        print('Nenhuma ROI candidata foi gerada para esta imagem.')


def _df_validacao_visual():
    df = STATE.get('df_dataset', pd.DataFrame())
    if df is None or len(df) == 0:
        return pd.DataFrame()

    modo = w_validacao_split.value
    partes = []
    if modo in ['train', 'ambos']:
        partes.append(df_por_selecao_persistente(df, 'train'))
    if modo in ['test', 'ambos']:
        partes.append(df_por_selecao_persistente(df, 'test'))

    partes = [p for p in partes if p is not None and len(p) > 0]
    if not partes:
        return pd.DataFrame(columns=df.columns)

    return pd.concat(partes, ignore_index=True).reset_index(drop=True)


def render_grade_validacao_aleatoria():
    with out_grade_validacao:
        clear_output(wait=True)

        df_val = _df_validacao_visual()
        if df_val is None or len(df_val) == 0:
            print('Nenhuma imagem selecionada disponível para a grade de validação.')
            return

        clf, modelo_ok = _modelo_valido_para_assinatura_atual(verbose=False)
        if not modelo_ok:
            print('Modelo ausente ou desatualizado. A grade mostrará somente ROIs geométricas/caixas YOLO.')
            clf = None

        n = min(15, len(df_val))
        rng = np.random.default_rng(int(STATE.get('validacao_random_seed', 42)))
        idxs = rng.choice(len(df_val), size=n, replace=False)
        rows = [df_val.iloc[int(i)] for i in idxs]

        # Grade mais espaçada: com 5 colunas os títulos comprimiam os eixos e as imagens ficavam minúsculas.
        ncols = 3
        nrows = int(math.ceil(n / ncols))
        fig, axs = plt.subplots(nrows, ncols, figsize=(16, 4.8 * nrows), constrained_layout=False)
        axs = np.array(axs).reshape(-1)

        resumo = []
        erros = []
        for ax, row in zip(axs, rows):
            try:
                img_out, resultados, candidatos, boxes_gt, zoom_info, info = aplicar_modelo_em_row(
                    row, clf, desenhar_negativos=False, desenhar_gt=True,
                    score_min=w_vis_score_min.value, nms_iou=w_vis_nms_iou.value, max_det=w_vis_max_det.value
                )
                ax.imshow(img_out, interpolation='nearest')
                ztxt = f" | z{zoom_info.get('zoom_out_pct')}%" if zoom_info.get('zoom_out_aplicado') else ''
                titulo_nome = str(row['arquivo'])[:14] + ('...' if len(str(row['arquivo'])) > 14 else '')
                ax.set_title(
                    f"{row['split']} | {titulo_nome}\nGT={len(boxes_gt)} | pred={info['pred_final']}/{info['pred_raw']} | ROI={len(candidatos)} | IoU={info.get('mean_best_iou_pred',0):.2f}{ztxt}",
                    fontsize=8, pad=3
                )
                resumo.append({
                    'arquivo': row['arquivo'],
                    'split': row['split'],
                    'gt': len(boxes_gt),
                    'rois': len(candidatos),
                    'pred_bruto': info['pred_raw'],
                    'pred_pos_nms': info['pred_final'],
                    'iou_medio_pred_gt': info['mean_best_iou_pred'],
                    'erros_roi': info['erro_rois'],
                    'zoom_out': zoom_info.get('zoom_out_aplicado'),
                    'zoom_pct': zoom_info.get('zoom_out_pct') if zoom_info.get('zoom_out_aplicado') else 0
                })
            except Exception as e:
                msg = str(e)[:80]
                ax.text(0.5, 0.5, f"Erro\n{row.get('arquivo','')}\n{msg}", ha='center', va='center', fontsize=8)
                erros.append({'arquivo': row.get('arquivo', ''), 'split': row.get('split', ''), 'erro': msg})

            ax.axis('off')

        for ax in axs[len(rows):]:
            ax.axis('off')

        fig.subplots_adjust(left=0.01, right=0.99, top=0.96, bottom=0.02, wspace=0.04, hspace=0.18)
        plt.show()

        if resumo:
            display(HTML('<small><b>Legenda:</b> amarelo = YOLO/GT; verde = predição positiva pós-NMS; laranja = ROI inicial; magenta/rosa = arcos/tampas. No título, pred=a/b significa a caixas após NMS de b caixas positivas brutas.</small>'))
            exibir_tabela_compacta(pd.DataFrame(resumo), max_rows=15)
        if erros:
            display(HTML('<b>Imagens com erro de carregamento/anotação ignoradas parcialmente:</b>'))
            exibir_tabela_compacta(pd.DataFrame(erros), max_rows=15)


def sortear_novas_15_validacao(_=None):
    STATE['validacao_random_seed'] = int(STATE.get('validacao_random_seed', 42)) + 1
    render_grade_validacao_aleatoria()


def on_validacao_split_change(change):
    if change.get('name') == 'value':
        render_grade_validacao_aleatoria()


def on_validacao_param_change(change):
    if change.get('name') == 'value':
        render_grade_validacao_aleatoria()


btn_novas_15_validacao.on_click(sortear_novas_15_validacao)
w_validacao_split.observe(on_validacao_split_change, names='value')
w_vis_score_min.observe(on_validacao_param_change, names='value')
w_vis_nms_iou.observe(on_validacao_param_change, names='value')
w_vis_max_det.observe(on_validacao_param_change, names='value')

registrar_renderer('modelo_referencia', out_model_ref, render_modelo_referencia)
render_grade_validacao_aleatoria()


## Observações finais para uso no experimento

- Ajuste primeiro a seleção manual de treino/teste. As imagens marcadas com **zoom out** geram cópias salvas e ajustam as labels YOLO, sem sobrescrever o dataset original.
- Ajuste o pré-processamento olhando a imagem de referência.
- Depois confira a Hough em qualquer direção, a união de segmentos colineares e as ROIs orientadas.
- A nova estratégia usa retas como guias, procura arcos/tampas nas regiões próximas e escolhe a expansão por score combinado.
- A expansão guiada pelo SVM só tem efeito completo depois que um modelo já foi treinado/carregado.
- O histórico cumulativo será atualizado em `CLASSICAL_DIR / 'historico_treinos.csv'`.
- O modelo será salvo em `CLASSICAL_DIR / 'modelo_hog_svm_cilindros.joblib'`.
- O arquivo `metadata_modelo.json` guarda a assinatura do último treinamento, permitindo pular treinamentos repetidos.


## Nota sobre as novas métricas

As métricas `accuracy`, `precision`, `recall` e `f1` continuam avaliando o **classificador HOG/SVM nos recortes**. Já as colunas `mean_iou_best`, `precision_det`, `recall_det`, `f1_det`, `tp_det`, `fp_det` e `fn_det` avaliam o **detector completo em imagem inteira**, comparando as caixas previstas com as caixas YOLO reais por IoU. Assim, é possível separar dois problemas: classificar bem um recorte e posicionar corretamente a bounding box.